# NB08 — WWF RGB × LiDAR × TMF–Hansen | Musanga V3_1

## Three-stage workflow
**1. Inventory and acquisition/image quality:** 216 georeferenced RGB + LiDAR/DSM strips, flight-time evidence, solar zenith/elevation/azimuth, RGB brightness/texture audit, DSM canopy cast-shadow pilot, manual cloud/illumination annotations. **Sunny and uniformly overcast are acceptable; mixed illumination and cloud shadows are flagged.** Crown shadows are handled separately with DSM and solar geometry.

**2. TMF × Hansen forest history:** all-strip forest-history statistics, ecological stratification, availability of homogeneous 125 m neighbourhoods and spatial replication.

**3. Joint sampling:** combine ecological G6 circles with weather/quality screening, review exclusions and representation by stratum. **Original 114 circles and previous reports remain unchanged** until replacement candidates are validated.

Use cached data under `MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar/`. Existing code-cell identifiers (2.F, 4.A, G6 etc.) are retained for compatibility; stage headers give the actual logical order. Do not rerun costly GCS / Earth Engine blocks when their outputs already exist.

**Caveats:** Flight time reconstructed from LiDAR is an approximate RGB acquisition-time proxy, with ambiguous dates for some strips. DSM cast shadows describe geometric canopy shading and cannot independently detect clouds. Existing annotations classify *circles*, not individual pixels. Any cloud classifier must be validated with strip-level holdout before use on the full 216-strip survey.

## Stage 1 — Inventory, acquisition times, solar geometry, RGB audit and cloud contamination
First load survey inventory/geography, then LAS flight times, solar geometry, RGB audit, shadow pilot and the human review workbook. Original code IDs are kept.

## 1 — Environment and authentication
Install dependencies and connect Colab to Google Drive / GCS. Earth Engine is optional for the RGB-first objective.

In [ ]:
# 1.A — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 1.B — Authenticate GCS, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
# The registered project used in the existing Musanga satellite notebooks.
EE_PROJECT = (os.environ.get('EE_PROJECT') or 'musanga-511110').strip()
import ee
def initialize_earth_engine():
    """Initialize with Musanga cloud project; authenticate interactively if necessary."""
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception as first_error:
        print('Earth Engine needs authentication:', str(first_error)[:350])
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    # Lightweight server read verifies the project actually works.
    print('Earth Engine connected:', EE_PROJECT,
          '| server check:', ee.Number(1).getInfo())
    return True
EE_READY=False
try:
    EE_READY=initialize_earth_engine()
except Exception as err:
    print('Earth Engine setup did not complete:',repr(err))
    print('The GCS inventory and RGB tests work without EE. For TMF, fix the project or authenticate before H2.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'Final'/'Outputs'/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

## 2 — WWF inventory and survey geography
Inspect bucket object sizes, filenames and matched per-plot RGB, LiDAR and CHM coverage. Metadata only.

In [ ]:
# 2.A — Inventory WWF GCS products
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 2.A.1 — Match RGB, LiDAR and CHM products
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

### 2.B — WWF survey footprints
Load the 216 georeferenced plot footprints and display a reliable offline-first survey map. Esri satellite is optional; OSM 403 errors are avoided.

In [ ]:
# 2.B.1 — Load georeferenced WWF survey polygons
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 2.B.2 — Map all WWF strips
# The previous OSM basemap returned HTTP 403 within Colab.
# Esri satellite is optional; the default offline map always works.
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape
import pandas as pd

# tiles=None ensures ZERO requests to restricted OSM tile servers.
m = folium.Map(location=[-2.5, 23.5], zoom_start=6, tiles=None,
               control_scale=True, prefer_canvas=True)

# Reliable fallback: blank geographic background (no external tile dependency).
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Local blank geographic background', name='Offline (always works)',
    overlay=False, show=True, control=True, max_zoom=19
).add_to(m)

# Optional imagery; switch it on only if your Colab browser can access Esri.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Imagery (optional)',
    overlay=False, show=False, control=True, max_zoom=19
).add_to(m)

colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c',
        'UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}

# Plot geometry gives survey context even if ALL basemap servers fail.
foot=folium.FeatureGroup(name='Survey footprints',show=True)
folium.GeoJson(
    plots[['plot_id','geometry']].to_json(),
    style_function=lambda f: {'color':'#555555','weight':1.5,'fillOpacity':0.08},
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'])
).add_to(foot)
foot.add_to(m)

for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)} plots',show=True)
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r[k+'_n']) if pd.notna(r[k+'_n']) else 0
                for k in ('RGB','LAS','CHM','DTM','DSM')}
        label=escape(str(r.plot_id))
        html='<b>'+label+'</b><br>'+'<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker(
            [pt.y,pt.x],radius=5,color=col,weight=2,fill=True,
            fill_color=col if counts['RGB'] else '#ffffff',fill_opacity=.9,
            tooltip=label,popup=folium.Popup(html,max_width=300)
        ).add_to(layer)
    layer.add_to(m)

# Geographic graticule: geographic reference even with the offline background.
graticule=folium.FeatureGroup(name='Latitude / longitude grid',show=True)
for latitude in range(-12,7,2):
    folium.PolyLine([[latitude,12],[latitude,32]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{latitude}° latitude').add_to(graticule)
for longitude in range(12,33,2):
    folium.PolyLine([[-13,longitude],[7,longitude]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{longitude}° longitude').add_to(graticule)
graticule.add_to(m)

w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

output=WORK_DIR/'wwf_lidar_survey_map.html'
m.save(str(output))
print('Offline-capable map:',output)
print('To see satellite imagery: select Esri World Imagery in the layer control.')
display(m)

#### 2.C — Optional UAV ROI context
Their spatial overlap is **not** required. Run this cell only if you want to see the original UAV sites on the map.

In [ ]:
# 2.C.1 — Overlay earlier UAV ROIs (optional)
# Source of truth: V3_1 audited manifest and its ROI.shp (no guessed coordinates).
from pathlib import Path
from shapely.geometry import box
import geopandas as gpd
import rasterio
import folium
import pandas as pd
from IPython.display import display
import json
from html import escape

drive_root=Path('/content/drive/MyDrive')
final_root=next((q for q in [
    drive_root/'Colab Notebooks/Musanga/Final',
    drive_root/'Musanga/Final'] if q.is_dir()), None)
assert final_root, 'Musanga/Final not found. Mount Drive and check V3_1 paths.'
manifest_file=final_root/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest_file.exists(), (
    'Approved V3_1 input manifest missing: '+str(manifest_file)
    +'. Run V3_1 Notebook 00 / Block 10 first.')
manifest=json.loads(manifest_file.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.is_file(), f'ROI shapefile missing: {roi_file}'
uav_roi=gpd.read_file(roi_file)
assert uav_roi.crs is not None, 'ROI.shp has no CRS; refusing to guess.'
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
uav_roi.geometry=uav_roi.geometry.make_valid()
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
assert len(uav_roi), 'No valid UAV ROI geometries.'

# Assign ROI polygons to a site by physical overlap with real mosaic footprints.
# Intersections are measured in the ROI's native metric CRS, not lat/lon degrees.
mosaic_records=[]
for entry in manifest['mosaics']:
    image_file=Path(entry['path'])
    if not image_file.exists():
        print('Mosaic missing; cannot assign this footprint:',image_file)
        continue
    with rasterio.open(image_file) as ds:
        assert ds.crs is not None, f'No CRS: {image_file}'
        g=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(uav_roi.crs).iloc[0]
        mosaic_records.append(dict(site=str(entry['site']),file=str(entry['file']),geometry=g))
assert mosaic_records, 'No readable UAV mosaics for ROI attribution.'

# Aggregate overlapping mosaics by forest site, avoiding duplicate MOS geometry.
from shapely.ops import unary_union
site_footprints={
    site:unary_union([x['geometry'] for x in mosaic_records if x['site']==site])
    for site in sorted(set(x['site'] for x in mosaic_records))}
roi_rows=[]
for index,row in uav_roi.iterrows():
    g=row.geometry
    overlaps={site:float(g.intersection(foot).area)
              for site,foot in site_footprints.items()}
    best=max(overlaps,key=overlaps.get)
    if overlaps[best]<=0:
        best='Unassigned'
    roi_rows.append(best)
uav_roi['site']=roi_rows
uav_roi['roi_row']=[int(i) for i in range(len(uav_roi))]
print('UAV ROI features:',len(uav_roi))
print('Site assignment:',uav_roi.site.value_counts().to_dict())
if (uav_roi.site=='Unassigned').any():
    print('WARNING: Unassigned ROI polygons are shown separately, not silently removed.')

roi_wgs=uav_roi.to_crs('EPSG:4326')
roi_colors={'Baego':'#ef4444','MOS':'#f59e0b','Yangambi':'#10b981',
            'Yoko':'#ec4899','Unassigned':'#111827'}
for site, group in roi_wgs.groupby('site'):
    layer=folium.FeatureGroup(name=f'UAV ROI — {site} ({len(group)})',show=True)
    color=roi_colors.get(site,'#111827')
    folium.GeoJson(
        group[['site','roi_row','geometry']].to_json(),
        style_function=lambda feat,c=color:{
            'color':c,'weight':3.5,'fillColor':c,'fillOpacity':.16},
        highlight_function=lambda feat:{'weight':5,'fillOpacity':.3},
        tooltip=folium.GeoJsonTooltip(fields=['site','roi_row'],
            aliases=['Musanga site','ROI row'])
    ).add_to(layer)
    # Distinct labels at true ROI interiors
    for _,rr in group.iterrows():
        pt=rr.geometry.representative_point()
        folium.Marker([pt.y,pt.x],
            icon=folium.DivIcon(html=(
                '<div style="font: bold 11px sans-serif;white-space:nowrap;'
                'text-shadow:1px 1px 2px white,-1px -1px 2px white;color:'
                +color+';">'+escape(site)+'</div>'))
        ).add_to(layer)
    layer.add_to(m)

# Drop old layer controls before adding the revised one.
for key, child in list(m._children.items()):
    if isinstance(child,folium.map.LayerControl):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

# Plot boundaries remain visible against the complete WWF footprint.
allbounds=pd.concat([
    pd.Series([plots.total_bounds[0],roi_wgs.total_bounds[0]]),
    pd.Series([plots.total_bounds[1],roi_wgs.total_bounds[1]]),
    pd.Series([plots.total_bounds[2],roi_wgs.total_bounds[2]]),
    pd.Series([plots.total_bounds[3],roi_wgs.total_bounds[3]])
],axis=1)
m.fit_bounds([[min(plots.total_bounds[1],roi_wgs.total_bounds[1]),
               min(plots.total_bounds[0],roi_wgs.total_bounds[0])],
              [max(plots.total_bounds[3],roi_wgs.total_bounds[3]),
               max(plots.total_bounds[2],roi_wgs.total_bounds[2])]])

uav_roi.to_file(WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg',driver='GPKG')
map_path=WORK_DIR/'wwf_lidar_plus_musanga_uav_roi.html'
m.save(str(map_path))
print('Saved map:',map_path)
print('Saved georeferenced ROI:',WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg')
display(m)

### Stage 1B — Acquisition timing and solar geometry

### 2.F — Acquisition timing from raw LiDAR and survey documentation

Read LAS/LAZ file headers using GCS **byte ranges**, test representative uncompressed LAS point GPS times, and inventory potential survey reports. No entire point clouds or RGB mosaics are downloaded. The results are **evidence candidates, not confirmed RGB acquisition times**: LiDAR and RGB may have different flight times, GPS-week encoding requires an independently known week, and GPS-scale dates are not UTC until leap seconds have been accounted for.

Run after **1.B and 2.A**; it does not require re-running 2.D or 2.E. Defaults to a 12-file pilot prioritizing plot-specific LAS (not Ferry_Data) across UTM zones, with point samples from at most eight LAS files. The full-header audit can be enabled after inspecting the pilot. Results are saved to RGB_quality_audit/acquisition_timing.

In [ ]:
# 2.F.1 — LAS header/GPS-time audit (range reads; no full point-cloud download)
import json, struct, math, re
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd
from IPython.display import display
from google.cloud import storage
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
DEST=ROOT/'RGB_quality_audit'/'acquisition_timing'
DEST.mkdir(parents=True,exist_ok=True)
if 'client' not in globals(): client=storage.Client()
if 'df' not in globals():
    df=pd.read_csv(ROOT/'carbon_maps_inventory.csv')
assert {'path','plot_id'}.issubset(df.columns), 'Run 2.A to obtain WWF object inventory'
BUCKET='carbon-maps'
ALL_LAS_HEADERS=False       # True after pilot: read headers for all LAS/LAZ objects
MAX_PILOT_FILES=12
MAX_SAMPLE_FILES=8          # max files for first/mid/last point reads
POINTS_PER_RANGE=48
RECHECK=True  # update previous header-only results with correct LAS GPS offsets
inventory=df.copy()
inventory['path']=inventory.path.astype(str)
las=inventory[inventory.path.str.contains(r'\.(?:las|laz)$',case=False,regex=True,na=False)].copy()
docs=inventory[inventory.path.str.contains(r'\.(?:pdf|xml|txt|csv|log|json|xlsx|xls|kml|kmz)$',
    case=False,regex=True,na=False)].copy()
docs=docs[docs.path.str.contains(r'report|flight|mission|survey|gps|metadata|raw.data|plot',
    case=False,regex=True,na=False)]
docs[['path','plot_id']].to_csv(DEST/'WWF_potential_flight_reports.csv',index=False)
print('LAS/LAZ objects:',len(las),'| potential metadata/report files:',len(docs))
display(las[['path','plot_id']].head(15))
display(docs[['path','plot_id']].head(20))
if not las.empty:
    # Ferry_Data may be transit LiDAR, not the survey plot itself. Prioritize plot-specific LAS.
    las['is_plot_lidar']=las.path.str.contains(
        r'^00-raw-data/Plots/UTM\d{1,2}[NS]/Plot\d+/Las/',case=False,regex=True,na=False)
    las['zone']=las.path.str.extract(r'(UTM\d{1,2}[NS])',flags=re.I,expand=False)
    las['size_sort']=pd.to_numeric(las.get('size_GB',pd.Series(index=las.index,dtype=float)),errors='coerce').fillna(0)
    plot_las=las[las.is_plot_lidar].copy()
    print('Plot-specific raw LAS files:',len(plot_las),'| ferry/other:',len(las)-len(plot_las))
    pool=plot_las if not plot_las.empty else las
    # One large plot LAS per zone, then additional plots (not repeats from a ferry corridor).
    pool=pool.sort_values(['size_sort','path'],ascending=[False,True])
    if ALL_LAS_HEADERS: selected=pool
    else:
        first=pool.groupby('zone',sort=True,dropna=False).head(1)
        extra=pool[~pool.path.isin(first.path)].drop_duplicates('plot_id')
        selected=pd.concat([first,extra],ignore_index=True).head(MAX_PILOT_FILES)
    display(selected[['path','plot_id','size_sort']].to_string(index=False))
    gps_offsets={1:20,3:20,4:20,5:20,6:22,7:22,8:22,9:22,10:22}
    bkt=client.bucket(BUCKET)
    rows=[]
    for idx,ob in enumerate(selected.itertuples(index=False)):
        path=str(ob.path)
        cache=DEST/('LAS_'+str(idx+1).zfill(4)+'.json')
        if cache.exists() and not RECHECK:
            record=json.loads(cache.read_text())
            if record.get('path')==path:
                rows.append(record)
                continue
        record=dict(path=path,plot_id=str(ob.plot_id) if pd.notna(ob.plot_id) else '',
           status='UNKNOWN',gps_samples_valid=0,
           gps_time_min=None,gps_time_max=None,gps_calendar_start_candidate=None,
           gps_calendar_end_candidate=None,rgb_time_verified=False,note='')
        blob=bkt.blob(path)
        try:
            h=blob.download_as_bytes(start=0,end=374)
            if len(h)<227 or h[:4]!=b'LASF': raise ValueError('Not an LAS header')
            major,minor=h[24],h[25]
            encoding=struct.unpack_from('<H',h,6)[0]
            pf=h[104]&63
            compressed=bool(h[104]&192) or path.lower().endswith('.laz')
            rec_len=struct.unpack_from('<H',h,105)[0]
            start=struct.unpack_from('<I',h,96)[0]
            legacy_n=struct.unpack_from('<I',h,107)[0]
            extended_n=(struct.unpack_from('<Q',h,247)[0]
                         if (major,minor)>=(1,4) and len(h)>=255 else 0)
            n=int(extended_n or legacy_n)
            blob_size=int(ob.size_GB*1e9) if hasattr(ob,'size_GB') and pd.notna(ob.size_GB) else None
            # Approximate file bytes from the cached inventory; informational only.
            inferred_n=(max(0,(blob_size-start)//rec_len)
                        if blob_size is not None and rec_len>0 and start<blob_size else 0)
            encoding_name='adjusted_standard_GPS' if encoding&1 else 'GPS_week_seconds'
            record.update(status='HEADER_OK',las_version=f'{major}.{minor}',point_format=pf,
                point_count=n,point_count_legacy=legacy_n,point_count_extended=extended_n,
                estimated_record_count_from_inventory=inferred_n,point_data_offset=start,
                object_bytes=blob_size,point_length=rec_len,compressed=compressed,gps_encoding=encoding_name)
            if pf not in gps_offsets:record['note']='GPS time not available in this point format.'
            elif compressed:record['note']='LAZ is compressed: header only; point decoding needs LAZ reader.'
            elif rec_len<gps_offsets[pf]+8:
                record['note']=f'Point record length {rec_len} too short for format {pf}.'
            elif n<=0:
                record['note']=(f'Header declares zero points; file-size estimate={inferred_n}; '
                                'not sampling until header validity is established.')
            elif idx>=MAX_SAMPLE_FILES:record['note']='Header checked; point samples limited by setting.'
            else:
                values=[]
                starts=[0,max(0,n//2-POINTS_PER_RANGE//2),max(0,n-POINTS_PER_RANGE)]
                for p in starts:
                    nr=min(POINTS_PER_RANGE,n-p)
                    if nr<=0:continue
                    pos=start+p*rec_len
                    chunk=blob.download_as_bytes(start=pos,end=pos+nr*rec_len-1)
                    for k in range(min(nr,len(chunk)//rec_len)):
                        t=struct.unpack_from('<d',chunk,k*rec_len+gps_offsets[pf])[0]
                        if math.isfinite(t) and t>0:values.append(t)
                record['gps_samples_valid']=len(values)
                if values:
                    lo,hi=min(values),max(values)
                    record.update(status='GPS_POINTS_SAMPLED',gps_time_min=lo,gps_time_max=hi)
                    if encoding&1:
                        origin=datetime(1980,1,6,tzinfo=timezone.utc)
                        record['gps_calendar_start_candidate']=(origin+timedelta(seconds=lo+1e9)).isoformat()
                        record['gps_calendar_end_candidate']=(origin+timedelta(seconds=hi+1e9)).isoformat()
                        record['note']='GPS-scale date, NOT UTC; verify GPS–UTC offset and RGB co-acquisition.'
                    else:record['note']='GPS week seconds; cannot resolve a calendar date without week number.'
                else:record['note']='No plausible nonzero GPS times in sampled point ranges.'
        except Exception as exc:
            record['status']='ERROR'
            record['note']=str(exc)[:250]
        rows.append(record)
        cache.write_text(json.dumps(record,indent=2))
        if (idx+1)%10==0:print('Audited headers:',idx+1,'/',len(selected))
    results=pd.DataFrame(rows)
    results.to_csv(DEST/'WWF_LAS_GPS_time_evidence.csv',index=False)
    cols=['path','las_version','point_format','point_count','point_length',
          'point_data_offset','estimated_record_count_from_inventory','compressed','gps_encoding',
          'status','gps_samples_valid','gps_calendar_start_candidate','note']
    display(results[[c for c in cols if c in results.columns]])
else:
    results=pd.DataFrame()
    print('No LAS/LAZ in inventory; review the available survey documents.')
rgb=inventory[inventory.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',
    case=False,na=False)][['plot_id','path']].drop_duplicates('plot_id').rename(columns={'path':'rgb_path'})
rgb['rgb_acquisition_UTC_verified']=pd.NA
rgb['same_RGB_LiDAR_flight_verified']=False
rgb['status']='UNVERIFIED'
rgb['raw_LAS_LAZ_matches']=rgb.plot_id.map(las.groupby('plot_id').size()).fillna(0).astype(int)
if not results.empty and 'gps_calendar_start_candidate' in results:
    candidates=results.dropna(subset=['gps_calendar_start_candidate'])
    if not candidates.empty:
        mapping=candidates.drop_duplicates('plot_id').set_index('plot_id').gps_calendar_start_candidate
        rgb['LiDAR_GPS_date_candidate_NOT_RGB']=rgb.plot_id.map(mapping)
rgb.to_csv(DEST/'WWF_RGB_216_acquisition_evidence.csv',index=False)
print('RGB strips:',len(rgb),'| confirmed RGB acquisition times: 0 (not yet established)')
print('Saved:',DEST)


### 2.F.2 — Reconstruct candidate LiDAR acquisition dates for all 216 RGB plots

The pilot has confirmed readable LAS point GPS timestamps with plausible **2014–2015 GPS-scale dates**. This block extends the test to **all 216 WWF RGB plots**, preferentially sampling **two separate plot-specific LAS files per RGB plot** to reveal possible multiple acquisition dates. It reads only small GCS byte ranges and caches results **per LAS file** so reruns skip completed tests. It avoids 9,823-file scans and excludes ferry data.

An adjusted GPS date is converted to UTC using the **16-second GPS–UTC offset valid during 2014 and January–June 2015**. Dates outside that period, inconsistent dates between files in the same plot, and invalid GPS times are explicitly flagged. The point-time ranges represent sampled flight fragments, **not guaranteed earliest and latest flight times for the complete plot**.

**Crucial:** matching LiDAR plot IDs do *not* demonstrate simultaneous RGB collection. Output is `LiDAR UTC acquisition evidence` and **not confirmed RGB acquisition time**. Do not calculate or interpret crown-shadow solar geometry from this until RGB/LiDAR co-acquisition is documented using survey reports or image metadata. The original 114 circles and RGB QC results remain untouched.

In [ ]:
# 2.F.2 — All 216 WWF RGB plots: LAS GPS timing evidence, cached and resumable
# Requires 1.B / 2.A or saved carbon_maps_inventory.csv. NO full LiDAR downloads.
import hashlib, json, struct, math, re
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd
from IPython.display import display
from google.cloud import storage

ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
DEST=ROOT/'RGB_quality_audit'/'acquisition_timing'
CACHE=DEST/'all_plot_las_samples'
CACHE.mkdir(parents=True,exist_ok=True)
if 'df' not in globals(): df=pd.read_csv(ROOT/'carbon_maps_inventory.csv')
if 'client' not in globals(): client=storage.Client()
assert {'path','plot_id'}.issubset(df.columns)
BUCKET='carbon-maps'
FILES_PER_PLOT=2       # two independent flight fragments per plot
POINT_SAMPLES_PER_RANGE=24
MAX_PLOTS=None         # set to 12 for a small pilot; None evaluates all 216
REDO_CACHED=False

inv=df.copy()
inv['path']=inv.path.astype(str)
rgbs=inv[inv.path.str.fullmatch(
    r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)].copy()
rgbids=sorted(rgbs.plot_id.dropna().unique())
assert len(rgbids)==216,f'Expected 216 WWF RGB plots, found {len(rgbids)}'
las=inv[inv.path.str.contains(
    r'^00-raw-data/Plots/UTM\d{1,2}[NS]/Plot\d+/Las/.*\.las$',
    case=False,regex=True,na=False)].copy()
# Derive plot identity from the actual directory rather than ferry-like path fragments.
m=las.path.str.extract(r'^00-raw-data/Plots/(UTM\d{1,2}[NS])/Plot(\d+)/Las/',
    flags=re.I)
las['confirmed_plot_id']=m[0].str.upper()+'_Plot'+m[1].fillna('')
las=las[las.confirmed_plot_id.isin(rgbids)].copy()
las['size_bytes_approx']=pd.to_numeric(las.size_GB,errors='coerce').fillna(0)*1e9
las=las.sort_values(['confirmed_plot_id','size_bytes_approx','path'],
                    ascending=[True,False,True])
las=las.groupby('confirmed_plot_id',sort=False).head(FILES_PER_PLOT)
bucket=client.bucket(BUCKET)
gps_offsets={1:20,3:20,4:20,5:20,6:22,7:22,8:22,9:22,10:22}
gps_origin=datetime(1980,1,6,tzinfo=timezone.utc)
def acquire_las_range(path):
    blob=bucket.blob(path)
    h=blob.download_as_bytes(start=0,end=374)
    if len(h)<227 or h[:4]!=b'LASF':raise ValueError('Invalid LAS header')
    v=(h[24],h[25])
    pf=h[104]&63
    zipped=bool(h[104]&192)
    enc=struct.unpack_from('<H',h,6)[0]
    rlen=struct.unpack_from('<H',h,105)[0]
    start=struct.unpack_from('<I',h,96)[0]
    n=struct.unpack_from('<I',h,107)[0]
    if v>=(1,4) and len(h)>=255:
        n=struct.unpack_from('<Q',h,247)[0] or n
    out=dict(las_version=f'{v[0]}.{v[1]}',point_count=n,point_format=pf,
        point_length=rlen,gps_encoding='adjusted_standard_GPS' if enc&1 else 'GPS_week_seconds',
        status='HEADER_READ',sample_count=0)
    if zipped or pf not in gps_offsets or rlen<gps_offsets.get(pf,0)+8 or not n:
        out['status']='NO_READABLE_GPS_POINT_RECORDS'
        return out
    vals=[]
    starts=[0,max(0,n//2-POINT_SAMPLES_PER_RANGE//2),
            max(0,n-POINT_SAMPLES_PER_RANGE)]
    for q in starts:
        count=min(POINT_SAMPLES_PER_RANGE,n-q)
        if count<=0:continue
        pos=start+q*rlen
        data=blob.download_as_bytes(start=pos,end=pos+count*rlen-1)
        for k in range(min(count,len(data)//rlen)):
            raw=struct.unpack_from('<d',data,k*rlen+gps_offsets[pf])[0]
            if math.isfinite(raw) and raw>0:vals.append(raw)
    out['sample_count']=len(vals)
    if not vals:
        out['status']='NO_VALID_SAMPLE_GPS_TIMES'
        return out
    out['gps_min_raw']=min(vals);out['gps_max_raw']=max(vals)
    if not(enc&1):
        out['status']='GPS_WEEK_UNKNOWN'
        return out
    try:
        gps_lo=gps_origin+timedelta(seconds=min(vals)+1e9)
        gps_hi=gps_origin+timedelta(seconds=max(vals)+1e9)
        out['gps_clock_start']=gps_lo.isoformat()
        out['gps_clock_end']=gps_hi.isoformat()
        # GPS was 16 seconds ahead of UTC in 2014 and through 2015-06-30.
        if datetime(2014,1,1,tzinfo=timezone.utc)<=gps_lo<datetime(2015,7,1,tzinfo=timezone.utc):
            off=16
        elif datetime(2015,7,2,tzinfo=timezone.utc)<=gps_lo<datetime(2017,1,1,tzinfo=timezone.utc):
            off=17
        else:
            out['status']='GPS_DATE_OUTSIDE_EXPECTED_WINDOW'
            return out
        utc_lo=gps_lo-timedelta(seconds=off)
        utc_hi=gps_hi-timedelta(seconds=off)
        out['utc_start_candidate']=utc_lo.isoformat()
        out['utc_end_candidate']=utc_hi.isoformat()
        out['utc_date_candidate']=utc_lo.date().isoformat()
        out['status']='LIDAR_GPS_UTC_CANDIDATE'
    except (OverflowError,ValueError) as ex:
        out['status']='GPS_DATE_CONVERSION_ERROR';out['error']=str(ex)[:200]
    return out

rows=[]
plot_scope=rgbids if MAX_PLOTS is None else rgbids[:MAX_PLOTS]
selected=las[las.confirmed_plot_id.isin(plot_scope)]
print('RGB plots:',len(plot_scope),'| selected plot-specific LAS files:',len(selected))
for i,(idx,ob) in enumerate(selected.iterrows(),1):
    path=ob.path
    cache=CACHE/(hashlib.sha256(path.encode()).hexdigest()[:20]+'.json')
    if cache.exists() and not REDO_CACHED:
        obj=json.loads(cache.read_text())
        if obj.get('path')==path:
            rows.append(obj)
            continue
    obj={'path':path,'plot_id':ob.confirmed_plot_id}
    try:obj.update(acquire_las_range(path))
    except Exception as ex:obj.update(status='GCS_READ_ERROR',error=repr(ex)[:250])
    rows.append(obj)
    cache.write_text(json.dumps(obj,indent=2))
    if i%24==0:print('LAS files sampled:',i,'/',len(selected))

times=pd.DataFrame(rows)
times.to_csv(DEST/'WWF_216_plots_LAS_file_timing_evidence.csv',index=False)
summaries=[]
for pid in plot_scope:
    pp=times[times.plot_id==pid]
    valid=pp[pp.status=='LIDAR_GPS_UTC_CANDIDATE'] if not pp.empty else pp
    # Capture every UTC date touched by a sampled LAS range, not only its start date.
    days=sorted(set(
        pd.to_datetime(valid.utc_start_candidate.dropna(),utc=True).dt.strftime('%Y-%m-%d').tolist()
        +pd.to_datetime(valid.utc_end_candidate.dropna(),utc=True).dt.strftime('%Y-%m-%d').tolist()
    )) if not valid.empty else []
    summaries.append(dict(plot_id=pid,las_files_tested=len(pp),
         las_files_with_UTC_evidence=len(valid),different_dates=len(days),
         candidate_lidar_dates='; '.join(days),
         candidate_lidar_start_UTC=(valid.utc_start_candidate.min() if len(valid) else None),
         candidate_lidar_end_UTC=(valid.utc_end_candidate.max() if len(valid) else None),
         multiple_lidar_dates=len(days)>1,
         lidar_date_evidence_available=bool(days),
         rgb_same_flight_confirmed=False,rgb_acquisition_UTC_confirmed=False))
overview=pd.DataFrame(summaries)
overview.to_csv(DEST/'WWF_216_plot_LiDAR_acquisition_timing_summary.csv',index=False)
print('Plots with sampled LAS dates:',int(overview.lidar_date_evidence_available.sum()),'of',len(overview))
print('Plots with multiple sampled LAS acquisition dates:',int(overview.multiple_lidar_dates.sum()))
print('RGB acquisition time still unverified without same-flight evidence.')
display(overview.head(20))
display(times.status.value_counts().rename_axis('LAS status').reset_index(name='files'))
print('Saved results:',DEST)


### 2.F.3 — RGB flight timing relative to local solar noon and solar zenith

Since LiDAR and RGB were collected simultaneously, use the recovered LiDAR GPS UTC timestamps as **RGB flight-time evidence**. For each plot, calculate the **solar noon at its geographic location on its acquisition date**, solar zenith (0° = sun overhead; 90° = horizon), and signed time from solar noon (negative = morning). Plot the distribution **once per plot**, not once per LAS file, to avoid pseudoreplication.

This step reads only cached 2.F.2 CSV files and the 216 WWF footprint polygons (`plots` from 2.B.1). **No GCS requests or reprocessing of images/LiDAR.** Where two LAS fragments disagree on the day, flag the plot as multi-day and show one representative fragment rather than silently averaging dates. Sun elevation describes expected geometric shadows, **not actual cloud shadow**, crown visibility or exposure metadata. Save publication-ready PNGs and a plot-level CSV.

In [ ]:
# 2.F.3 — Solar-noon offset and solar-zenith distributions for WWF RGB flights
# Requires 2.B.1 ('plots') and completed 2.F.2 CSV; NO network reads of GCS objects.
%pip -q install "astral>=3,<4"
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
from datetime import timezone
from astral import Observer
from astral.sun import noon,elevation,azimuth
from IPython.display import display

ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
TIME_DIR=ROOT/'RGB_quality_audit'/'acquisition_timing'
FILES=TIME_DIR/'WWF_216_plots_LAS_file_timing_evidence.csv'
assert FILES.exists(), 'Run 2.F.2 first to obtain dates and times for all 216 plots.'
assert 'plots' in globals(), 'Run 2.B.1 to restore WWF survey footprints.'
raw=pd.read_csv(FILES)
required={'plot_id','status','utc_start_candidate','utc_end_candidate'}
assert required.issubset(raw.columns),f'Missing columns in saved file: {required-set(raw.columns)}'
raw=raw.loc[raw.status.eq('LIDAR_GPS_UTC_CANDIDATE')].copy()
assert not raw.empty,'No usable UTC timing evidence. Inspect output of 2.F.2.'
raw['start']=pd.to_datetime(raw.utc_start_candidate,utc=True,errors='coerce')
raw['end']=pd.to_datetime(raw.utc_end_candidate,utc=True,errors='coerce')
raw=raw.dropna(subset=['start','end'])
raw=raw.loc[raw.end>=raw.start].copy()
raw['mid_UTC']=raw.start+(raw.end-raw.start)/2
raw['start_day_UTC']=raw.start.dt.date.astype(str)
raw['end_day_UTC']=raw.end.dt.date.astype(str)
raw['crosses_UTC_day']=raw.start_day_UTC.ne(raw.end_day_UTC)
raw['day_UTC']=raw.mid_UTC.dt.date.astype(str)
# Respect original strip geometry and avoid averaging dates from separate flights.
locations=plots[['plot_id','geometry']].drop_duplicates('plot_id').to_crs('EPSG:4326').copy()
pt=locations.geometry.representative_point()
locations['latitude']=pt.y.values;locations['longitude']=pt.x.values
loc=locations.drop(columns='geometry')
raw=raw.merge(loc,on='plot_id',how='inner',validate='many_to_one')
rows=[]
for pid,g in raw.groupby('plot_id',sort=True):
    days=sorted(set(g.start_day_UTC.tolist()+g.end_day_UTC.tolist()))
    # Multi-day acquisitions are ambiguous for a single solar-angle estimate.
    # Exclude them from this single-time plot distribution pending image-level matching.
    if len(days)>1:
        print('EXCLUDED multi-day plot pending acquisition review:',pid,', '.join(days))
        continue
    chosen_day=days[0]
    subset=g[g.day_UTC.eq(chosen_day)].sort_values('mid_UTC')
    # Midpoint of sampled fragment midtimes for a single day; proxy for flight time.
    t=pd.Timestamp(subset.mid_UTC.iloc[len(subset)//2]).to_pydatetime()
    lat=float(subset.latitude.iloc[0]);lon=float(subset.longitude.iloc[0])
    observer=Observer(latitude=lat,longitude=lon,elevation=0)
    try:
        solar_noon=noon(observer,date=t.date(),tzinfo=timezone.utc)
        elev=float(elevation(observer,t));azi=float(azimuth(observer,t))
        delta=(t-solar_noon).total_seconds()/3600
        zenith=90-elev
        rows.append(dict(plot_id=pid,latitude=lat,longitude=lon,
            UTC_flight_time_proxy=t.isoformat(),solar_noon_UTC=solar_noon.isoformat(),
            hours_from_solar_noon=delta,solar_zenith_deg=zenith,solar_elevation_deg=elev,
            solar_azimuth_deg=azi,shadow_length_per_m_height=(
                1/np.tan(np.deg2rad(elev)) if elev>0 else np.nan),
            lidar_files_usable=len(g),distinct_acquisition_days=len(days),
            multi_day_flag=len(days)>1,all_candidate_days='; '.join(days)))
    except Exception as ex:
        print('Solar geometry skipped:',pid,repr(ex)[:180])
solar=pd.DataFrame(rows)
assert len(solar)>0,'No solar calculations succeeded'
solar['zone']=solar.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
solar.to_csv(TIME_DIR/'WWF_RGB_solar_geometry_by_plot.csv',index=False)
print('Plots with single-day solar geometry:',len(solar),'/ 216')
print('Multi-day plots were excluded, not silently assigned one date.')
print('Multiple candidate acquisition days:',int(solar.multi_day_flag.sum()))
print('Zenith is measured from overhead; smaller zenith = higher sun and shorter shadows.')
display(solar[['plot_id','UTC_flight_time_proxy','hours_from_solar_noon',
    'solar_zenith_deg','solar_elevation_deg','shadow_length_per_m_height',
    'multi_day_flag']].head(20))

fig,axs=plt.subplots(1,3,figsize=(16,4.7),constrained_layout=True)
axs[0].hist(solar.hours_from_solar_noon,bins=np.arange(-7,7.51,.5),
            edgecolor='white')
axs[0].axvline(0,color='black',ls='--',lw=1,label='Solar noon')
axs[0].set(xlabel='Hours relative to local solar noon',ylabel='Number of WWF plots',
           title='Acquisition timing');axs[0].legend()
axs[1].hist(solar.solar_zenith_deg,bins=np.arange(0,95,5),edgecolor='white')
axs[1].set(xlabel='Solar zenith (degrees)',ylabel='Number of WWF plots',
           title='Solar zenith distribution')
axs[2].scatter(solar.hours_from_solar_noon,solar.solar_zenith_deg,
               s=25,alpha=.75)
axs[2].axvline(0,color='black',ls='--',lw=1)
axs[2].set(xlabel='Hours relative to local solar noon',
           ylabel='Solar zenith (degrees)',title='Timing versus solar geometry')
p=TIME_DIR/'WWF_solar_noon_and_zenith_distributions.png'
fig.savefig(p,dpi=200,bbox_inches='tight');plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(7,4.5))
valid=solar[(solar.solar_elevation_deg>0)&solar.shadow_length_per_m_height.notna()]
ax.hist(valid.shadow_length_per_m_height.clip(upper=10),
        bins=np.linspace(0,10,31),edgecolor='white')
ax.set(xlabel='Idealized shadow length / vertical object height (capped at 10)',
       ylabel='Number of WWF plots',title='Potential shadow geometry during acquisition')
p2=TIME_DIR/'WWF_idealized_shadow_ratio_distribution.png'
fig.savefig(p2,dpi=200,bbox_inches='tight');plt.show();plt.close(fig)
print('Solar zenith median [IQR]:',
      round(float(solar.solar_zenith_deg.median()),1),
      [round(float(x),1) for x in solar.solar_zenith_deg.quantile([.25,.75])])
print('Flight time offset from noon median [IQR] (hours):',
      round(float(solar.hours_from_solar_noon.median()),2),
      [round(float(x),2) for x in solar.hours_from_solar_noon.quantile([.25,.75])])
print('Saved charts and CSV to:',TIME_DIR)
print('Interpretation: a plot-time proxy inferred from sampled LiDAR file fragments;')
print('some mosaics may span multiple flight strips/dates and solar angles.')


### Stage 1C — RGB audit and canopy cast-shadow pilot

### 2.D — **NEW: all-216 WWF acquisition metadata and RGB-quality audit** (run before sampling)

**Purpose:** detect mosaics with cloud-shadow contamination, poor illumination, extreme radiometric differences, blur and nodata margins **before** selecting final annotation circles. This section is a diagnostic screen of *all 216 strips*, not an image transformation and not an automatic rejection rule.

**Methods and interpretation:**
- Read each RGB TIFF **header/tags** and inspect **five spatially distributed, bounded native-resolution windows** selected from interior survey geometry. Never download or resample an entire mosaic. Cache individual strip records; reruns skip completed strips.
- For each window, record nonblack fraction, median brightness, channel medians, dark-pixel fraction, lower/upper brightness quantiles, local luminance variation and edge-strength (a **relative** sharpness proxy). An extremely dark forest, shadow under crowns or low-texture vegetation is **not necessarily a cloud shadow**. Broad low-brightness spatial contrasts will be flagged for **manual interpretation**; no unvalidated cloud mask is generated.
- Search existing RGB tags and GCS object filenames for acquisition-time evidence, and identify LAS/LAZ and flight/report metadata candidates. **LAS headers alone do not guarantee the survey's acquisition time**; individual GPS point times may require separate inspection. TIFF DateTime and GCS update timestamps are **not** interpreted as flight dates.
- Create one per-window CSV, one **216-strip QC inventory**, an acquisition-evidence CSV, and an RGB **overview contact sheet**. A strip's metrics represent sampled windows, **not complete-image cloud-shadow fractions**.
- Work in `Final/Outputs/V3_1/NB08_WWF_Lidar/RGB_quality_audit/`; all cached scientific outputs and original imagery remain unchanged.

The audit is **not a substitute for local candidate-circle QC**: cloud shadow can be confined to part of an otherwise good mosaic. Use this audit to prioritize detailed candidate screening in §7.C, then evaluate whether a V2 selection is justified.

In [ ]:
# 2.D.1 — FULL WWF RGB SURVEY QUALITY AUDIT (216 STRIPS, SMALL BOUNDED READS)
# Run after 1.B and 2.A–2.B; no RGB downloads or raster exports.
import os, re, json, math, time, tempfile, hashlib, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import from_bounds, Window
from rasterio.enums import Resampling
from pyproj import Transformer
from shapely.geometry import Point
from PIL import Image as PILImage, ImageOps, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from google.colab import auth,drive
import google.auth
from google.auth.transport.requests import Request
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR=BASE/'Final'/'Outputs'/'V3_1'/'NB08_WWF_Lidar'
AUDIT=WORK_DIR/'RGB_quality_audit'
CACHE=AUDIT/'per_strip'
THUMBS=AUDIT/'sample_previews'
for d in [AUDIT,CACHE,THUMBS]:d.mkdir(parents=True,exist_ok=True)
assert 'plots' in globals() and 'df' in globals(), 'Run blocks 2.A and 2.B.1 first.'
foot=plots[plots.plot_id.notna()].copy().to_crs(4326)
foot=foot[foot.plot_id.str.match(r'^UTM\d+[NS]_Plot\d+$')]
foot=foot.drop_duplicates('plot_id')
assert len(foot)==216, f'Expected exactly 216 survey plots, got {len(foot)}'
AUDIT_FORCE=False
AUDIT_TEST_N=None         # e.g. 12 for an initial pilot; None -> all 216
AUDIT_WINDOWS=5           # systematic interior probes per strip (not full raster coverage)
AUDIT_NATIVE_WINDOW_M=64  # each probe reads ~640x640 source pixels at 10 cm
AUDIT_RGB_SAMPLE_PX=360   # bounded decimation per probe for metrics/previews
AUDIT_SEED=20261028
AUDIT_ISSUE_DARK_THRESHOLD=.12  # provisional dark-tone heuristic, not cloud classifier
AUDIT_LOW_VALID_THRESHOLD=.90
AUDIT_THUMB_SIZE=220
def get_gcs_token():
    try:
        credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
        if not credentials.valid or not credentials.token:credentials.refresh(Request())
        assert credentials.token
        return str(credentials.token)
    except Exception:
        auth.authenticate_user()
        credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
        if not credentials.valid or not credentials.token:credentials.refresh(Request())
        assert credentials.token
        return str(credentials.token)
token=get_gcs_token()
print('WWF GCS read credentials ready; token hidden.')
rgb_inv=df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)]
paths=rgb_inv.drop_duplicates('plot_id').set_index('plot_id').path.to_dict()
assert len(set(foot.plot_id)-set(paths))==0,'RGB missing for one or more WWF plots'
# The original acquisition-specific timestamp is not verified by file mtime.
evidence_terms=re.compile(r'flight|acqui|survey|date|exposure|gps|capture|sensing',re.I)
support=df[df.path.str.contains(r'\.(?:las|laz|xml|pdf|csv|txt|log|json)$',
                                 case=False,regex=True,na=False)].copy()
support_paths=support.path.astype(str).tolist()
grid=np.array([(u,v) for v in (.18,.50,.82) for u in (.18,.50,.82)])
chosen_grid=grid[[0,2,4,6,8]]  # distributed across footprint bounding box
records=[]
selection=foot.sort_values('plot_id') if AUDIT_TEST_N is None else foot.sort_values('plot_id').head(AUDIT_TEST_N)
for i,rr in enumerate(selection.itertuples(index=False),1):
    pid=str(rr.plot_id)
    file=CACHE/(pid+'.json')
    if file.exists() and not AUDIT_FORCE:
        records.append(json.loads(file.read_text()))
        if i%24==0:print('Cached',i,'/',len(selection))
        continue
    info={'plot_id':pid,'source_uri':'gs://carbon-maps/'+paths[pid],
          'sample_windows':[],'tag_evidence':[],'errors':[]}
    zone=pid.split('_')[0]
    info['supporting_files']=[p for p in support_paths
        if pid.lower() in p.lower() or (zone.lower() in p.lower() and evidence_terms.search(p))][:30]
    rng=np.random.default_rng(AUDIT_SEED+int(hashlib.md5(pid.encode()).hexdigest()[:8],16))
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
                           GS_OAUTH2_ACCESS_TOKEN=token):
            with rasterio.open('/vsigs/carbon-maps/'+paths[pid]) as ds:
                info.update(crs=str(ds.crs),width=int(ds.width),height=int(ds.height),
                    pixel_size_x=float(ds.res[0]),pixel_size_y=float(ds.res[1]),
                    bands=int(ds.count),dtype=str(ds.dtypes[0]),
                    block_shapes=[list(x) for x in ds.block_shapes[:3]],
                    has_overviews=bool(ds.overviews(1)),nodata=str(ds.nodata))
                if ds.count<3:raise ValueError('Fewer than 3 raster bands')
                for ns in ['']+list(ds.tag_namespaces()):
                    try: tags=ds.tags(ns=ns) if ns else ds.tags()
                    except Exception:continue
                    for key,value in tags.items():
                        if evidence_terms.search(key) or re.search('datetime|software',key,re.I):
                            info['tag_evidence'].append(dict(namespace=ns,key=key,value=str(value)[:230],
                                acquisition_verified=False))
                bounds=rr.geometry.bounds
                probes=[]
                for u,v in chosen_grid:
                    lon=bounds[0]+u*(bounds[2]-bounds[0])
                    lat=bounds[1]+v*(bounds[3]-bounds[1])
                    if rr.geometry.covers(Point(lon,lat)):probes.append((lon,lat))
                # Fill difficult or narrow/irregular footprints using deterministic rejection sampling.
                attempts=0
                while len(probes)<AUDIT_WINDOWS and attempts<250:
                    attempts+=1
                    lon=rng.uniform(bounds[0],bounds[2])
                    lat=rng.uniform(bounds[1],bounds[3])
                    if rr.geometry.covers(Point(lon,lat)):probes.append((lon,lat))
                trans=Transformer.from_crs('EPSG:4326',ds.crs,always_xy=True)
                montage=[]
                for j,(lon,lat) in enumerate(probes[:AUDIT_WINDOWS]):
                    x,y=trans.transform(lon,lat)
                    h=AUDIT_NATIVE_WINDOW_M/2
                    win=from_bounds(x-h,y-h,x+h,y+h,ds.transform)
                    # Boundless read represents TIFF area outside actual RGB coverage as black.
                    raw=ds.read(indexes=(1,2,3),window=win,
                        out_shape=(3,AUDIT_RGB_SAMPLE_PX,AUDIT_RGB_SAMPLE_PX),
                        resampling=Resampling.bilinear,boundless=True,fill_value=0)
                    a=raw.astype(np.float32)
                    if raw.dtype==np.uint8:a/=255.
                    else:
                        vals=a[np.isfinite(a)&(a>0)]
                        scale=np.percentile(vals,99.9) if len(vals)>100 else 255
                        a=np.clip(a/max(float(scale),1),0,1)
                    valid=np.isfinite(a).all(axis=0)&np.any(a>0,axis=0)
                    lum=(.2126*a[0]+.7152*a[1]+.0722*a[2])
                    values=lum[valid]
                    frac=float(valid.mean())
                    if len(values)>25:
                        qs=np.percentile(values,[5,25,50,75,95])
                        dark=float((values<AUDIT_ISSUE_DARK_THRESHOLD).mean())
                        # robustly sensitive to boundaries/texture; not absolute image sharpness
                        gy,gx=np.gradient(lum)
                        texture=float(np.median(np.hypot(gx,gy)[valid]))
                        channel=np.median(a[:,valid],axis=1)
                    else:
                        qs=[np.nan]*5;dark=np.nan;texture=np.nan;channel=[np.nan]*3
                    info['sample_windows'].append(dict(window=j,lon=float(lon),lat=float(lat),
                        valid_fraction=frac,median_luminance=float(qs[2]),
                        luminance_p05=float(qs[0]),luminance_p25=float(qs[1]),
                        luminance_p75=float(qs[3]),luminance_p95=float(qs[4]),
                        very_dark_fraction=dark,edge_proxy=texture,
                        red_median=float(channel[0]),green_median=float(channel[1]),
                        blue_median=float(channel[2])))
                    if len(montage)<5:
                        thumb=PILImage.fromarray((np.clip(a.transpose(1,2,0),0,1)*255).astype(np.uint8))
                        thumb.thumbnail((AUDIT_THUMB_SIZE,AUDIT_THUMB_SIZE))
                        montage.append(thumb.copy())
                if montage:
                    sheet=PILImage.new('RGB',(AUDIT_THUMB_SIZE*len(montage),AUDIT_THUMB_SIZE),'white')
                    for jj,img in enumerate(montage):sheet.paste(img,(jj*AUDIT_THUMB_SIZE,0))
                    sheet.save(THUMBS/(pid+'_samples.jpg'),quality=80)
    except Exception as ex:
        info['errors'].append(str(ex)[:400])
    file.write_text(json.dumps(info,indent=1))
    records.append(info)
    if i%12==0:print('Audited',i,'/',len(selection))

strip_rows=[];win_rows=[];meta_rows=[]
for r in records:
    p=r['plot_id']
    d=dict(plot_id=p,rgb_path=r.get('source_uri',''),
        n_sample_windows=len(r.get('sample_windows',[])),
        acquisition_date_verified=False,acquisition_time_verified=False,
        has_candidate_metadata=bool(r.get('tag_evidence') or r.get('supporting_files')),
        has_overviews=r.get('has_overviews'),crs=r.get('crs'),
        error='; '.join(r.get('errors',[])))
    for w in r.get('sample_windows',[]):win_rows.append(dict(plot_id=p,**w))
    for m in r.get('tag_evidence',[]):
        meta_rows.append(dict(plot_id=p,source='rgb_tiff_tag',**m))
    for path in r.get('supporting_files',[]):
        meta_rows.append(dict(plot_id=p,source='related_object_filename',namespace='',key='path',
                              value=path,acquisition_verified=False))
    for key in ['valid_fraction','median_luminance','very_dark_fraction','edge_proxy']:
        values=[w[key] for w in r.get('sample_windows',[]) if np.isfinite(w.get(key,np.nan))]
        d[key+'_median_sample']=float(np.median(values)) if values else np.nan
        d[key+'_min_sample']=float(np.min(values)) if values else np.nan
        d[key+'_max_sample']=float(np.max(values)) if values else np.nan
    d['brightness_spread_sample']=(d['median_luminance_max_sample']-
        d['median_luminance_min_sample']) if pd.notna(d['median_luminance_min_sample']) else np.nan
    # Review flags are intentionally descriptive, NOT a cloud-shadow classifier.
    d['review_dark_illumination']=bool(d['very_dark_fraction_max_sample']>.45) if pd.notna(d['very_dark_fraction_max_sample']) else False
    d['review_illumination_variability']=bool(d['brightness_spread_sample']>.22) if pd.notna(d['brightness_spread_sample']) else False
    d['review_low_rgb_coverage']=bool(d['valid_fraction_min_sample']<AUDIT_LOW_VALID_THRESHOLD) if pd.notna(d['valid_fraction_min_sample']) else True
    d['review_required']=bool(d['error'] or d['review_dark_illumination'] or
         d['review_illumination_variability'] or d['review_low_rgb_coverage'])
    strip_rows.append(d)
summary=pd.DataFrame(strip_rows)
windows=pd.DataFrame(win_rows)
evidence=pd.DataFrame(meta_rows)
summary.to_csv(AUDIT/'WWF_216_strip_RGB_quality_audit.csv',index=False)
windows.to_csv(AUDIT/'WWF_RGB_audit_window_metrics.csv',index=False)
evidence.to_csv(AUDIT/'WWF_acquisition_metadata_evidence_UNVERIFIED.csv',index=False)
print('Audited strips:',len(summary),'of',216,
      '| sampled windows:',len(windows),'| flagged for visual review:',summary.review_required.sum())
print('No acquisition date or time was inferred from TIFF processing dates or GCS timestamps.')
display(summary[['plot_id','n_sample_windows','median_luminance_median_sample',
 'very_dark_fraction_max_sample','brightness_spread_sample','review_required','error']].head(20))
print('Results:',AUDIT)


### 2.E — **RGB audit synthesis and review atlas**

Summarize the 216-strip quality metrics, map review flags geographically and generate contact sheets from already cached five-window strip previews. **Do not interpret these automated heuristics as verified cloud-shadow masks.** Their role is to prioritize manual review and decide which strips need more local candidate-circle checks.

The preliminary **G.6 V1 circle selection remains unchanged**. A later V2 selection should use verified local annotations of cloud shadows/poor visibility, not solely median brightness.

In [ ]:
# 2.E.1 — REVIEW ALL-STRIP RGB QC; NO GCS READS OR REPROCESSING
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display,Image as IPImage
BASE=Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
AUDIT=BASE/'Final'/'Outputs'/'V3_1'/'NB08_WWF_Lidar'/'RGB_quality_audit'
QC=AUDIT/'WWF_216_strip_RGB_quality_audit.csv'
assert QC.exists(),'Run 2.D.1 first'
q=pd.read_csv(QC)
footprints=plots.to_crs(4326)
joined=footprints.merge(q,on='plot_id',how='inner',validate='one_to_one')
assert len(joined)==len(q),'Footprint alignment problem: verify plot IDs'
print('WWF QUALITY AUDIT')
display(pd.DataFrame({
    'metric':['Strips audited','Windows sampled',
       'Strips requiring visual review (heuristic)','Strips with read errors',
       'Strips with verified acquisition time'],
    'value':[len(q),int(q.n_sample_windows.sum()),
      int(q.review_required.sum()),int(q.error.notna().sum()),
      int(q.acquisition_time_verified.sum())]
}))
print('Important: zero verified timestamps means NOT YET VERIFIED, not that metadata are absent.')
fig,axs=plt.subplots(1,2,figsize=(15,6))
joined.plot(ax=axs[0],column='median_luminance_median_sample',cmap='viridis',
            legend=True,legend_kwds={'label':'Median sampled brightness (0–1)'})
axs[0].set_title('WWF survey: sampled RGB brightness')
joined.plot(ax=axs[1],column='review_required',categorical=True,cmap='coolwarm',
            legend=True)
axs[1].set_title('Review flags (not cloud-shadow labels)')
for ax in axs:ax.set(xlabel='Longitude',ylabel='Latitude')
fig.tight_layout()
fig.savefig(AUDIT/'WWF_RGB_QC_strip_overview.png',dpi=140)
plt.close(fig)
# Contact sheet: one 5-window strip montage per survey plot, for rapid inspection.
from PIL import ImageDraw
thumbdir=AUDIT/'sample_previews'
ids=q.sort_values(['review_required','plot_id'],ascending=[False,True]).plot_id.tolist()
perpage=24
for start in range(0,len(ids),perpage):
    chosen=ids[start:start+perpage]
    sheet=Image.new('RGB',(1000,150*len(chosen)),color='white')
    draw=ImageDraw.Draw(sheet)
    for j,pid in enumerate(chosen):
        p=thumbdir/(pid+'_samples.jpg')
        if p.exists():
            with Image.open(p) as im:
                im=im.convert('RGB')
                im.thumbnail((825,135))
                sheet.paste(im,(175,150*j))
        draw.text((7,150*j+10),pid,fill=(0,0,0))
        flag=q.set_index('plot_id').loc[pid,'review_required']
        draw.text((7,150*j+30),'REVIEW' if flag else 'no auto flag',
                  fill=(175,40,20) if flag else (40,100,60))
    path=AUDIT/f'WWF_RGB_review_montage_{start//perpage+1:02d}.jpg'
    sheet.save(path,quality=85)
print('Saved map and review montages under:',AUDIT)
print('All G.6 V1 candidate circles are preserved. No masking/exclusions performed.')


### 2.G — Quick DSM cast-shadow pilot: low / intermediate / high solar zenith

Automatically choose **one single-day WWF strip near the low, median and high solar-zenith quantiles** (restricted to plots with available DSM), then retrieve **only three small RGB crops** and their matching 2 m DSM windows. For each strip, display: **RGB**, **DSM canopy surface**, **predicted cast shadows**, and **unexplained RGB darkness**.

Shadow simulation ray-traces neighbouring DSM heights along the solar azimuth from the acquisition timestamp. It uses an ~60 × 60 m display crop with a 100 m extra DSM margin to capture external shadow casters. **Solar geometry is a plot-level flight-time proxy**, so exact agreement is not guaranteed for every orthomosaic pixel; ray tracing also omits clouds, multiple scattering, exposure settings and small crown details.

**Cloud-shadow screening:** dark pixels not predicted to lie in cast shadows are marked as *candidate unexplained darkness*. They may be cloud shadows, but may also be mosaicking seams, naturally dark forest or DSM misregistration. This is **not** a validated cloud classifier. The block preserves the original RGB, never modifies the original 114 sampling circles, and saves a comparison figure/CSV under `RGB_quality_audit/DSM_shadow_pilot/`.

**Run after:** 1.B (GCS authentication) and 2.F.3 (saved solar geometry table). The inventory is loaded from disk if missing; there is no need to rerun 2.D, 2.E, 2.F.2, or the ecological analysis.

In [ ]:
# 2.G.1 — QUICK low/median/high solar-zenith RGB–DSM cast-shadow pilot
# 3 tiny native RGB windows, 3 bounded 2-m DSM windows; no full mosaic downloads.
from pathlib import Path
import os,re,math
import numpy as np,pandas as pd,matplotlib.pyplot as plt, rasterio
from rasterio.windows import from_bounds
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from scipy.ndimage import map_coordinates, label
from pyproj import Transformer
from IPython.display import display
import google.auth
from google.auth.transport.requests import Request

BASE=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
OUT=BASE/'RGB_quality_audit'/'DSM_shadow_pilot'
OUT.mkdir(parents=True,exist_ok=True)
SOLS=BASE/'RGB_quality_audit'/'acquisition_timing'/'WWF_RGB_solar_geometry_by_plot.csv'
assert SOLS.exists(),'Run 2.F.3 once; solar-geometry CSV not found'
solar=pd.read_csv(SOLS)
if 'df' not in globals():df=pd.read_csv(BASE/'carbon_maps_inventory.csv')
if 'plots' not in globals():
    import geopandas as gpd
    plots=gpd.read_file(BASE/'wwf_lidar_plots.gpkg').to_crs('EPSG:4326')
BUCKET='carbon-maps'
# Restrict to 3 micro-crops; a DSM buffer is necessary to cast shadows into the crop.
CROP_M=60
DSM_MARGIN_M=105
DSM_STEP_M=2
RGB_OUTPUT_PX=600
SUN_RAY_MAX_M=100
DARK_RELATIVE_TO_MEDIAN=0.48  # exploratory local brightness cut, not cloud threshold
DARK_PATCH_MIN_PIXELS=9       # in 2 m DSM grid (~36 m2), still not a cloud label
RESELECT_PLOTS=False         # True to alter selections below
MANUAL_PLOTS=None            # optional 3 plot IDs ordered low/mid/high zenith

credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
if not credentials.valid or not credentials.token:credentials.refresh(Request())
token=str(credentials.token)
inv=df.copy();inv['path']=inv.path.astype(str)
rgb=inv[inv.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',
                             case=False,na=False)].drop_duplicates('plot_id').set_index('plot_id').path
dsm=inv[inv.path.str.contains(r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
                             case=False,regex=True,na=False)].copy()
print('DSM inventory objects:',len(dsm))
# The inventory plot_id is derived from filenames in 2.A; retain only valid matches.
dsm=dsm.dropna(subset=['plot_id']).copy()
dsm['plot_id']=dsm.plot_id.astype(str)
dsm_map=dsm.groupby('plot_id').path.apply(list).to_dict()
eligible=solar[solar.plot_id.isin(rgb.index)&solar.plot_id.isin(dsm_map)].copy()
eligible=eligible[eligible.solar_elevation_deg.between(5,90)].sort_values('solar_zenith_deg')
assert len(eligible)>=3,'Fewer than 3 plots have both solar geometry and DSM products; inspect DSM filenames.'
qs=[.10,.50,.90]
if MANUAL_PLOTS:
    selected=eligible.set_index('plot_id').loc[MANUAL_PLOTS].reset_index()
else:
    targets=[eligible.solar_zenith_deg.quantile(q) for q in qs]
    chosen=[];used=set()
    for v in targets:
        options=eligible.assign(dist=(eligible.solar_zenith_deg-v).abs()).sort_values('dist')
        pid=next(p for p in options.plot_id if p not in used)
        chosen.append(eligible[eligible.plot_id.eq(pid)].iloc[0])
        used.add(pid)
    selected=pd.DataFrame(chosen)
selected['zenith_category']=['Low','Intermediate','High']
display(selected[['plot_id','zenith_category','solar_zenith_deg','solar_elevation_deg',
                  'solar_azimuth_deg','UTC_flight_time_proxy']])
# Obtain crop centres from survey polygons (not the global RGB rectangle).
foot=plots.to_crs('EPSG:4326').drop_duplicates('plot_id').set_index('plot_id')
def crop_center(pid,src):
    poly=foot.loc[pid,'geometry']
    ll=poly.representative_point()
    transform=Transformer.from_crs('EPSG:4326',src.crs,always_xy=True)
    cx,cy=transform.transform(ll.x,ll.y)
    # representative_point is within polygon, but outside RGB raster bounds in rare cases.
    if not(src.bounds.left+CROP_M/2<cx<src.bounds.right-CROP_M/2 and
           src.bounds.bottom+CROP_M/2<cy<src.bounds.top-CROP_M/2):
        cx=(src.bounds.left+src.bounds.right)/2
        cy=(src.bounds.bottom+src.bounds.top)/2
    return cx,cy
def ray_cast(dsm_grid,elev_deg,azimuth_deg,res_m,max_distance_m=100):
    """True if another DSM surface blocks sunlight. Solar azimuth: clockwise from north."""
    a=np.asarray(dsm_grid,dtype=float)
    ny,nx=a.shape
    rr,cc=np.indices(a.shape,dtype=np.float32)
    out=np.zeros(a.shape,dtype=bool)
    valid=np.isfinite(a)
    elev_tan=math.tan(math.radians(float(elev_deg)))
    az=math.radians(float(azimuth_deg))
    # Evaluate DSM heights in the sunward direction at successive distances.
    for distance in np.arange(res_m,max_distance_m+res_m,res_m):
        dr=-distance*math.cos(az)/res_m  # rows increase south
        dc=distance*math.sin(az)/res_m   # columns increase east
        z=map_coordinates(a,[rr+dr,cc+dc],order=1,mode='constant',cval=np.nan)
        out|=(valid&np.isfinite(z)&(z>a+distance*elev_tan))
    return out & valid

results=[]
fig,axs=plt.subplots(3,4,figsize=(17,12),constrained_layout=True)
for row,rec in enumerate(selected.itertuples(index=False)):
    pid=rec.plot_id
    rgb_path=rgb[pid]
    dsm_path=dsm_map[pid][0]
    try:
        with rasterio.Env(GS_OAUTH2_ACCESS_TOKEN=token,GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rgb_path) as sr:
                cx,cy=crop_center(pid,sr)
                # Fixed physical dimensions, bounded RGB window at approximately native image scale.
                extent=(cx-CROP_M/2,cy-CROP_M/2,cx+CROP_M/2,cy+CROP_M/2)
                w=from_bounds(*extent,transform=sr.transform)
                rgb_arr=sr.read(indexes=[1,2,3],window=w,
                    out_shape=(3,RGB_OUTPUT_PX,RGB_OUTPUT_PX),
                    boundless=True,fill_value=0,resampling=Resampling.bilinear)
                # RGB orthos are usually uint8. For non-uint8 samples, avoid untracked stretching.
                if rgb_arr.dtype==np.uint8: rgb_image=rgb_arr.transpose(1,2,0)/255.
                else:
                    val=rgb_arr.astype('float32')
                    denom=65535.0 if rgb_arr.dtype==np.uint16 else max(1.0,float(np.nanmax(val)))
                    rgb_image=np.clip(val.transpose(1,2,0)/denom,0,1)
                valid_rgb=np.all(np.isfinite(rgb_image),axis=2)&np.any(rgb_image>0,axis=2)
                with rasterio.open('/vsigs/'+BUCKET+'/'+dsm_path) as ds:
                    size=CROP_M+2*DSM_MARGIN_M
                    n=int(np.ceil(size/DSM_STEP_M))
                    grid_transform=from_origin(cx-size/2,cy+size/2,DSM_STEP_M,DSM_STEP_M)
                    heights=np.full((n,n),np.nan,dtype=np.float32)
                    # Target projection is exactly the RGB CRS; DSM correctly reprojected.
                    reproject(source=rasterio.band(ds,1),destination=heights,
                              dst_transform=grid_transform,dst_crs=sr.crs,
                              src_nodata=ds.nodata,dst_nodata=np.nan,
                              resampling=Resampling.bilinear,
                              warp_mem_limit=32)
                cast=ray_cast(heights,rec.solar_elevation_deg,
                              rec.solar_azimuth_deg,DSM_STEP_M,SUN_RAY_MAX_M)
                edge=int(round(DSM_MARGIN_M/DSM_STEP_M))
                inner=int(round(CROP_M/DSM_STEP_M))
                dsm_crop=heights[edge:edge+inner,edge:edge+inner]
                shadow_crop=cast[edge:edge+inner,edge:edge+inner]
                # Convert RGB down to DSM grid with simple averaging for visual assessment.
                # RGB is 600 px; DSM display grid 30 px -> 20x20 source blocks.
                factor=RGB_OUTPUT_PX//inner
                a=rgb_image[:inner*factor,:inner*factor,:].reshape(
                    inner,factor,inner,factor,3).mean(axis=(1,3))
                valid=valid_rgb[:inner*factor,:inner*factor].reshape(
                    inner,factor,inner,factor).mean(axis=(1,3))>.7
                lum=.2126*a[:,:,0]+.7152*a[:,:,1]+.0722*a[:,:,2]
                ref=np.median(lum[valid]) if valid.any() else 0
                observed_dark=(lum < ref*DARK_RELATIVE_TO_MEDIAN)&valid
                unexplained=observed_dark&~shadow_crop&np.isfinite(dsm_crop)
                objects,nobj=label(unexplained)
                retained=np.zeros_like(unexplained)
                for k in range(1,nobj+1):
                    region=(objects==k)
                    if region.sum()>=DARK_PATCH_MIN_PIXELS:retained|=region
                results.append(dict(plot_id=pid,zenith_category=rec.zenith_category,
                    solar_zenith_deg=rec.solar_zenith_deg,
                    solar_azimuth_deg=rec.solar_azimuth_deg,
                    dsm_path=dsm_path,rgb_path=rgb_path,
                    valid_grid_fraction=float(valid.mean()),
                    predicted_cast_shadow_fraction=float(shadow_crop[np.isfinite(dsm_crop)].mean()),
                    rgb_dark_fraction=float(observed_dark[valid].mean()) if valid.any() else np.nan,
                    unexplained_dark_fraction=float(retained[valid].mean()) if valid.any() else np.nan,
                    note='Unexplained darkness is NOT a validated cloud-shadow mask'))
                axs[row,0].imshow(rgb_image);axs[row,0].set_title('Original 10 cm RGB')
                axs[row,1].imshow(dsm_crop,cmap='terrain');axs[row,1].set_title('DSM (2 m)')
                axs[row,2].imshow(shadow_crop,cmap='gray_r',vmin=0,vmax=1)
                axs[row,2].set_title('Predicted cast shadow')
                axs[row,3].imshow(a)
                axs[row,3].imshow(np.ma.masked_where(~retained,retained),cmap='autumn',alpha=.65,
                                   interpolation='nearest',vmin=0,vmax=1)
                axs[row,3].set_title('Unexplained dark patches (orange)')
    except Exception as exc:
        results.append(dict(plot_id=pid,zenith_category=rec.zenith_category,
                            error=repr(exc)[:250]))
        for ax in axs[row]:ax.text(.05,.5,repr(exc)[:65],transform=ax.transAxes,wrap=True)
    for ax in axs[row]:ax.set_xticks([]);ax.set_yticks([])
    axs[row,0].set_ylabel(
        f'{rec.zenith_category}\n{pid}\nZenith {rec.solar_zenith_deg:.1f}°',
        rotation=0,labelpad=90,va='center')
plt.suptitle('WWF RGB / LiDAR: DSM cast-shadow pilot (60 × 60 m crops)',fontsize=14)
path=OUT/'WWF_RGB_DSM_shadow_low_mid_high_zenith.png'
fig.savefig(path,dpi=175,bbox_inches='tight')
plt.show();plt.close(fig)
stats=pd.DataFrame(results)
stats.to_csv(OUT/'WWF_DSM_shadow_pilot_metrics.csv',index=False)
display(stats)
print('Saved:',path)
print('Important: orange regions are candidates for VISUAL review, not confirmed cloud shadows.')


### Stage 1D — Import the 100 completed circle annotations
**Sunny or uniformly overcast is usable** unless cloud-shadowed. Mixed or moderate/severe cloud shadow is flagged. Minor cloud effects need review. Crown shadow is not a cloud exclusion.

In [ ]:
# Stage 1D — Load annotated WWF 114-circle workbook (no imagery downloaded)
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
QCDIR=ROOT/'RGB_quality_audit'
QCDIR.mkdir(parents=True,exist_ok=True)
book=ROOT/'WWF_114_circle_visual_quality_review.xlsx'
assert book.exists(),f'Missing annotated Excel file: {book}'
review=pd.read_excel(book,sheet_name='Circle review')
expected=['Circle_ID','Illumination','Cloud_shadow','Crown_shadow','Annotation_quality']
assert all(x in review for x in expected),f'Expected {expected}; got {list(review)}'
assert len(review)==114 and review.Circle_ID.is_unique
def code(v):
    return str(v).split(' - ',1)[0].strip().upper() if pd.notna(v) else ''
for old,new in [('Illumination','illum'),('Cloud_shadow','cloud'),
                ('Crown_shadow','crown'),('Annotation_quality','annotation')]:
    review[new]=review[old].map(code)
review['reviewed']=review[['illum','cloud','crown','annotation']].ne('').all(axis=1)
def assess(r):
    if not r.reviewed:return 'NEEDS_REVIEW'
    if r.illum=='M' or r.cloud in ('2','3'):return 'AVOID_CLOUD'
    if r.illum=='U' or r.cloud=='1':return 'CHECK_MINOR_CLOUD'
    if r.illum in ('S','O') and r.cloud=='0':return 'USABLE_WEATHER'
    return 'NEEDS_REVIEW'
review['weather_screen']=review.apply(assess,axis=1)
review=review.rename(columns={'Circle_ID':'circle_id'})
review.to_csv(QCDIR/'WWF_G6_manual_cloud_illumination_review.csv',index=False)
print('Fully annotated:',int(review.reviewed.sum()),'of',len(review))
display(pd.crosstab(review.Illumination.fillna('Unreviewed'),review.weather_screen,margins=True))
display(review.weather_screen.value_counts().rename_axis('weather_screen').to_frame('circles'))
print('Note: crown shade does NOT imply cloud contamination; clean overcast remains eligible.')


### Stage 1D.2 — Can existing audit metrics predict cloud contamination?

**Lightweight first validation**, using the **existing 216-strip RGB audit**, your circle labels, and available single-day solar geometry. Train and evaluate with WWF **strip-grouped cross-validation**, not random circles (multiple circles from one strip are not independent). Solar angle is *not* a reliable cloud predictor by itself, so test RGB-only vs RGB+solar.

**Important limits:** existing five RGB windows per plot were not sampled at the annotation circles. This is a **strip-proxy feasibility experiment**, not a trained 125 m/pixel cloud mask. If recall/precision are poor or too few labels support meaningful cross-validation, refrain from automatic rejection; instead use the output for manual review prioritization. This block makes **no new GCS reads** and does not override the user’s review labels. A real all-216 cloud classifier will need circle-matched RGB/DSM feature extraction and independently checked strip-level validation.

In [ ]:
# Stage 1D.2 — First grouped-CV test of cloud issues from EXISTING strip audit CSV
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,classification_report,balanced_accuracy_score
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
QA=ROOT/'RGB_quality_audit'
G6=ROOT/'TMF_strip_descriptors'/'G6_125m_sampling_design'
lab=pd.read_csv(QA/'WWF_G6_manual_cloud_illumination_review.csv')
cir=pd.read_csv(G6/'WWF_G6_provisional_125m_circles.csv',usecols=['circle_id','plot_id'])
q=pd.read_csv(QA/'WWF_216_strip_RGB_quality_audit.csv')
train=cir.merge(lab[['circle_id','weather_screen']],on='circle_id').merge(q,on='plot_id')
# Exclude minor/ambiguous and missing labels from binary training.
train=train[train.weather_screen.isin(['AVOID_CLOUD','USABLE_WEATHER'])].copy()
train['cloud_issue']=(train.weather_screen=='AVOID_CLOUD').astype(int)
features=[c for c in [
 'median_luminance_median_sample','median_luminance_min_sample',
 'median_luminance_max_sample','very_dark_fraction_max_sample',
 'brightness_spread_sample','edge_proxy_median_sample',
 'valid_fraction_min_sample'] if c in train and pd.api.types.is_numeric_dtype(train[c])]
assert len(features)>=2,f'Not enough numeric RGB audit features: {features}'
solar=QA/'acquisition_timing'/'WWF_RGB_solar_geometry_by_plot.csv'
if solar.exists():
    s=pd.read_csv(solar)
    train=train.merge(s[['plot_id','solar_zenith_deg']].drop_duplicates('plot_id'),on='plot_id',how='left')
    q=q.merge(s[['plot_id','solar_zenith_deg']].drop_duplicates('plot_id'),on='plot_id',how='left')
print('Independent labelled plots:',train.plot_id.nunique(),
      'positive cloud/mixed circles:',int(train.cloud_issue.sum()),'total train circles:',len(train))
def validate(cols):
    z=train.copy()
    grouped=z.groupby('plot_id').cloud_issue.agg(['min','max'])
    # Avoid ambiguous plot labels: one strip should not be both clear and cloudy
    # at circle level in a strip-level proxy model. Keep these for spatial follow-up.
    ambiguous=set(grouped[grouped['min']!=grouped['max']].index)
    z=z[~z.plot_id.isin(ambiguous)].copy()
    z=z.drop_duplicates('plot_id')
    counts=z.cloud_issue.value_counts()
    if len(counts)<2 or counts.min()<4 or len(z)<12:
        print('INSUFFICIENT independent strip-level examples to validate',cols,
              '| class counts:',counts.to_dict(),'| mixed-label plots:',len(ambiguous))
        return None
    folds=min(5,int(counts.min()))
    model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),
                        LogisticRegression(max_iter=1500,class_weight='balanced'))
    cv=StratifiedGroupKFold(n_splits=folds,shuffle=True,random_state=20261029)
    pred=cross_val_predict(model,z[cols],z.cloud_issue,groups=z.plot_id,cv=cv,method='predict_proba')[:,1]
    cls=(pred>=0.5).astype(int)
    cm=confusion_matrix(z.cloud_issue,cls,labels=[0,1])
    print('\nFeatures:',cols,'| plots:',len(z),'| mixed-label plots excluded:',len(ambiguous))
    print('Confusion matrix [[true usable, false alarm],[missed cloud, caught cloud]]:\n',cm)
    print(classification_report(z.cloud_issue,cls,labels=[0,1],zero_division=0))
    scored=z[['plot_id','cloud_issue']].copy();scored['heldout_cloud_probability']=pred
    return scored,model,z
rgb_res=validate(features)
solar_res=validate(features+['solar_zenith_deg']) if 'solar_zenith_deg' in train else None
if rgb_res:
    score,model,z=rgb_res
    score.to_csv(QA/'WWF_cloud_issue_strip_heldout_validation_RGB_only.csv',index=False)
    # Fit for exploratory *review prioritization* only, NOT automatic cloud masking.
    model.fit(z[features],z.cloud_issue)
    q['cloud_review_priority_score_UNVALIDATED']=model.predict_proba(q[features])[:,1]
    q[['plot_id','cloud_review_priority_score_UNVALIDATED']].sort_values(
        'cloud_review_priority_score_UNVALIDATED',ascending=False).to_csv(
        QA/'WWF_216_strip_cloud_review_priority_UNVALIDATED.csv',index=False)
    print('216-strip scores saved for HUMAN REVIEW ONLY, NOT an exclusion mask.')
print('Model assessment must precede any attempt to reject/mask all-216 WWF imagery.')


### Stage 1E — Optional historic RGB feasibility checks
Retained for reference; not required for the systematic all-strip survey audit.

## 3 — RGB image quality and feasibility
Measure pixel size **from raster transforms**, not nominal filenames. Report footprint dimensions, band count, CRS, overviews, block layout and color interpretation. Avoid full-resolution reads.

In [ ]:
# 3.A — Read one RGB mosaic header
import rasterio
import pandas as pd
from rasterio.crs import CRS
from pyproj import Geod
sample_path='001_Lidar_Rasters/RGB_mosaic/UTM33S_Plot135_RGB.tif'
uri='/vsigs/'+BUCKET+'/'+sample_path
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
    with rasterio.open(uri) as src:
        print('File:',sample_path)
        print('CRS:',src.crs)
        print('Width × height:',src.width,'×',src.height)
        print('Bands:',src.count, 'Dtypes:',src.dtypes, 'Color interpretation:',[v.name for v in src.colorinterp])
        print('Pixel size (native CRS units):',src.res)
        print('GSD in cm if CRS projected in metres:',tuple(round(100*abs(v),2) for v in src.res) if src.crs and src.crs.is_projected and src.crs.linear_units=='metre' else 'requires CRS unit conversion')
        print('Extent:',src.bounds)
        print('Internal overviews:',src.overviews(1))
        print('Storage layout:',src.profile.get('tiled'), 'Block shapes:',src.block_shapes)
        print('Compression:',src.profile.get('compress'),'Nodata:',src.nodata)
        print('Pixel count:',f'{src.width*src.height:,}')
print('NOTE: Missing overviews — NEVER downsample entire full-resolution TIFF for an overview.')

### 3.B — RGB header audit across UTM zones
Start with **6 mosaics from different UTM zones** (modifiable), not all 216. Remote metadata access can still incur GCS requests. This saves a QC table with resolution in centimetres and flags missing overviews.

In [ ]:
# 3.B.1 — Sample native GSD across UTM zones
import math, pandas as pd, rasterio
MAX_RGB_METADATA=6
# One example per UTM zone; choose more later when working
rgb_paths=(df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)]
           .copy())
rgb_paths['utm']=rgb_paths.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
pick=rgb_paths.groupby('utm',sort=True).head(1).head(MAX_RGB_METADATA)
metadata_rows=[]
for rec in pick.itertuples(index=False):
    row=dict(plot_id=rec.plot_id,path=rec.path,size_GB=rec.size_GB)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rec.path) as s:
                row.update(crs=str(s.crs),width=s.width,height=s.height,
                           bands=s.count,gsd_x_native=s.res[0],gsd_y_native=s.res[1],
                           overviews=len(s.overviews(1)),tiled=s.profile.get('tiled'),
                           compression=s.profile.get('compress'),
                           block_x=s.block_shapes[0][1],block_y=s.block_shapes[0][0])
                if s.crs and s.crs.is_projected and s.crs.linear_units=='metre':
                    row.update(gsd_cm_x=100*abs(s.res[0]),gsd_cm_y=100*abs(s.res[1]))
                else:
                    row.update(gsd_cm_x=float('nan'),gsd_cm_y=float('nan'))
    except Exception as e:
        row['error']=str(e)[:250]
    metadata_rows.append(row)
meta=pd.DataFrame(metadata_rows)
meta.to_csv(WORK_DIR/'wwf_rgb_metadata_audit.csv',index=False)
display(meta)
print('Saved:',WORK_DIR/'wwf_rgb_metadata_audit.csv')

### 3.C — Native-resolution RGB image inspection
The main annotation decision requires individual tree crown visibility. **Do not use `src.read(out_shape=(3,650,650))` on an entire 13 km mosaic without overviews.** Instead, read **small native-pixel windows** (40 × 40 m at 10 cm = 400 × 400 pixels). Inspect a few different locations within each mosaic; the default is 4 small windows on the first pilot mosaic.

Note: a large striped TIFF can cause some extra data transfer even for a small window; inspect block shapes and monitor network usage.

In [ ]:
# 3.C.1 — Inspect RGB pixel quality
# Important: zero-valued margins are common in orthomosaics, even without nodata metadata.
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from pathlib import Path
from rasterio.plot import reshape_as_image

QC_PLOT_ID = 'UTM33S_Plot135'
QC_WINDOW_METRES = 40
QC_MAX_WINDOWS = 4
QC_MAX_PROBES = 30  # bounded window reads only; no full-mosaic scan
QC_VALID_THRESHOLD = 0.80  # require at least 80% nonblack pixels
# RGB/BGR ambiguity is diagnosed by displaying both on the first selected patch.
source_rows = df[df.path.str.fullmatch(
    r'001_Lidar_Rasters/RGB_mosaic/' + QC_PLOT_ID + r'_RGB\.tif',
    case=False, na=False)]
assert len(source_rows) == 1, f'Expected exactly 1 RGB mosaic, got {len(source_rows)}'
qc_path = source_rows.iloc[0].path
preview_dir = WORK_DIR / 'RGB_QC_native_windows_v2'
preview_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(8108)

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
    with rasterio.open('/vsigs/' + BUCKET + '/' + qc_path) as src:
        assert src.crs and src.crs.is_projected and src.crs.linear_units == 'metre'
        assert src.count >= 3
        W = min(src.width, max(1, round(QC_WINDOW_METRES / abs(src.res[0]))))
        H = min(src.height, max(1, round(QC_WINDOW_METRES / abs(src.res[1]))))
        print('Raster:',qc_path)
        print('Native GSD (cm):',100*abs(src.res[0]),100*abs(src.res[1]))
        print('Band colors:',[c.name for c in src.colorinterp])
        print('Bands dtype:',src.dtypes,'Nodata:',src.nodata)
        print('Block shapes:',src.block_shapes,'Overviews:',src.overviews(1))
        print('Scanning up to',QC_MAX_PROBES,'small windows for nonblack coverage...')
        candidates = [(0.5,0.5),(0.35,0.5),(0.65,0.5),
                      (0.5,0.35),(0.5,0.65),
                      (0.25,0.25),(0.75,0.25),(0.25,0.75),(0.75,0.75)]
        candidates += list(map(tuple,rng.uniform(0.1,0.9,(QC_MAX_PROBES,2))))
        keep = []
        checked = []
        for fx, fy in candidates[:QC_MAX_PROBES]:
            col = int(np.clip(round(fx*src.width-W/2),0,src.width-W))
            row = int(np.clip(round(fy*src.height-H/2),0,src.height-H))
            win = Window(col,row,W,H)
            a = src.read(indexes=[1,2,3],window=win,masked=True)
            # A pixel is usable only if all channels are valid AND not all zeros.
            data = np.ma.filled(a,0)
            valid_mask = (~np.ma.getmaskarray(a).any(axis=0)) & np.any(data != 0,axis=0)
            good = float(valid_mask.mean())
            checked.append((col,row,round(good,3)))
            if good >= QC_VALID_THRESHOLD:
                keep.append((win,data,valid_mask,good))
                if len(keep) >= QC_MAX_WINDOWS: break
        print('Window diagnostics (col,row,valid fraction):',checked)
        if not keep:
            print('No windows meet minimum coverage. Could be sparse flight footprint or nodata encoding.')
        for i,(win,data,valid,good) in enumerate(keep,1):
            # Native integer channels: retain relative R/G/B levels (no separate 2-98% per-band stretch).
            img = np.moveaxis(data.astype('float32'),0,-1)
            vals = img[valid]
            if np.issubdtype(np.dtype(src.dtypes[0]),np.integer):
                scale=float(np.iinfo(np.dtype(src.dtypes[0])).max)
                natural=np.clip(img/scale,0,1)
            else:
                # Shared stretch across all channels; preserves relative band colours.
                hi=np.percentile(vals,99) if vals.size else 1
                natural=np.clip(img/max(hi,1e-6),0,1)
            natural[~valid] = 0
            bgr=natural[:,:,::-1]
            if i==1:
                fig,axs=plt.subplots(1,2,figsize=(12,6))
                axs[0].imshow(natural,interpolation='nearest')
                axs[0].set_title('Original band order (1,2,3)')
                axs[1].imshow(bgr,interpolation='nearest')
                axs[1].set_title('Reversed order (3,2,1) — diagnostic only')
                for ax in axs:ax.axis('off')
                plt.show();plt.close(fig)
            fig,ax=plt.subplots(figsize=(7,7))
            ax.imshow(natural,interpolation='nearest')
            ax.set_title(f'{QC_PLOT_ID} | native {100*abs(src.res[0]):g} cm | valid {good:.1%} | window {i}')
            ax.axis('off')
            output=preview_dir/f'{QC_PLOT_ID}_window{i}.png'
            fig.savefig(output,dpi=160,bbox_inches='tight')
            plt.show();plt.close(fig)
            print('Saved:',output,'pixel window:',win)
print('Judge crown delineation ONLY using valid imagery and verified band order.')


### 3.D — Manual visual quality score
Record whether crowns can be delineated, not merely nominal pixel size. Evaluate blur, shadows, mosaicking seams, color balance and the visibility of small pioneer crowns. Save a simple screening table for each inspected sample.

In [ ]:
# 3.D.1 — Prepare manual quality scores
qc_form=pd.DataFrame([{
    'plot_id':'UTM33S_Plot135','gsd_cm':10.0,
    'crown_visibility_1to5':None,'sharpness_1to5':None,
    'shadow_problem_1to5':None,'seam_problem_1to5':None,
    'possible_musanga_visible':'UNCERTAIN',
    'forest_structure':'UNREVIEWED',
    'exhaustive_annotation_feasible':'UNREVIEWED',
    'notes':'Manual review required; no species inference from GSD alone'
}])
qc_form.to_csv(WORK_DIR/'wwf_rgb_visual_qc_template.csv',index=False)
display(qc_form)
print('QC template:',WORK_DIR/'wwf_rgb_visual_qc_template.csv')

### 3.E — Six-strip RGB feasibility atlas
Test a deliberately small sample across UTM zones. For each mosaic, inspect up to three **35 × 35 m native-resolution crops** (with at most 12 spatial probes). Export both original band order and reversed order, because the blue/yellow colours might reflect BGR interpretation. Results include a contact sheet, individual PNGs and a CSV containing exact raster-window bounds. This is a **visual screening step**, not a validated Musanga identification or forest age label. Some mosaics may have no valid crop among the few sampled positions: record that explicitly, do not label them poor-quality. The first run makes at most 72 small window reads; large striped TIFFs may still incur extra I/O.

In [ ]:
# 3.E.1 — Create native RGB feasibility atlas
# Output: 6 plots × 3 examples, individual PNGs, contact sheet, QC table.
# Zero full-mosaic reads and zero full TIFF downloads.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, rasterio
from rasterio.windows import Window, bounds as window_bounds
from pathlib import Path
from PIL import Image
from IPython.display import display
import warnings

ATLAS_PLOTS = [
    'UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
    'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104'
]
ATLAS_CROP_M = 35                # 350 px if GSD is 10 cm
ATLAS_EXAMPLES_PER_PLOT = 3
ATLAS_MAX_PROBES_PER_PLOT = 12  # cap remote reads if flight footprint has holes
ATLAS_MIN_VALID = 0.70
# NOTE: plot-wide 1-99% stretches are not used; the 8-bit original bands are retained.
atlas_dir = WORK_DIR/'RGB_annotation_feasibility_atlas'
atlas_dir.mkdir(parents=True,exist_ok=True)

def make_display(raw, mask, dtype):
    """Both original and reversed RGB channels; identical scaling in both."""
    array = np.moveaxis(raw.astype(np.float32), 0, -1)
    good = array[mask]
    if np.issubdtype(np.dtype(dtype), np.integer):
        max_value=float(np.iinfo(np.dtype(dtype)).max)
        original=np.clip(array/max_value,0,1)
    else:
        hi=float(np.nanpercentile(good,99.5)) if good.size else 1.
        original=np.clip(array/max(hi,1e-8),0,1)
    original[~mask]=0
    return original, original[:,:,::-1]

rows=[]
images=[]
for plot_id in ATLAS_PLOTS:
    selection=df[df.path.eq(f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif')]
    if len(selection)!=1:
        print('SKIP: missing mosaic',plot_id)
        continue
    uri='/vsigs/'+BUCKET+'/'+selection.iloc[0].path
    print('\nOPEN',plot_id,flush=True)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(uri) as ds:
                assert ds.crs and ds.crs.is_projected and ds.crs.linear_units=='metre'
                assert ds.count>=3
                nx=min(ds.width,max(1,int(round(ATLAS_CROP_M/abs(ds.res[0])))))
                ny=min(ds.height,max(1,int(round(ATLAS_CROP_M/abs(ds.res[1])))))
                print('GSD (cm):',tuple(round(abs(r)*100,2) for r in ds.res),
                      '| bands:',[c.name for c in ds.colorinterp],
                      '| overviews:',ds.overviews(1),'| blocks:',ds.block_shapes[0],flush=True)
                # Deterministic staggered probes, including central transect;
                # avoids sampling only the corners of a narrow flight swath.
                locations=[(.5,.5),(.25,.5),(.75,.5),(.5,.25),(.5,.75),
                           (.35,.35),(.65,.65),(.35,.65),(.65,.35),
                           (.15,.5),(.85,.5),(.5,.85)]
                n_saved=0
                for k,(fx,fy) in enumerate(locations[:ATLAS_MAX_PROBES_PER_PLOT]):
                    left=int(np.clip(round(fx*ds.width-nx/2),0,ds.width-nx))
                    top=int(np.clip(round(fy*ds.height-ny/2),0,ds.height-ny))
                    win=Window(left,top,nx,ny)
                    ma=ds.read(indexes=[1,2,3],window=win,masked=True)
                    raw=np.ma.filled(ma,0)
                    good=(~np.ma.getmaskarray(ma).any(axis=0)) & np.any(raw!=0,axis=0)
                    fraction=float(good.mean())
                    if fraction<ATLAS_MIN_VALID:
                        continue
                    original,reversed_rgb=make_display(raw,good,ds.dtypes[0])
                    n_saved+=1
                    rgb=(original*255).round().astype('uint8')
                    bgr=(reversed_rgb*255).round().astype('uint8')
                    stem=f'{plot_id}_crop{n_saved:02d}'
                    Image.fromarray(rgb).save(atlas_dir/f'{stem}_bands123.png')
                    Image.fromarray(bgr).save(atlas_dir/f'{stem}_bands321.png')
                    # Actual window bounding box for relocation; in source CRS
                    x0,y0,x1,y1=window_bounds(win,ds.transform)
                    rows.append(dict(plot_id=plot_id,crop=n_saved,probe=k+1,
                         x_min=x0,y_min=y0,x_max=x1,y_max=y1,
                         crs=str(ds.crs),gsd_cm_x=abs(ds.res[0])*100,
                         gsd_cm_y=abs(ds.res[1])*100,
                         valid_fraction=fraction,channel_order='UNVERIFIED',
                         crown_delineation='UNREVIEWED',musanga_identifiable='UNREVIEWED',
                         forest_environment='UNREVIEWED',notes=''))
                    images.append((stem,original,reversed_rgb,fraction))
                    print(f'  crop {n_saved}: probe {k+1}, valid {fraction:.1%}',flush=True)
                    if n_saved>=ATLAS_EXAMPLES_PER_PLOT:break
                if n_saved==0:
                    print('  WARNING: no valid crop among probed positions; not proof of bad imagery')
    except Exception as exc:
        print('ERROR inspecting',plot_id,type(exc).__name__,str(exc)[:400],flush=True)

if rows:
    table=pd.DataFrame(rows)
    table.to_csv(atlas_dir/'atlas_annotation_QC.csv',index=False)
    display(table[['plot_id','crop','gsd_cm_x','valid_fraction','channel_order',
                   'crown_delineation','musanga_identifiable']])
    # Contact sheet at modest display size only; native PNGs are retained.
    fig,axes=plt.subplots(len(images),2,figsize=(9,3.8*len(images)),squeeze=False)
    for j,(stem,a,b,valid) in enumerate(images):
        for col,img in enumerate((a,b)):
            axes[j,col].imshow(img,interpolation='nearest')
            axes[j,col].set_title(f'{stem} | '+('bands 1/2/3' if col==0 else 'bands 3/2/1'),fontsize=9)
            axes[j,col].axis('off')
    fig.suptitle('WWF RGB — native crops, identical channel scaling (colour order unverified)',fontsize=13)
    fig.tight_layout()
    fig.savefig(atlas_dir/'atlas_contact_sheet.jpg',dpi=110,bbox_inches='tight')
    display(fig);plt.close(fig)
    print('Saved contact sheet, individual native crops and QC:',atlas_dir)
else:
    print('No valid crop produced. Check nodata encoding and file layout; do NOT infer that RGB is bad.')


### 3.F — Geographic screening of RGB mosaics
Choose spatially dispersed candidate RGB mosaics as a starting list. **Spatial dispersion is not a proxy for forest age.** Classification into young secondary forest, regrowth, disturbed forest, mature canopy or open land requires image review and external land-cover history.

In [ ]:
# 3.F.1 — Screen candidate mosaics
import numpy as np, pandas as pd
from pathlib import PurePosixPath
rgb=df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)].copy()
geo=plots.copy()
cent=geo.geometry.representative_point()
geo['longitude']=cent.x;geo['latitude']=cent.y
catalog=geo[['plot_id','utm_zone','longitude','latitude','geometry']].drop(columns='geometry').merge(
    rgb[['plot_id','path','size_GB']].rename(columns={'path':'rgb_gcs_path','size_GB':'rgb_GB'}),
    how='inner',on='plot_id',validate='one_to_one')
catalog['visual_forest_class']='UNREVIEWED'
catalog['musanga_presence']='UNREVIEWED'
catalog['annotation_suitability']='UNREVIEWED'
catalog['notes']=''
catalog.to_csv(WORK_DIR/'wwf_rgb_annotation_catalog.csv',index=False)
print('RGB candidates:',len(catalog))
display(catalog.groupby('utm_zone').agg(n=('plot_id','size'),GB=('rgb_GB','sum')))
# Screen a geographically dispersed subset across all UTM zones.
# Farthest-point sampling on geographic coordinates within zones; not interpreted as age.
from pyproj import Geod
geod=Geod(ellps='WGS84')
chosen=[]
for zone, group in catalog.groupby('utm_zone'):
    target=min(5,len(group))
    pool=group.reset_index(drop=True)
    selected=[0]
    while len(selected)<target:
        distances=[]
        for j,rr in pool.iterrows():
            if j in selected: distances.append(-1);continue
            mind=min(geod.inv(rr.longitude,rr.latitude,
                              pool.iloc[k].longitude,pool.iloc[k].latitude)[2]
                     for k in selected)
            distances.append(mind)
        selected.append(int(np.argmax(distances)))
    chosen.append(pool.iloc[selected])
screen=pd.concat(chosen,ignore_index=True)
screen['review_priority']='INITIAL_SCREEN'
screen.to_csv(WORK_DIR/'wwf_rgb_initial_screen.csv',index=False)
print('Initial screening set:',len(screen),'(spatially dispersed, NOT preclassified)')
display(screen[['plot_id','utm_zone','latitude','longitude','rgb_GB']])

## Stage 2 — TMF × Hansen forest history and ecological stratification
Characterize all 216 WWF strips, reconstruct the integrated disturbance/recovery strata, then test 125 m neighbourhood feasibility. Reuse previously cached EE outputs.

## 4 — TMF forest-history characterization
Compare native GSD and observed RGB quality across zones; shortlist mosaics with visually annotatable crowns and a balanced gradient of forest environments, including Musanga-free exhaustive negatives. Expand the screening set only after reviewing the pilot true-resolution crops.

### 4.A — TMF status across all 216 survey strips

**Purpose:** characterize the **complete 216-strip survey**, before selecting further 125 m annotation circles. This uses the actual WWF survey footprints, not the six pilot strips, and is fully independent of UAV sites. The JRC TMF 30 m annual classification supplies *landscape-scale* evidence: proportions of undisturbed, degraded, deforested, regrowing, water, and other land cover in 2014 and 2015; fraction of the 2014–2015 landscape that changed class; and a conservative indicator of clearing already present in 2014 following forest in 2011/2012. The acquisition year is only known as 2014–2015, so no later-year conditions enter sampling descriptors.

**Interpretation:** class fractions describe historical TMF conditions **within the mapped transect polygon**, which is not necessarily the exact valid 10 cm RGB coverage. Deforested ≠ bare clearing, and regrowth ≠ known Musanga. A simple diversity index helps flag mixed landscapes; it is not a biological diversity measurement. All output values are computed server-side on Earth Engine, downloaded as small aggregated tables, then ranked and plotted. No large WWF imagery files are downloaded.

This cell can run after **C / cell 04** (once `plots` and `WORK_DIR` exist). It does **not** require H.1, H.2 or annotation circles.

In [ ]:
# 4.A.1 — Calculate TMF 2014–2015 distributions
# 2014/2015 composition, recent clearing, class-change, landscape diversity
import ee, geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import mapping
from pathlib import Path
from IPython.display import display
import math, time

EE_PROJECT = (globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine:', EE_PROJECT)

assert 'plots' in globals(), 'Run Section C / cell 04 to load WWF footprints'
assert 'plot_id' in plots.columns and plots.plot_id.is_unique
assert plots.crs is not None

TMF_STRIP_DIR = WORK_DIR / 'TMF_strip_descriptors'
TMF_STRIP_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE = 12    # small requests; keeps Colab responsive and allows resume
RECOMPUTE = False  # false: reuse completed batch CSVs on rerun
EE_SCALE_M = 30
LABELS = ['undisturbed','degraded','deforested','regrowth','water','other']
TMF = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
a11, a12, a14, a15 = [TMF.select('Dec'+str(y)) for y in [2011,2012,2014,2015]]
area_ha = ee.Image.pixelArea().divide(10000)
bands = [area_ha.rename('footprint_ha')]
for year, img in [(2014,a14),(2015,a15)]:
    for value, name in enumerate(LABELS, 1):
        bands.append(area_ha.multiply(img.eq(value).unmask(0)).rename(
            f'ha_{year}_{name}'))
# Describes pixels already deforested by 2014 that were forest in 2011 or 2012.
forest_pre = (a11.eq(1).Or(a11.eq(2))
              .Or(a12.eq(1)).Or(a12.eq(2)))
recent_clear = a14.eq(3).And(forest_pre).And(a15.eq(3))
bands += [
    area_ha.multiply(recent_clear.unmask(0)).rename('ha_recent_clear_pre2014'),
    area_ha.multiply(a14.neq(a15).And(a14.mask()).And(a15.mask()).unmask(0))
       .rename('ha_changed_class_2014_2015'),
]
descriptor_image = ee.Image.cat(bands).toFloat()

geo = plots[['plot_id','utm_zone','geometry']].copy().to_crs('EPSG:4326')
geo = geo.loc[geo.geometry.notna() & ~geo.geometry.is_empty].sort_values('plot_id')
# Use compact coordinates; spatial analysis is carried out using the original polygons.
print('Strips:',len(geo), '| EE scale:',EE_SCALE_M,'m')
partial_files = []
for start in range(0,len(geo),BATCH_SIZE):
    batch=geo.iloc[start:start+BATCH_SIZE]
    outfile=TMF_STRIP_DIR/f'tmf_strip_batch_{start:03d}_{start+len(batch):03d}.csv'
    partial_files.append(outfile)
    if outfile.exists() and not RECOMPUTE:
        print('Cached:',outfile.name)
        continue
    features=[]
    for rec in batch.itertuples(index=False):
        features.append(ee.Feature(
            ee.Geometry(mapping(rec.geometry),proj='EPSG:4326',geodesic=False),
            {'plot_id':str(rec.plot_id),'utm_zone':str(rec.utm_zone)}))
    collection=ee.FeatureCollection(features)
    result=descriptor_image.reduceRegions(
        collection=collection,
        reducer=ee.Reducer.sum(),
        scale=EE_SCALE_M,
        tileScale=4
    )
    raw=result.getInfo()['features']
    records=[{k:v for k,v in f['properties'].items()} for f in raw]
    out=pd.DataFrame(records)
    assert len(out)==len(batch),f'Unexpected row count: {len(out)} vs {len(batch)}'
    out.to_csv(outfile,index=False)
    print(f'Calculated {start+1}–{start+len(batch)} / {len(geo)}')

desc=pd.concat([pd.read_csv(p) for p in partial_files],ignore_index=True)
assert len(desc)==len(geo) and desc.plot_id.nunique()==len(geo), (
    'Missing or duplicate WWF strip summaries')
numeric=[c for c in desc if c.startswith('ha_') or c=='footprint_ha']
desc[numeric]=desc[numeric].apply(pd.to_numeric,errors='coerce')
for year in (2014,2015):
    for name in LABELS:
        desc[f'pct_{year}_{name}']=100*desc[f'ha_{year}_{name}']/desc.footprint_ha
    desc[f'pct_{year}_TMF_forest']=(
        desc[f'pct_{year}_undisturbed']+
        desc[f'pct_{year}_degraded']+
        desc[f'pct_{year}_regrowth'])
    desc[f'pct_{year}_disturbed_or_regrowth']=(
        desc[f'pct_{year}_degraded']+
        desc[f'pct_{year}_regrowth'])
desc['pct_recent_clear_pre2014']=100*desc.ha_recent_clear_pre2014/desc.footprint_ha
desc['pct_changed_class_2014_2015']=100*desc.ha_changed_class_2014_2015/desc.footprint_ha
# Normalized Shannon diversity of the 6-class landscape (not biological diversity).
p=desc[[f'pct_2014_{x}' for x in LABELS]].fillna(0).to_numpy()/100.
p=np.clip(p,0,1)
desc['landscape_diversity_2014']=-(np.where(p>0,p*np.log(np.maximum(p,1e-12)),0)
                                   .sum(axis=1))/np.log(len(LABELS))
desc['max_class_share_2014_pct']=100*p.max(axis=1)
# Ranking emphasizes disturbed / regrowing contexts, but retains entire 6-class landscape.
desc['gradient_priority_score']=(
    desc.pct_2014_degraded*0.25+
    desc.pct_2014_regrowth*0.40+
    desc.pct_recent_clear_pre2014*0.20+
    desc.landscape_diversity_2014*100*0.15
)
desc=desc.sort_values('gradient_priority_score',ascending=False).reset_index(drop=True)
desc['tmf_priority_rank']=np.arange(1,len(desc)+1)
out_csv=TMF_STRIP_DIR/'WWF_all_strips_TMF_2014_2015_descriptors.csv'
desc.to_csv(out_csv,index=False)
joined=geo.merge(desc,on=['plot_id','utm_zone'],validate='one_to_one')
out_gpkg=TMF_STRIP_DIR/'WWF_all_strips_TMF_2014_2015_descriptors.gpkg'
joined.to_file(out_gpkg,driver='GPKG')
print('Saved:',out_csv)
print('Saved:',out_gpkg)
important=['plot_id','utm_zone','footprint_ha','pct_2014_undisturbed',
           'pct_2014_degraded','pct_2014_deforested','pct_2014_regrowth',
           'pct_2014_other','pct_recent_clear_pre2014',
           'pct_changed_class_2014_2015','landscape_diversity_2014',
           'tmf_priority_rank']
display(desc[important].head(25).round(2))
print('\nOverall across-strip variability, percentages (not area weighted):')
display(desc[[c for c in important if c.startswith('pct_')]].describe(
    percentiles=[.1,.25,.5,.75,.9]).round(1))
print('\nMean composition by WWF UTM zone:')
display(desc.groupby('utm_zone')[[f'pct_2014_{x}' for x in LABELS]].mean().round(1))
# Readable stacked composition of the top 30 diversified/disturbed strips.
top=desc.head(30).iloc[::-1]
fig,axs=plt.subplots(1,2,figsize=(16,11),gridspec_kw={'width_ratios':[3,2]})
left=np.zeros(len(top))
palette=['#215b45','#c5a642','#de6b56','#80b779','#4d86b0','#bbb9b4']
for label,color in zip(LABELS,palette):
    v=top[f'pct_2014_{label}'].fillna(0).to_numpy()
    axs[0].barh(top.plot_id,v,left=left,label=label,color=color)
    left+=v
axs[0].set_xlim(0,100)
axs[0].set_xlabel('TMF class area within strip (%) — 2014')
axs[0].set_title('Top 30 strips ranked by recovery / disturbance diversity')
axs[0].legend(ncol=3,fontsize=9,loc='lower center',bbox_to_anchor=(.5,-.14))
axs[1].scatter(desc.pct_2014_undisturbed,desc.pct_2014_regrowth,
               s=20+desc.pct_recent_clear_pre2014*2,
               c=desc.pct_2014_degraded,cmap='viridis',vmin=0,
               vmax=max(20,float(desc.pct_2014_degraded.quantile(.95))),
               alpha=.75,edgecolors='none')
axs[1].set(xlabel='Undisturbed forest (%) — 2014',
           ylabel='Regrowth (%) — 2014',
           title='All 216 strips (size = recent clearing)')
axs[1].grid(alpha=.15)
fig.tight_layout()
png=TMF_STRIP_DIR/'WWF_strip_TMF_gradient_overview.png'
fig.savefig(png,dpi=160,bbox_inches='tight')
plt.show()
print('Saved:',png)
print('Note: these are TMF landscape descriptors, not Musanga prevalence or RGB annotation QC.')

### 4.B — Historical degradation and recovery descriptors through 2014

G.1 quantified the six annual classes for all 216 WWF strips, but **forest-age and disturbance history** are more informative for targeting Musanga pioneer stands.

The JRC provides dedicated **first degradation year** and **first deforestation year** layers, as well as a **2025 endpoint transition map**, intensity and duration. **Do not interpret 2025 endpoint transition subtypes/intensity/duration as conditions visible in WWF imagery acquired in 2014–2015.** This block reconstructs **pre-2015** transitions using annual classes through **December 2014**, plus first-event years with a year ≤2014. We characterize:
- share ever first-degraded / first-deforested by 2014; proportions with first events in **2005–2009** and **2010–2014** (and earlier histories);
- consecutive years in the **degraded (class 2)** and **regrowth (class 4)** states as of end-2014, grouped by **1–3, 4–7 and ≥8 years**;
- historical percentages undisturbed, degraded, deforested and regrowth in **1990, 2000, 2005, 2010 and 2014**;
- **annual TMF class transitions** (1990–2014), distinguished from the number of physical disturbance episodes, and a conservative **recovered from deforested** area measure.

The original G.1 CSV is reused; batch output is cached separately. This is a historical *sampling audit* (not a ground-truth age determination): first-event years are NOT necessarily the latest disturbance date; consecutive-state age is NOT necessarily the exact stand age. TMF uses 30 m pixels, and valid RGB footprint/annotation feasibility must still be checked.

Official TMF sources: <https://forobs.jrc.ec.europa.eu/TMF/data> and <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>.

In [ ]:
# 4.B.1 — Calculate historical TMF descriptors
# Runs after G.1; reuses its CSV, writes separately cached batches.
import ee, geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import mapping
from pathlib import Path
from IPython.display import display

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

ROOT_TMFD=WORK_DIR/'TMF_strip_descriptors'
BASE_CSV=ROOT_TMFD/'WWF_all_strips_TMF_2014_2015_descriptors.csv'
assert BASE_CSV.exists(), 'First run G.1, which computes the initial 216-strip TMF composition.'
base=pd.read_csv(BASE_CSV)
assert len(base)==216 and base.plot_id.is_unique
assert 'plots' in globals(), 'Run Section C to restore WWF strip geometries.'

OUT_DIR=ROOT_TMFD/'historical_2014'
OUT_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE_HIST=8
RECOMPUTE_HIST=False
SCALE_HIST_M=30
JRC='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(JRC+'AnnualChanges').mosaic()
first_deg=ee.ImageCollection(JRC+'DegradationYear').mosaic().select(0)
first_def=ee.ImageCollection(JRC+'DeforestationYear').mosaic().select(0)
def year_img(y):
    return annual.select('Dec'+str(y))
a2014=year_img(2014)
area=ee.Image.pixelArea().divide(10000)
metrics=[]
def area_band(mask,name):
    metrics.append(area.multiply(mask.unmask(0)).rename('ha_'+name))
# First event: exclude events AFTER the WWF acquisition.
for layer,short in [(first_deg,'first_degradation'),(first_def,'first_deforestation')]:
    valid=layer.gte(1982).And(layer.lte(2014))
    area_band(valid,short+'_by2014')
    area_band(valid.And(layer.lte(2004)),short+'_through2004')
    area_band(layer.gte(2005).And(layer.lte(2009)),short+'_2005_2009')
    area_band(layer.gte(2010).And(layer.lte(2014)),short+'_2010_2014')
    # Area-weighted mean first-event YEAR (not latest event).
    metrics.append(area.multiply(layer).updateMask(valid).unmask(0)
                   .rename('yeararea_'+short+'_through2014'))
# End-of-2014 current-state consecutive run-lengths: *age in TMF state*, not stand age.
age_reg=ee.Image.constant(0).toInt16()
age_deg=ee.Image.constant(0).toInt16()
transitions=ee.Image.constant(0).toInt16()
prior=year_img(1990)
last_def_pre2014=ee.Image.constant(0).toByte()
for yr in range(1990,2015):
    now=year_img(yr)
    age_reg=age_reg.add(1).multiply(now.eq(4)).toInt16()
    age_deg=age_deg.add(1).multiply(now.eq(2)).toInt16()
    if yr>1990:
        # A change of annual categorical state is not necessarily a distinct disturbance.
        transitions=transitions.add(now.neq(prior)).toInt16()
        last_def_pre2014=last_def_pre2014.Or(prior.eq(3))
    prior=now
for current_age,current_state,short in [(age_reg,4,'regrowth'),(age_deg,2,'degraded')]:
    for lo,hi,label in [(1,3,'1_3y'),(4,7,'4_7y'),(8,25,'8plus_y')]:
        area_band(a2014.eq(current_state).And(current_age.gte(lo))
                  .And(current_age.lte(hi)),short+'_state_age_'+label)
# Pixels in regrowth in 2014 that had at least one year classified deforested before 2014.
area_band(a2014.eq(4).And(last_def_pre2014),
          'regrowth2014_previously_deforested')
for yr in [1990,2000,2005,2010,2014]:
    img=year_img(yr)
    for cls,label in [(1,'undisturbed'),(2,'degraded'),(3,'deforested'),(4,'regrowth')]:
        area_band(img.eq(cls),f'{label}_{yr}')
# Number of 1990–2014 annual state transitions (each categorical transition counts once).
metrics.append(area.multiply(transitions).rename('area_times_transition_count'))
area_band(transitions.gte(1),'any_transition_1990_2014')
area_band(transitions.gte(2),'multiple_transitions_1990_2014')
metric_img=ee.Image.cat(metrics).toFloat()

geo=plots[['plot_id','geometry']].copy().to_crs(4326)
geo=geo[geo.plot_id.isin(base.plot_id)].sort_values('plot_id')
assert len(geo)==216 and geo.plot_id.is_unique
files=[]
for start in range(0,len(geo),BATCH_SIZE_HIST):
    batch=geo.iloc[start:start+BATCH_SIZE_HIST]
    fname=OUT_DIR/f'history_{start:03d}_{start+len(batch):03d}.csv'
    files.append(fname)
    if fname.exists() and not RECOMPUTE_HIST:
        print('Cached',fname.name)
        continue
    features=[ee.Feature(
        ee.Geometry(mapping(rec.geometry),proj='EPSG:4326',geodesic=False),
        {'plot_id':str(rec.plot_id)}) for rec in batch.itertuples(index=False)]
    stats=metric_img.reduceRegions(collection=ee.FeatureCollection(features),
        reducer=ee.Reducer.sum(),scale=SCALE_HIST_M,tileScale=4)
    raw=[f['properties'] for f in stats.getInfo()['features']]
    result=pd.DataFrame(raw)
    assert len(result)==len(batch) and result.plot_id.is_unique
    result.to_csv(fname,index=False)
    print(f'Historical TMF: {start+1}–{start+len(batch)} / {len(geo)}')
hist=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
assert len(hist)==216 and hist.plot_id.is_unique
merged=base.merge(hist,on='plot_id',how='left',validate='one_to_one')
assert len(merged)==216 and merged['ha_first_deforestation_by2014'].notna().all()
# Convert area metrics to fraction of full survey footprint, not fraction of forest.
for key in hist.columns:
    if key.startswith('ha_'):
        merged['pct_'+key[3:]]=100*pd.to_numeric(merged[key])/merged.footprint_ha
for label in ['first_degradation','first_deforestation']:
    merged['mean_'+label+'_year_by2014']=np.where(
        merged['ha_'+label+'_by2014']>0,
        merged['yeararea_'+label+'_through2014']/merged['ha_'+label+'_by2014'],
        np.nan)
merged['mean_class_transitions_1990_2014']=(
    merged.area_times_transition_count/merged.footprint_ha)
# Conditional proportions within existing degraded or regrowth 2014 classes:
for cat in ['degraded','regrowth']:
    denom=merged[f'ha_{cat}_2014'].replace(0,np.nan)
    for age in ['1_3y','4_7y','8plus_y']:
        key=f'ha_{cat}_state_age_{age}'
        merged[f'pct_within_{cat}_{age}']=100*merged[key]/denom

merged.to_csv(OUT_DIR/'WWF_all_216_strips_TMF_history_asof2014.csv',index=False)
geo_out=geo.merge(merged,on='plot_id',validate='one_to_one')
geo_out.to_file(OUT_DIR/'WWF_all_216_strips_TMF_history_asof2014.gpkg',driver='GPKG')
cols=['plot_id','utm_zone','pct_2014_undisturbed',
      'pct_2014_degraded','pct_2014_regrowth',
      'pct_first_degradation_2010_2014','pct_first_deforestation_2010_2014',
      'pct_regrowth_state_age_1_3y','pct_regrowth_state_age_4_7y',
      'pct_regrowth_state_age_8plus_y','pct_multiple_transitions_1990_2014']
print('Most first-degraded strips in 2010–2014:')
display(merged.sort_values('pct_first_degradation_2010_2014',
                           ascending=False)[cols].head(20).round(2))
print('Most recently deforested strips (first event 2010–2014):')
display(merged.sort_values('pct_first_deforestation_2010_2014',
                           ascending=False)[cols].head(20).round(2))
print('Strips richest in 2014 recent-state regrowth (1–3 yr):')
display(merged.sort_values('pct_regrowth_state_age_1_3y',
                           ascending=False)[cols].head(20).round(2))
print('Distribution across the 216 WWF strips:')
display(merged[[c for c in cols if c.startswith('pct_')]].describe(
    percentiles=[.1,.25,.5,.75,.9]).round(2))

fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
axes[0].scatter(merged.pct_first_degradation_2010_2014,
                merged.pct_first_deforestation_2010_2014,
                s=20+3*merged.pct_2014_regrowth,
                alpha=.7)
axes[0].set(xlabel='First degradation in 2010–2014 (% of strip)',
            ylabel='First deforestation in 2010–2014 (% of strip)',
            title='Historical first-event gradient')
axes[1].scatter(merged.pct_2014_regrowth,
                merged.pct_regrowth_state_age_1_3y,
                s=25,alpha=.7)
axes[1].set(xlabel='All regrowth in 2014 (% of strip)',
            ylabel='Regrowth in 1–3 year consecutive state (% of strip)',
            title='Young TMF regrowth vs total regrowth')
for ax in axes:ax.grid(alpha=.15)
fig_path=OUT_DIR/'WWF_TM F_historical_gradient_2014.png'.replace(' ','')
fig.savefig(fig_path,dpi=160)
plt.show()
print('Saved history to',OUT_DIR)
print('Caution: 2025 TransitionMap_Subtypes / duration / intensity are not 2014 classes.')


### 4.C — Full TMF pixel distributions within potential RGB footprints

This is the spatial audit **before selecting any circles**. For each WWF RGB GeoTIFF, read **only its header**, intersect its georeferenced raster bounds with the corresponding LiDAR survey polygon, and summarize **every TMF 30 m pixel** within that intersection (not a random sample). There are **216** available RGB mosaics, rather than 226.

**Footprint caveat:** an intersection between the raster rectangle and survey polygon is only the *potential* RGB-covered footprint. No-data/black swaths remain possible. Computing an *exact* native RGB-valid mask for all mosaics requires reading many GB of native RGB pixels because the GeoTIFFs lack overviews. Before final annotation-site selection, check true image validity in promising locations.

**TMF outputs:** full per-strip **frequency histograms** for 2014/2015 status, first degradation/deforestation year up through 2014, end-2014 consecutive degraded/regrowth-state age, and number of annual class transitions through 2014. Plus annual 1990/2000/2005/2010 composition. A **separately labeled 2025 endpoint** histogram covers the JRC transition subtype map: useful retrospectively, **not** a 2014 forest condition. Percentage profiles, CSVs and GeoPackage are created, plus a map with *actual 30 m TMF pixels* (not just coloured strip aggregates). The raster layers are shown **only inside the candidate RGB footprints**.

**No circles are generated here.** Use the resulting spatial map to choose a sampling frame, then confirm native RGB coverage.

In [ ]:
# 4.C.1 — Audit pixel-level TMF in RGB footprints
# Run after Sections A–C / 'plots' + 'BUCKET' + 'WORK_DIR' exist.
import re, json, rasterio, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box, mapping
from rasterio.warp import transform_bounds
from IPython.display import display
import ee

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G3_DIR.mkdir(parents=True,exist_ok=True)
REBUILD_RGB_FOOTPRINTS=False
RECOMPUTE_TMF_HIST=False
TMF_BATCH=8
TMF_SCALE=30
ADD_2025_ENDPOINT=True  # separate retrospective descriptor: NEVER a 2014 label

# -------------------------------------------------------------------------
# 1. Header-only actual TIFF bounds intersected with each LiDAR plot polygon
# -------------------------------------------------------------------------
foot_gpkg=G3_DIR/'WWF_RGB_potential_footprints.gpkg'
if foot_gpkg.exists() and not REBUILD_RGB_FOOTPRINTS:
    rgb_foot=gpd.read_file(foot_gpkg).to_crs(4326)
    print('Reused cached TIFF bounds × plot polygons:',len(rgb_foot))
else:
    assert 'plots' in globals() and 'BUCKET' in globals()
    source=plots.to_crs(4326).copy()
    assert source.plot_id.is_unique
    records=[]
    for k,row in enumerate(source.itertuples(index=False),1):
        plot_id=str(row.plot_id)
        uri=f'/vsigs/{BUCKET}/001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as src:
                    bb=transform_bounds(src.crs,'EPSG:4326',*src.bounds,densify_pts=21)
                    nominal_gsd_cm=100*abs(src.res[0])
            overlap=row.geometry.intersection(box(*bb))
            if overlap.is_empty or overlap.area==0:
                print('NO OVERLAP:',plot_id)
                continue
            records.append(dict(plot_id=plot_id,utm_zone=str(row.utm_zone),
                native_gsd_cm=nominal_gsd_cm,footprint_method='survey_polygon_intersect_rgb_bounds',
                valid_rgb_checked=False,geometry=overlap))
        except Exception as e:
            print('FAILED RGB HEADER:',plot_id,str(e)[:200])
        if k%24==0:print('RGB header checked',k,'/',len(source))
    assert records,'No RGB footprint intersections found.'
    rgb_foot=gpd.GeoDataFrame(records,geometry='geometry',crs='EPSG:4326')
    rgb_foot=rgb_foot[rgb_foot.geometry.is_valid & ~rgb_foot.geometry.is_empty]
    rgb_foot.to_file(foot_gpkg,driver='GPKG')
assert rgb_foot.plot_id.is_unique
print('Potential RGB-mosaic footprints:',len(rgb_foot),'/ 216; only these will be processed')
if len(rgb_foot)!=216:print('WARNING: some RGB mosaics lack usable header/footprint intersections')

# -------------------------------------------------------------------------
# 2. Historical TMF pixel bands; no data after 2014 used in historical fields.
# -------------------------------------------------------------------------
jrc='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(jrc+'AnnualChanges').mosaic()
def annual_y(y):return annual.select('Dec'+str(y))
a14=annual_y(2014)
first_deg=ee.ImageCollection(jrc+'DegradationYear').mosaic().select(0)
first_def=ee.ImageCollection(jrc+'DeforestationYear').mosaic().select(0)
# 0 = first event not observed by 2014; also masks any later first events.
def history_year(im):
    return im.where(im.gt(2014),0).unmask(0).toInt16()
age_reg=ee.Image.constant(0).toInt16()
age_deg=ee.Image.constant(0).toInt16()
n_trans=ee.Image.constant(0).toInt16()
prev=None
for year in range(1990,2015):
    curr=annual_y(year)
    age_reg=age_reg.add(1).multiply(curr.eq(4)).toInt16()
    age_deg=age_deg.add(1).multiply(curr.eq(2)).toInt16()
    if prev is not None:
        n_trans=n_trans.add(curr.neq(prev)).toInt16()
    prev=curr
bands=[annual_y(y).unmask(0).toInt16().rename('status_'+str(y))
       for y in [1990,2000,2005,2010,2014,2015]]
bands.extend([
    history_year(first_deg).rename('first_degradation_by2014'),
    history_year(first_def).rename('first_deforestation_by2014'),
    age_reg.unmask(0).rename('regrowth_run_age2014'),
    age_deg.unmask(0).rename('degraded_run_age2014'),
    n_trans.unmask(0).rename('annual_class_transitions_1990_2014'),
])
if ADD_2025_ENDPOINT:
    endpoint=ee.ImageCollection(jrc+'TransitionMap_Subtypes').mosaic().select(0)
    bands.append(endpoint.unmask(0).toInt16().rename('transition_subtype_ENDPOINT_2025'))
tmf_hist=ee.Image.cat(bands)

# -------------------------------------------------------------------------
# 3. Count all 30 m pixel categories in each intersected polygon (not samples)
# -------------------------------------------------------------------------
rows=[]
ordered=rgb_foot.sort_values('plot_id').reset_index(drop=True)
expected=list(tmf_hist.bandNames().getInfo())
for start in range(0,len(ordered),TMF_BATCH):
    chunk=ordered.iloc[start:start+TMF_BATCH]
    fn=G3_DIR/f'pixel_hist_{start:03d}_{start+len(chunk):03d}.json'
    if fn.exists() and not RECOMPUTE_TMF_HIST:
        with open(fn) as f:partial=json.load(f)
        rows.extend(partial)
        print('Cached pixel histories:',fn.name)
        continue
    features=[ee.Feature(ee.Geometry(mapping(x.geometry),geodesic=False),
                         {'plot_id':str(x.plot_id)})
              for x in chunk.itertuples(index=False)]
    reduced=tmf_hist.reduceRegions(
        collection=ee.FeatureCollection(features),
        reducer=ee.Reducer.frequencyHistogram(),
        scale=TMF_SCALE,tileScale=4)
    response=reduced.getInfo()['features']
    partial=[]
    for feat in response:
        p=feat['properties']
        rec={'plot_id':p['plot_id']}
        for band in expected:
            hist=p.get(band) or {}
            rec[band]={str(k):int(round(v)) for k,v in hist.items()}
        partial.append(rec)
    assert len(partial)==len(chunk),'Unexpected EE results for batch'
    with open(fn,'w') as f:json.dump(partial,f)
    rows.extend(partial)
    print('TMF all-pixel histograms:',start+1,'–',start+len(chunk),'/',len(ordered))
assert len(rows)==len(ordered) and len({r['plot_id'] for r in rows})==len(ordered)
with open(G3_DIR/'WWF_all_RGB_footprints_full_TMF_pixel_histograms.json','w') as f:
    json.dump(rows,f,indent=1)

# -------------------------------------------------------------------------
# 4. Summaries and time/age bins; all derived from the per-pixel histograms
# -------------------------------------------------------------------------
def count_if(hist,predicate):
    return sum(int(count) for key,count in hist.items() if predicate(int(key)))
tab=[]
for r in rows:
    d={'plot_id':r['plot_id']}
    for b in expected:
        h=r[b]
        d['n_valid_'+b]=count_if(h,lambda x:x!=0)
        d['n_all_'+b]=sum(h.values())
    n=d['n_all_status_2014']
    d['tmf_pixels_2014']=n
    for year in [2014,2015]:
        h=r['status_'+str(year)]
        for c,label in enumerate(['undisturbed','degraded','deforested',
                                   'regrowth','water','other'],1):
            d[f'pct_{year}_{label}']=100*h.get(str(c),0)/max(n,1)
    for key,out in [('first_degradation_by2014','degradation'),
                    ('first_deforestation_by2014','deforestation')]:
        h=r[key]
        for lo,hi,label in [(1982,1999,'before2000'),(2000,2009,'2000_2009'),
                             (2010,2014,'2010_2014')]:
            d[f'pct_first_{out}_{label}']=100*count_if(h,lambda y:lo<=y<=hi)/max(n,1)
        d[f'pct_first_{out}_by2014']=100*count_if(h,lambda y:1982<=y<=2014)/max(n,1)
    for key,label in [('regrowth_run_age2014','regrowth'),
                      ('degraded_run_age2014','degraded')]:
        h=r[key]
        for lo,hi,cls in [(1,3,'1_3y'),(4,7,'4_7y'),(8,100,'8plus_y')]:
            d[f'pct_{label}_state_age_{cls}']=100*count_if(h,lambda a:lo<=a<=hi)/max(n,1)
    h=r['annual_class_transitions_1990_2014']
    d['pct_2plus_annual_transitions']=100*count_if(h,lambda x:x>=2)/max(n,1)
    if ADD_2025_ENDPOINT:
        d['n_2025_transition_subtypes']=count_if(
            r['transition_subtype_ENDPOINT_2025'],lambda x:x!=0)
    tab.append(d)
summary=pd.DataFrame(tab).merge(ordered[['plot_id','utm_zone','native_gsd_cm']],
                                on='plot_id',validate='one_to_one')
summary['potential_RGB_footprint_only']=True
summary['TMF_pixel_scale_m']=TMF_SCALE
outcsv=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.csv'
summary.to_csv(outcsv,index=False)
merged=ordered.merge(summary.drop(columns=['utm_zone','native_gsd_cm']),
                     on='plot_id',validate='one_to_one')
outgpkg=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
merged.to_file(outgpkg,driver='GPKG')
print('Saved full categorical pixel histograms:',G3_DIR/'WWF_all_RGB_footprints_full_TMF_pixel_histograms.json')
print('Saved descriptor table:',outcsv)
print('Saved spatial table:',outgpkg)
display(summary.sort_values('pct_regrowth_state_age_1_3y',ascending=False).head(20).round(2))
print('Total 2014 TMF pixel observations:',summary.tmf_pixels_2014.sum())

# Map lives in G.3B below: rerun ONLY G.3B if summary CSV/GPKG already exists.
print('Run G.3B for interactive TMF maps. No extra histogram processing required.')


### 4.D — Interactive TMF pixel map (Earth Engine tiles)

This **standalone** visualization reads the saved G.3 GeoPackage and uses native **Folium + Earth Engine map tiles**. It does **not** reprocess TIFF headers, rerun 4.84 million-pixel histograms or depend on `geemap.foliumap` / `xyz_to_folium`.

Switch layers to inspect geographic patterns of 2014 status, young regrowth, degradation and deforestation years, and 1990–2014 class-transition counts. The coloured TMF pixels are clipped to **survey polygon ∩ georeferenced RGB TIFF bounds**, not verified RGB-valid pixel masks. In historical maps all time-dependent layers stop at 2014.

In [ ]:
# 4.D.1 — Interactive TMF raster map
# Standalone: mount Drive and reconstruct saved-result directory if this is a new runtime.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
assert WORK_DIR.exists(), f'WWF NB08 output folder not found: {WORK_DIR}'
# Only needs saved G.3 GeoPackage, WORK_DIR and Google Earth Engine access.
import ee, folium, geopandas as gpd
from pathlib import Path
from shapely.geometry import mapping
from IPython.display import display
from folium.plugins import Fullscreen

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
saved=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert saved.exists(), f'G.3 spatial results missing: {saved}'
g3_map_data=gpd.read_file(saved).to_crs('EPSG:4326')
assert len(g3_map_data)==216 and g3_map_data.plot_id.is_unique
print('Loaded saved RGB-potential footprints:',len(g3_map_data))

# Server-side clip to collection of all candidate georeferenced RGB footprints.
fc=ee.FeatureCollection([
    ee.Feature(ee.Geometry(mapping(row.geometry),geodesic=False))
    for row in g3_map_data.itertuples(index=False)
])
JRC='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(JRC+'AnnualChanges').mosaic()
state14=annual.select('Dec2014')
firstdeg=ee.ImageCollection(JRC+'DegradationYear').mosaic().select(0)
firstdef=ee.ImageCollection(JRC+'DeforestationYear').mosaic().select(0)
age_reg=ee.Image.constant(0).toInt16()
nchanges=ee.Image.constant(0).toInt16()
prev=None
for year in range(1990,2015):
    current=annual.select('Dec'+str(year))
    age_reg=age_reg.add(1).multiply(current.eq(4)).toInt16()
    if prev is not None:
        nchanges=nchanges.add(current.neq(prev)).toInt16()
    prev=current

def add_ee_layer(folium_map,ee_image,vis,label,show=False):
    # Earth Engine returns a standard HTTPS XYZ tile URL for Folium.
    tile=ee_image.clipToCollection(fc).getMapId(vis)['tile_fetcher'].url_format
    folium.TileLayer(tiles=tile,attr='Google Earth Engine / JRC TMF',
                     name=label,overlay=True,control=True,show=show,
                     max_zoom=20,opacity=0.85).add_to(folium_map)

m=folium.Map(location=[-2.0,23.5],zoom_start=6,tiles=None,
             control_scale=True,prefer_canvas=True)
# Reliable background independent of public OSM / Carto tile availability.
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Offline background',name='Blank (offline)',overlay=False,
    show=True,control=True,max_zoom=20).add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri imagery',name='Esri satellite (optional)',overlay=False,
    show=False,control=True,max_zoom=20).add_to(m)
add_ee_layer(m,state14,{'min':1,'max':6,
    'palette':['005a25','8cba44','e97649','a0db4d','0089c6','cccccc']},
    'TMF 2014 forest classes',True)
add_ee_layer(m,age_reg.updateMask(state14.eq(4)),
    {'min':1,'max':12,'palette':['f6e799','a4cf50','368f49']},
    'Regrowth state age in 2014 (years)')
add_ee_layer(m,firstdeg.updateMask(firstdeg.gte(2010).And(firstdeg.lte(2014))),
    {'min':2010,'max':2014,'palette':['ffda75','ffb04b','ff6b42','c52d36']},
    'First degradation year 2010–2014')
add_ee_layer(m,firstdef.updateMask(firstdef.gte(2010).And(firstdef.lte(2014))),
    {'min':2010,'max':2014,'palette':['ffe2a8','ffa556','f7603a','a9152d']},
    'First deforestation year 2010–2014')
add_ee_layer(m,nchanges.updateMask(nchanges.gt(0)),
    {'min':1,'max':5,'palette':['fff5b6','fcb45a','e4573a','8d2052']},
    'Number of annual TMF status changes (1990–2014)')

outline_cols=['plot_id','pct_2014_undisturbed',
              'pct_2014_degraded','pct_2014_regrowth',
              'pct_first_deforestation_2010_2014','geometry']
outline=g3_map_data[outline_cols].copy()
for col in outline_cols[1:-1]:
    outline[col]=outline[col].round(2)
folium.GeoJson(
    outline.to_json(),
    style_function=lambda f:{'color':'#222222','weight':1.0,
                             'fillColor':'#000000','fillOpacity':0.0},
    tooltip=folium.GeoJsonTooltip(
        fields=outline_cols[:-1],
        aliases=['WWF plot','Undisturbed (%)','Degraded (%)',
                 'Regrowth (%)','First deforestation 2010–14 (%)'],
        localize=True),
    name='WWF potential RGB footprints'
).add_to(m)
bounds=g3_map_data.total_bounds
m.fit_bounds([[bounds[1],bounds[0]],[bounds[3],bounds[2]]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
out=G3_DIR/'WWF_all_216_RGB_TMF_pixel_map.html'
m.save(str(out))
print('Interactive TMF map saved:',out)
display(m)
print('NOTE: RGB no-data/black margins are still unmasked. No annotation sites selected.')


### 4.E — TMF distributions, tables and figures

This is a **pure local statistics and visualization block**: it loads the saved G.3 CSV and complete JSON pixel histograms. **No Earth Engine calls or TIFF downloads.** Two complementary perspectives are reported:

1. **Across-strip distributions:** descriptive statistics (min, P10, P25, median, P75, P90, P95, max, standard deviation and fraction of strips with zero) for all `pct_*` TMF features; histograms of key gradients and a per-strip composition heatmap. **Each strip has equal weight** here.
2. **Pooled TMF pixel distributions:** sum of the complete pixel histograms across all 216 potential RGB footprints. For each annual status year, degradation/deforestation *first-event* year, regrowth-state age, degraded-state age, historical class-transition count and **separately labeled 2025 transition subtypes**, display the full categorical distribution (counts and percentages). **Each mapped 30 m pixel observation has equal weight**, subject to the server's raster pixel grid and possible overlap between strips.

These are distributions **within survey polygon ∩ georeferenced RGB raster bounds**, **not** masks of genuine RGB-valid pixels. Numbers are historical TMF classifications, **not** Musanga occurrence and not unbiased landscape-wide prevalence.

In [ ]:
# 4.E.1 — Statistical TMF overview
# Standalone: mount Drive and reconstruct saved-result directory if this is a new runtime.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
assert WORK_DIR.exists(), f'WWF NB08 output folder not found: {WORK_DIR}'
import json, os, shutil, tempfile, pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')  # avoid slow interactive backend while rendering multiple figures
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
csv=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.csv'
histfile=G3_DIR/'WWF_all_RGB_footprints_full_TMF_pixel_histograms.json'
assert csv.exists() and histfile.exists(), 'Run G.3 once; the saved CSV and histogram JSON are required.'
strips=pd.read_csv(csv)
with open(histfile) as handle: pixel_rows=json.load(handle)
assert len(strips)==216 and strips.plot_id.nunique()==216
assert len(pixel_rows)==216 and len({r['plot_id'] for r in pixel_rows})==216
REPORT=G3_DIR/'distribution_report'
REPORT.mkdir(parents=True,exist_ok=True)
LOCAL_PLOTS=Path(tempfile.mkdtemp(prefix='wwf_tmf_figs_'))
def save_report_figure(fig, filename):
    # Avoid rendering directly onto mounted Drive; export modest PNG to local disk first.
    local=LOCAL_PLOTS/filename
    fig.savefig(local,dpi=110,bbox_inches='tight')
    plt.close(fig)
    final=REPORT/filename
    shutil.copyfile(local,final)
    print('Saved',final)
    from IPython.display import Image,display
    display(Image(filename=str(local),width=950))
    return final
print('Loaded',len(strips),'WWF strips and',len(pixel_rows),'complete TMF pixel histograms')

# -------------------------------------------------------------
# 1 — Equal-weight strip-level distributions for EVERY percent feature
# -------------------------------------------------------------
features=sorted(c for c in strips.columns if c.startswith('pct_'))
assert features,'Missing percent descriptors from G.3'
stat=[]
for name in features:
    values=pd.to_numeric(strips[name],errors='coerce').replace([np.inf,-np.inf],np.nan)
    valid=values.dropna()
    if len(valid)==0:continue
    q=valid.quantile([.10,.25,.50,.75,.90,.95])
    stat.append(dict(descriptor=name,n=len(valid),mean=valid.mean(),
         sd=valid.std(),min=valid.min(),p10=q.loc[.10],p25=q.loc[.25],
         median=q.loc[.50],p75=q.loc[.75],p90=q.loc[.90],p95=q.loc[.95],
         max=valid.max(),pct_strips_zero=100*(valid==0).mean()))
stats=pd.DataFrame(stat)
stats.to_csv(REPORT/'WWF_TMF_all_descriptors_across_216_strips.csv',index=False)
print('Across-strip distribution of ALL',len(stats),'TMF percentage descriptors:')
display(stats.round(2))

# -------------------------------------------------------------
# 2 — Aggregate the FULL categorical TMF histograms from G.3
# -------------------------------------------------------------
bands=sorted({k for row in pixel_rows for k in row if k!='plot_id'})
long=[]
for band in bands:
    counter={}
    for row in pixel_rows:
        for k,v in row.get(band,{}).items():
            key=int(k)
            counter[key]=counter.get(key,0)+int(v)
    total=sum(counter.values())
    nonzero=sum(v for k,v in counter.items() if k!=0)
    for key,value in sorted(counter.items()):
        long.append(dict(band=band,category_value=key,count=value,
                         total_pixels_for_band=total,
                         percent_of_all_pixels=100*value/max(total,1),
                         percent_of_nonzero_pixels=(100*value/nonzero
                            if key!=0 and nonzero else 0)))
    print(f'{band}: {len(counter)} distinct values | total={total:,} | '
          f'nonzero={nonzero:,}')
pooled=pd.DataFrame(long)
pooled.to_csv(REPORT/'WWF_TMF_complete_pooled_pixel_distributions.csv',index=False)
# Keep explicit status codes; 0 denotes masked/unclassified and must NOT be a forest class.
status_codes={0:'Masked / unclassified',1:'Undisturbed',2:'Degraded',
              3:'Deforested',4:'Regrowth',5:'Water',6:'Other'}
status=pooled[pooled.band.isin(['status_2014','status_2015'])].copy()
status['class']=status.category_value.map(status_codes).fillna('Other code')
display(status[['band','category_value','class','count','percent_of_all_pixels']].round(3))
print('Pooled-pixel statistics are area-like, whereas across-strip statistics give equal weight to each WWF strip.')

# -------------------------------------------------------------
# 3 — Figures: composition by 2014 class, strip distributions
# -------------------------------------------------------------
main_classes=['undisturbed','degraded','deforested','regrowth','water','other']
fig,axes=plt.subplots(2,3,figsize=(15,8),constrained_layout=True)
for ax,cl in zip(axes.flat,main_classes):
    x=strips[f'pct_2014_{cl}'].dropna()
    ax.hist(x,bins=np.arange(0,101,5),edgecolor='white')
    ax.axvline(x.median(),ls='--',lw=1.4)
    ax.set(title=f'{cl} (median {x.median():.2f}%)',xlabel='% of potential RGB footprint',
           ylabel='Number of strips',xlim=(0,100))
p1=REPORT/'2014_TMF_class_percentages_all_strips.png'
p1=save_report_figure(fig,p1.name)

# The long-tailed, rare regrowth fractions deserve a zoomed view.
rare=['pct_2014_regrowth','pct_regrowth_state_age_1_3y',
      'pct_regrowth_state_age_4_7y','pct_regrowth_state_age_8plus_y',
      'pct_first_degradation_2010_2014',
      'pct_first_deforestation_2010_2014','pct_2plus_annual_transitions']
rare=[x for x in rare if x in strips]
fig,axes=plt.subplots(2,4,figsize=(18,8),constrained_layout=True)
for ax,key in zip(axes.flat,rare):
    x=pd.to_numeric(strips[key],errors='coerce').dropna()
    ax.hist(x,bins=30,edgecolor='white')
    ax.axvline(x.median(),ls='--',lw=1.4)
    ax.set(title=key.replace('pct_','').replace('_',' '),xlabel='% of strip',
           ylabel='WWF strips')
for ax in list(axes.flat)[len(rare):]:ax.axis('off')
p2=REPORT/'2014_TMF_rare_recovery_disturbance_distributions.png'
p2=save_report_figure(fig,p2.name)

# -------------------------------------------------------------
# 4 — Complete year-by-year / age histograms (not just summary bins)
# -------------------------------------------------------------
requested_bands=[
    'first_degradation_by2014','first_deforestation_by2014',
    'regrowth_run_age2014','degraded_run_age2014',
    'annual_class_transitions_1990_2014',
    'transition_subtype_ENDPOINT_2025'
]
requested_bands=[x for x in requested_bands if x in bands]
fig,axes=plt.subplots(len(requested_bands),1,
    figsize=(16,max(4,2.7*len(requested_bands))),constrained_layout=True)
axes=np.atleast_1d(axes)
for ax,band in zip(axes,requested_bands):
    sub=pooled[(pooled.band==band)&(pooled.category_value!=0)]
    ax.bar(sub['category_value'].to_numpy(),sub['count'].to_numpy())
    ax.set(xlabel='Year, years-in-state, or categorical code',
           ylabel='30 m pixel observations',title=band)
    ax.tick_params(axis='x',labelsize=8)
p3=REPORT/'TMF_full_year_age_transition_histograms.png'
p3=save_report_figure(fig,p3.name)
print('Saved full year / age / transition distributions, including 2025 endpoint as separate series.')

# -------------------------------------------------------------
# 5 — Heatmap: how 2014 TMF composition varies among all strips
# -------------------------------------------------------------
heat=strips.sort_values('pct_2014_regrowth',ascending=False)
arr=heat[[f'pct_2014_{c}' for c in main_classes]].fillna(0).to_numpy()
fig,ax=plt.subplots(figsize=(9,13))
im=ax.imshow(arr,aspect='auto',interpolation='nearest',vmin=0,vmax=100,cmap='viridis')
ax.set_xticks(range(len(main_classes)),main_classes,rotation=40,ha='right')
ax.set_yticks(np.arange(0,len(heat),10))
ax.set_yticklabels(heat.plot_id.iloc[::10].tolist(),fontsize=7)
ax.set_title('All 216 WWF RGB footprints — 2014 class composition\n'
             '(sorted by regrowth proportion)')
fig.colorbar(im,ax=ax,label='TMF pixels in class (%)',fraction=.03,pad=.02)
p4=REPORT/'TMF_2014_composition_heatmap_all_216_strips.png'
p4=save_report_figure(fig,p4.name)

print('\nReport folder:',REPORT)
for file in [p1,p2,p3,p4]:
    print('Figure:',file.name)
print('Tables: WWF_TMF_all_descriptors_across_216_strips.csv; '
      'WWF_TMF_complete_pooled_pixel_distributions.csv')
print('NOTE: first-event years are not the most recent disturbance year;'
      ' 2025 endpoint subtypes are retrospective only.')


### 4.F — Offline HTML atlas of strip-level TMF statistics

Make one **genuinely self-contained HTML document**, embedding simplified WWF polygons and the saved TMF attributes. The HTML uses **native SVG + JavaScript** rather than Folium/Leaflet CDN, GEE tokens, external basemaps or external files; it can be emailed, placed on Google Drive or hosted as an ordinary static page. Colleagues can open the downloaded HTML locally and use the selector to switch among all available TMF percentage descriptors, hover a strip for its values, click for details, and zoom/pan. The background is a blank geographic coordinate plane (no satellite imagery); the survey geometry and statistics are embedded. The full TMF pixel rasters are **not embedded**: this is a per-strip thematic atlas, not the exact 30 m pixel map. It uses the existing G.3 results only and does not call Earth Engine.

**Sharing:** Download the resulting HTML and email/share as a file (some Google Drive previews block scripts, so collaborators should download and open it in a browser). It will remain usable without Earth Engine accounts and without expiring GEE tile URLs.

In [ ]:
# 4.F.1 — Durable strip-level HTML atlas
# Embedded geographic polygons + all TMF percentages; no Earth Engine or CDN.
from pathlib import Path
from google.colab import drive
import json, html
import pandas as pd, numpy as np, geopandas as gpd
from shapely.geometry import mapping

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
folder=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
source=folder/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert source.exists(), f'Run G.3 first: {source}'
g=gpd.read_file(source).to_crs('EPSG:4326')
assert len(g)==216 and g.plot_id.is_unique
# Keep a small HTML file: simplify only the display geometry in geographic degrees.
g['geometry']=g.geometry.simplify(.002,preserve_topology=True)
cols=sorted(k for k in g.columns if k.startswith('pct_'))
assert cols
records=[]
for row in g.itertuples(index=False):
    rec={'id':str(row.plot_id),'zone':str(row.utm_zone),
         'geometry':mapping(row.geometry)}
    for col in cols:
        val=getattr(row,col)
        rec[col]=None if pd.isna(val) else round(float(val),3)
    records.append(rec)
payload=json.dumps({'features':records,'metrics':cols},separators=(',',':'))
# A real single-file SVG explorer: no fetch(), external tiles, JS libraries or CDN.
page=r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>WWF RGB footprints | TMF forest gradient 2014–2015</title>
<style>
:root{font-family:system-ui,-apple-system,Segoe UI,sans-serif;color:#203235}
body{margin:0;background:#f5f7f6}
header{padding:16px 20px;background:#123a36;color:white}
header h1{font-size:20px;margin:0 0 5px}
header small{color:#cfe5db}
#tools{display:flex;align-items:center;flex-wrap:wrap;gap:10px;padding:12px 20px;background:white;border-bottom:1px solid #ddd}
select,button{font:inherit;padding:7px 10px;border:1px solid #bdc9c5;background:white;border-radius:5px}
main{display:grid;grid-template-columns:minmax(0,1fr) 290px;gap:14px;padding:14px}
#mapwrap{background:#edf3ee;border:1px solid #cbd8ce;border-radius:8px;position:relative;min-height:490px;overflow:hidden}
svg{display:block;width:100%;height:72vh;min-height:490px;cursor:grab;touch-action:none}
path.strip{stroke:#435f5a;stroke-width:0.9;vector-effect:non-scaling-stroke;cursor:pointer}
path.strip:hover{stroke:#131d1b;stroke-width:2;vector-effect:non-scaling-stroke}
#info{padding:15px;background:white;border:1px solid #dce4df;border-radius:8px;font-size:13px}
#info h2{font-size:16px;margin:0 0 9px}
#info table{border-collapse:collapse;width:100%}
#info td{padding:4px 3px;border-bottom:1px solid #eee}
#info td:last-child{text-align:right;font-variant-numeric:tabular-nums}
#tip{position:absolute;background:#173c38;color:white;padding:5px 8px;font-size:12px;border-radius:5px;pointer-events:none;display:none;z-index:4}
#legend{display:flex;align-items:center;gap:4px;font-size:11px;color:#394c48}
#legend i{width:130px;height:10px;display:inline-block;background:linear-gradient(90deg,#edf1e6,#d5df9b,#a5bd71,#569b71,#176c55)}
#note{font-size:11px;color:#53615b;margin-top:10px;line-height:1.5}
@media(max-width:800px){main{grid-template-columns:1fr}svg{height:58vh}}
</style></head><body>
<header><h1>WWF RGB forest-gradient atlas</h1>
<small>216 potential RGB footprints | JRC TMF 30 m | 2014–2015 survey period | shareable, no Earth Engine or external resources</small></header>
<section id="tools"><label for="metric">TMF descriptor:</label>
<select id="metric"></select><span id="legend"><span>0%</span><i></i><span id="upper">100%</span></span>
<button id="zoomIn">Zoom +</button><button id="zoomOut">Zoom −</button><button id="reset">Reset</button>
<span style="font-size:11px">Hover or click a strip · Wheel to zoom · Drag to pan</span></section>
<main><div id="mapwrap"><svg id="map" viewBox="0 0 1200 690" aria-label="WWF TMF survey geographic distribution">
<rect width="1200" height="690" fill="#eef2ee"/><g id="mapgroup"></g></svg>
<div id="tip"></div></div><aside id="info"><h2>Select a WWF strip</h2>
<div id="detail">Click any strip to see its TMF profile.</div>
<div id="note">TMF fractions refer to the intersection of the WWF survey polygon and RGB raster bounds, <strong>not validated nonblack RGB pixels</strong>. No actual WWF RGB photographs or 30 m TMF rasters are embedded. Historical categories are not Musanga annotations.</div></aside></main>
<script>
const DATA=__PAYLOAD__;
const features=DATA.features, metrics=DATA.metrics;
const WIDTH=1200,HEIGHT=690,PAD=38;
const sel=document.getElementById('metric'),root=document.getElementById('mapgroup');
const svg=document.getElementById('map'),tip=document.getElementById('tip');
metrics.forEach(m=>{const op=document.createElement('option');op.value=m;op.textContent=m.replace(/^pct_/,'').replaceAll('_',' ');sel.appendChild(op)});
sel.value='pct_2014_regrowth';
let bounds=[Infinity,Infinity,-Infinity,-Infinity];
function coordsWalk(c){if(typeof c[0]==='number'){bounds[0]=Math.min(bounds[0],c[0]);bounds[1]=Math.min(bounds[1],c[1]);bounds[2]=Math.max(bounds[2],c[0]);bounds[3]=Math.max(bounds[3],c[1]);}else c.forEach(coordsWalk)}
features.forEach(f=>coordsWalk(f.geometry.coordinates));
const midlat=(bounds[1]+bounds[3])/2,cos=Math.cos(midlat*Math.PI/180);
const extX=(bounds[2]-bounds[0])*cos,extY=bounds[3]-bounds[1];
const scale=Math.min((WIDTH-2*PAD)/extX,(HEIGHT-2*PAD)/extY);
function toXY(pt){return [WIDTH/2+((pt[0]-(bounds[0]+bounds[2])/2)*cos)*scale,
                    HEIGHT/2-((pt[1]-(bounds[1]+bounds[3])/2))*scale]}
function pathD(geo){const polys=geo.type==='Polygon'?[geo.coordinates]:geo.type==='MultiPolygon'?geo.coordinates:[];
 return polys.flatMap(poly=>poly.map(ring=>ring.map((p,i)=>{const xy=toXY(p);return (i?'L':'M')+xy[0].toFixed(2)+' '+xy[1].toFixed(2)}).join('')+'Z')).join(' ')}
const NS='http://www.w3.org/2000/svg';
features.forEach(f=>{
 const p=document.createElementNS(NS,'path');p.setAttribute('d',pathD(f.geometry));
 p.setAttribute('class','strip');p.setAttribute('fill-rule','evenodd');
 p.addEventListener('mouseenter',e=>{tip.style.display='block';tip.textContent=f.id+' · '+f.zone+' · '+(f[sel.value]??'NA')+'%';});
 p.addEventListener('mousemove',e=>{const b=document.getElementById('mapwrap').getBoundingClientRect();tip.style.left=(e.clientX-b.left+12)+'px';tip.style.top=(e.clientY-b.top+12)+'px';});
 p.addEventListener('mouseleave',()=>tip.style.display='none');
 p.addEventListener('click',()=>showDetails(f));
 root.appendChild(p);f.node=p;
});
let view={x:0,y:0,k:1};
function applyView(){root.setAttribute('transform',`translate(${view.x} ${view.y}) scale(${view.k})`)}
function zoom(factor,cx=600,cy=345){const beforeX=(cx-view.x)/view.k,beforeY=(cy-view.y)/view.k;
view.k=Math.min(35,Math.max(1,view.k*factor));view.x=cx-beforeX*view.k;view.y=cy-beforeY*view.k;applyView()}
document.getElementById('zoomIn').onclick=()=>zoom(1.5);
document.getElementById('zoomOut').onclick=()=>zoom(1/1.5);
document.getElementById('reset').onclick=()=>{view={x:0,y:0,k:1};applyView()};
let drag=null;
svg.addEventListener('pointerdown',e=>{drag={x:e.clientX,y:e.clientY,startX:view.x,startY:view.y};svg.setPointerCapture(e.pointerId)});
svg.addEventListener('pointermove',e=>{if(!drag)return;const b=svg.getBoundingClientRect();view.x=drag.startX+(e.clientX-drag.x)*1200/b.width;view.y=drag.startY+(e.clientY-drag.y)*690/b.height;applyView()});
svg.addEventListener('pointerup',()=>drag=null);
svg.addEventListener('wheel',e=>{e.preventDefault();const b=svg.getBoundingClientRect();
zoom(e.deltaY<0?1.25:0.8,(e.clientX-b.left)*1200/b.width,(e.clientY-b.top)*690/b.height)},{passive:false});
function palette(v,max){if(v===null||v===undefined)return '#c9ccca';let t=Math.max(0,Math.min(1,v/Math.max(max,1e-9)));return `hsl(${(75-72*t).toFixed(0)} 42% ${(92-54*t).toFixed(0)}%)`}
function paint(){const m=sel.value;const vals=features.map(f=>f[m]).filter(v=>v!==null&&Number.isFinite(v));
const sorted=vals.slice().sort((a,b)=>a-b);const q95=sorted[Math.floor(.95*(sorted.length-1))]||1;
features.forEach(f=>f.node.setAttribute('fill',palette(f[m],q95)));
document.getElementById('upper').textContent=q95.toFixed(2)+'% (P95)';
}
sel.onchange=paint;
function showDetails(f){let h='<h2>'+f.id+'</h2><p>UTM zone: '+f.zone+'</p><table>';
for(const m of metrics){const v=f[m];h+='<tr><td>'+m.replace(/^pct_/,'').replaceAll('_',' ')+'</td><td>'+(v===null?'NA':v.toFixed(2)+'%')+'</td></tr>'}
document.getElementById('detail').innerHTML=h+'</table>'}
paint();
</script></body></html>"""
page=page.replace('__PAYLOAD__',payload.replace('</','<\\/'))
out=folder/'WWF_TMF_216_strips_DURABLE_offline_atlas.html'
out.write_text(page,encoding='utf-8')
print('Saved permanent standalone map:',out)
print('Size:',round(out.stat().st_size/1024/1024,2),'MB')
print('Contains',len(records),'embedded polygons and',len(cols),'TMF descriptors.')
print('Download the HTML and open locally; external Earth Engine/Leaflet/CDN NOT required.')
from IPython.display import HTML,display
display(HTML(f'<a href="/content/drive/MyDrive/Colab%20Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar/TMF_strip_descriptors/rgb_footprint_pixels/{out.name}" target="_blank">HTML file saved to Drive (download to view)</a>'))


### 4.G — Offline 30 m TMF pixel atlas

**This is the pixel-map counterpart to G.3D.** It exports seven **historical** TMF bands directly from Earth Engine as small, clipped 30 m GeoTIFF windows, converts them to paletted transparent PNGs, and embeds the PNGs together with the WWF polygon boundaries and G.3 statistics in **one HTML file**. Earth Engine is needed **only to build it**; colleagues can open the resulting HTML offline with no expiring links, external libraries, or hosted resources. **Nothing is uploaded into Earth Engine.**

Layers: **2014 status, 2015 status, first degradation year up to 2014, first deforestation year up to 2014, regrowth consecutive-state age in 2014, degraded consecutive-state age in 2014, annual class-transition count 1990–2014**. Dates beyond 2014 are explicitly masked for first-event dates. Each plot's downloaded GeoTIFF and compressed tile PNGs are cached, so reruns skip completed strips. Starts with a **six-strip preview** to validate that the cartography works, then set `PREVIEW_ONLY=False` for all 216. A simple geographic SVG displays the tiles (small local geographic reprojection approximation for rendering only); the original TMF grid statistics are unchanged.

**Caveats:** this maps *potential* RGB footprint = LiDAR survey polygon intersected with GeoTIFF rectangular bounds, **not confirmed valid RGB pixels**; no 10 cm imagery is downloaded. TMF first-event years are not latest-disturbance years. This is intended for exploratory spatial sampling, not precise pixel-scale positional measurement or Musanga species labels.

In [ ]:
# 4.G.1 — Durable 30 m TMF HTML atlas
# One Earth Engine 7-band download PER WWF plot; no WWF RGB data downloaded.
import json, base64, io, time, shutil, requests
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from PIL import Image
from rasterio.features import geometry_mask
from shapely.geometry import mapping
from IPython.display import display, HTML
import ee

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
assert (G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg').exists(), 'Run G.3 first'
foot=gpd.read_file(G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg').to_crs(4326)
assert len(foot)==216 and foot.plot_id.is_unique

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

PREVIEW_ONLY=True     # Run 6 first; then False for ALL 216
MAX_PLOTS_PREVIEW=6
REFRESH_CACHED=False
MAX_FAILURES=8
ROOT=G3_DIR/'durable_30m_atlas'
RAW=ROOT/'source_geotiff_cache'
PNG=ROOT/'transparent_png_tiles'
for f in [ROOT,RAW,PNG]: f.mkdir(parents=True,exist_ok=True)

# 30 m source pixels, via GEE. First event dates after acquisition are excluded.
jrc='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(jrc+'AnnualChanges').mosaic()
firstdeg=ee.ImageCollection(jrc+'DegradationYear').mosaic().select(0)
firstdef=ee.ImageCollection(jrc+'DeforestationYear').mosaic().select(0)
status14=annual.select('Dec2014')
status15=annual.select('Dec2015')
reg_age=ee.Image(0).toInt16()
deg_age=ee.Image(0).toInt16()
nchange=ee.Image(0).toInt16()
prev=None
for year in range(1990,2015):
    current=annual.select('Dec'+str(year))
    reg_age=reg_age.add(1).multiply(current.eq(4)).toInt16()
    deg_age=deg_age.add(1).multiply(current.eq(2)).toInt16()
    if prev is not None:nchange=nchange.add(current.neq(prev)).toInt16()
    prev=current
img=ee.Image.cat([
    status14.rename('status2014'),
    status15.rename('status2015'),
    firstdeg.where(firstdeg.gt(2014),0).rename('firstdeg'),
    firstdef.where(firstdef.gt(2014),0).rename('firstdef'),
    reg_age.rename('regrowth_age'),
    deg_age.rename('degraded_age'),
    nchange.rename('transitions')
]).unmask(0).toInt16()
layer_names=['status2014','status2015','firstdeg','firstdef',
             'regrowth_age','degraded_age','transitions']

# Fixed 256-entry colour tables; index 0 is transparent in all layers.
def grad_palette(start,end,maximum=50):
    a=np.array(start,float);b=np.array(end,float)
    pal=[(0,0,0)]
    for i in range(1,256):
        t=min(1,max(0,(i-1)/max(1,maximum-1)))
        pal.append(tuple(np.round(a+(b-a)*t).astype('uint8')))
    return pal
palette={}
palette['status2014']=[(0,0,0),(25,104,53),(203,156,57),(225,95,60),
                       (107,186,77),(54,144,193),(185,189,188)]+[(185,189,188)]*249
palette['status2015']=palette['status2014']
palette['firstdeg']=grad_palette((253,225,150),(155,29,68),33)
palette['firstdef']=grad_palette((255,223,163),(126,18,70),33)
palette['regrowth_age']=grad_palette((245,223,102),(24,121,73),25)
palette['degraded_age']=grad_palette((250,222,123),(153,52,53),25)
palette['transitions']=grad_palette((255,218,114),(135,22,90),5)

def encode_index(band,values):
    v=values.astype(np.int32)
    if band.startswith('status'): out=np.where((v>=1)&(v<=6),v,0)
    elif band in ('firstdeg','firstdef'): out=np.where((v>=1982)&(v<=2014),v-1981,0)
    else: out=np.clip(v,0,255)
    return out.astype('uint8')

# Preview takes first geographically diverse plots; all-plot run uses complete frame.
if PREVIEW_ONLY:
    selection=foot.sort_values('plot_id').groupby('utm_zone',sort=True).head(1).head(MAX_PLOTS_PREVIEW)
else:
    selection=foot.sort_values('plot_id')
print('Processing',len(selection),'RGB potential footprints; first run may need Google EE reads.')

def cached_tile_info(plot_id):
    meta=PNG/(plot_id+'.json')
    if not meta.exists(): return None
    try:
        item=json.loads(meta.read_text())
        if set(item['tiles'])==set(layer_names) and all((PNG/x).exists() for x in item['tiles'].values()):
            return item
    except Exception: pass
    return None

failed=[]
for i,row in enumerate(selection.itertuples(index=False),1):
    pid=str(row.plot_id)
    if cached_tile_info(pid) and not REFRESH_CACHED:
        if i%12==0 or PREVIEW_ONLY: print('Cached',i,'/',len(selection),pid)
        continue
    source_tif=RAW/(pid+'.tif')
    try:
        if REFRESH_CACHED or not source_tif.exists():
            polygon=ee.Geometry(mapping(row.geometry),proj='EPSG:4326',geodesic=False)
            clipped=img.clip(polygon)
            url=clipped.getDownloadURL({
                'name':'wwf_tmf_'+pid,
                'region':polygon,
                'scale':30,
                'crs':'EPSG:4326',
                'format':'GEO_TIFF',
                'filePerBand':False
            })
            # The GEE download URL is temporary, but output TIFF/PNG files are permanent.
            response=requests.get(url,timeout=180)
            response.raise_for_status()
            source_tif.write_bytes(response.content)
        with rasterio.open(source_tif) as ds:
            assert ds.count==7, f'{pid}: expected 7 bands, got {ds.count}'
            assert ds.crs.to_epsg()==4326, f'{pid}: unexpected output CRS {ds.crs}'
            arr=ds.read()
            inside=geometry_mask([mapping(row.geometry)],out_shape=(ds.height,ds.width),
                                 transform=ds.transform,invert=True)
            bounds=list(ds.bounds)
        tiles={}
        for band_index,band in enumerate(layer_names):
            q=encode_index(band,arr[band_index])
            q[~inside]=0
            pic=Image.fromarray(q,mode='P')
            flat=np.array(palette[band],dtype=np.uint8).ravel().tolist()
            pic.putpalette(flat)
            pngname=pid+'_'+band+'.png'
            pic.save(PNG/pngname,optimize=True,transparency=0)
            tiles[band]=pngname
        item={'plot_id':pid,'bounds':bounds,'tiles':tiles}
        (PNG/(pid+'.json')).write_text(json.dumps(item))
        if i%12==0 or PREVIEW_ONLY:print('Prepared',i,'/',len(selection),pid)
    except Exception as err:
        failed.append((pid,str(err)[:250]))
        print('FAIL',pid,str(err)[:180])
        if len(failed)>=MAX_FAILURES:
            print('Stopping after repeated errors; previous strips remain cached.')
            break
print('Failed plot IDs:',failed[:MAX_FAILURES])

# Embed one 30 m image per plot per selectable layer, plus all TMF strip descriptors.
records=[]
for row in selection.itertuples(index=False):
    pid=str(row.plot_id)
    item=cached_tile_info(pid)
    if item is None:continue
    props={name:round(float(getattr(row,name)),3)
           for name in foot.columns if name.startswith('pct_')
           and pd.notna(getattr(row,name))}
    records.append(dict(id=pid,zone=str(row.utm_zone),
       bounds=item['bounds'],geometry=mapping(row.geometry),props=props,
       tiles={k:'data:image/png;base64,'+
              base64.b64encode((PNG/v).read_bytes()).decode('ascii')
              for k,v in item['tiles'].items()}))
assert records, 'No completed image tiles found; inspect failures above.'
print('Packing HTML using',len(records),'plots and',len(layer_names),'layers')

# SVG drawing: pixels and polygons in the same approximate lon-lat projection.
# All pixel positioning comes from georeferenced GeoTIFF bounds; SVG is for exploration.
html_template=r"""<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>WWF 30 m TMF pixel atlas — 2014–2015</title>
<style>
body{margin:0;background:#f4f7f5;font-family:system-ui,Arial;color:#243d37}
header{background:#153e34;color:#fff;padding:14px 18px}h1{font-size:19px;margin:0 0 4px}
small{opacity:.8}section{background:white;padding:9px 16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap}
select,button{font:inherit;padding:5px;border:1px solid #ccd5cf;background:white;border-radius:5px}
main{display:grid;grid-template-columns:minmax(0,1fr) 280px;gap:12px;padding:12px}
#canvas{position:relative;border:1px solid #d4ddd5;border-radius:8px;background:#edf1e9}
svg{width:100%;height:75vh;min-height:450px;display:block;touch-action:none;cursor:grab}
#panel{background:white;padding:13px;border:1px solid #d5ded7;border-radius:8px;font-size:12px;overflow:auto;max-height:75vh}
#panel h3{margin-top:0}#panel table{border-collapse:collapse;width:100%}
td{padding:4px;border-bottom:1px solid #eee}td:last-child{text-align:right}
#tip{display:none;position:absolute;background:#153e34;color:white;padding:5px;border-radius:4px;font-size:12px;pointer-events:none}
@media(max-width:800px){main{grid-template-columns:1fr}svg{height:55vh}}
</style></head><body>
<header><h1>WWF RGB survey — 30 m TMF historical pixel atlas</h1><small id="subtitle">2014–2015 imagery context · offline HTML · historical forest classes and disturbance histories</small></header>
<section><label>Raster layer: <select id="layer"></select></label><button id="plus">Zoom +</button><button id="minus">Zoom −</button><button id="reset">Reset</button><small>Scroll to zoom; drag to pan; click a strip to inspect</small></section>
<main><div id="canvas"><svg id="svg" viewBox="0 0 1200 700"><rect width="1200" height="700" fill="#e9eee9"/><g id="scene"><g id="tiles"></g><g id="outlines"></g></g></svg><div id="tip"></div></div>
<aside id="panel"><h3>WWF strip statistics</h3><div id="stats">Click a strip to inspect its 2014 forest composition and historical disturbance descriptors.</div>
<p><b>Legend:</b> 2014/15 classes: dark green undisturbed, ochre degraded, orange deforested, light green regrowth, blue water, grey other. Year and age layers: pale yellow older/shorter to dark red/green newer/longer. Transparent = not applicable or outside footprint.</p>
<p><b>Note:</b> source 30 m TMF pixels clipped to potential WWF RGB footprints, not confirmed valid RGB. Geographical placement is approximate for SVG visualization. First-event year ≠ latest event. No WWF photos or species identifications included.</p></aside></main>
<script>
const DATA=__DATA__;
const LAYERS=['status2014','status2015','firstdeg','firstdef','regrowth_age','degraded_age','transitions'];
const TITLES=['TMF status 2014','TMF status 2015','First degradation year by 2014','First deforestation year by 2014','Consecutive regrowth-state years in 2014','Consecutive degraded-state years in 2014','Annual class transitions 1990–2014'];
const select=document.getElementById('layer'),tileG=document.getElementById('tiles'),outlineG=document.getElementById('outlines');
const root=document.getElementById('scene'),svg=document.getElementById('svg'),tip=document.getElementById('tip');
LAYERS.forEach((x,i)=>{const o=document.createElement('option');o.value=x;o.textContent=TITLES[i];select.appendChild(o)});
const BB=[Infinity,Infinity,-Infinity,-Infinity];
for(const f of DATA){const b=f.bounds;BB[0]=Math.min(BB[0],b[0]);BB[1]=Math.min(BB[1],b[1]);BB[2]=Math.max(BB[2],b[2]);BB[3]=Math.max(BB[3],b[3]);}
const midlat=(BB[1]+BB[3])/2,C=Math.cos(midlat*Math.PI/180);
const scale=Math.min(1100/((BB[2]-BB[0])*C),620/(BB[3]-BB[1]));
function xy(lon,lat){return [600+(lon-(BB[0]+BB[2])/2)*C*scale,350-(lat-(BB[1]+BB[3])/2)*scale]}
const NS='http://www.w3.org/2000/svg';
function outline(geo){const polys=geo.type==='Polygon'?[geo.coordinates]:(geo.coordinates||[]);
return polys.flatMap(poly=>poly.map(ring=>ring.map((p,i)=>{
 const q=xy(p[0],p[1]);return (i?'L':'M')+q[0].toFixed(1)+','+q[1].toFixed(1)}).join('')+'Z')).join(' ')}
for(const f of DATA){
  const b=f.bounds,p0=xy(b[0],b[3]),p1=xy(b[2],b[1]);
  const im=document.createElementNS(NS,'image');
  im.setAttribute('x',p0[0]);im.setAttribute('y',p0[1]);
  im.setAttribute('width',p1[0]-p0[0]);im.setAttribute('height',p1[1]-p0[1]);
  im.setAttribute('preserveAspectRatio','none');im.setAttribute('href',f.tiles.status2014);
  tileG.appendChild(im);f.image=im;
  const p=document.createElementNS(NS,'path');p.setAttribute('d',outline(f.geometry));
  p.setAttribute('fill','none');p.setAttribute('stroke','#263f39');
  p.setAttribute('stroke-width','.8');p.setAttribute('vector-effect','non-scaling-stroke');
  p.setAttribute('cursor','pointer');outlineG.appendChild(p);
  p.addEventListener('mouseenter',()=>{tip.textContent=f.id;tip.style.display='block'});
  p.addEventListener('mousemove',e=>{const b=document.getElementById('canvas').getBoundingClientRect();tip.style.left=(e.clientX-b.left+10)+'px';tip.style.top=(e.clientY-b.top+10)+'px'});
  p.addEventListener('mouseleave',()=>tip.style.display='none');
  p.addEventListener('click',()=>{document.getElementById('stats').innerHTML='<h3>'+f.id+'</h3><p>Zone '+f.zone+'</p><table>'+
    Object.entries(f.props).map(([k,v])=>'<tr><td>'+k.replace('pct_','').replaceAll('_',' ')+'</td><td>'+v.toFixed(2)+'%</td></tr>').join('')+'</table>'});
}
select.onchange=()=>{const layer=select.value;DATA.forEach(f=>f.image.setAttribute('href',f.tiles[layer]))};
let t={x:0,y:0,z:1};function redraw(){root.setAttribute('transform','translate('+t.x+' '+t.y+') scale('+t.z+')')}
function zoom(amount,x=600,y=350){const old=t.z;t.z=Math.max(1,Math.min(55,t.z*amount));t.x=x-(x-t.x)*t.z/old;t.y=y-(y-t.y)*t.z/old;redraw()}
document.getElementById('plus').onclick=()=>zoom(1.5);
document.getElementById('minus').onclick=()=>zoom(1/1.5);
document.getElementById('reset').onclick=()=>{t={x:0,y:0,z:1};redraw()};
let drag=null;
svg.addEventListener('pointerdown',e=>{drag={x:e.clientX,y:e.clientY,ox:t.x,oy:t.y};svg.setPointerCapture(e.pointerId)});
svg.addEventListener('pointermove',e=>{if(!drag)return;const b=svg.getBoundingClientRect();t.x=drag.ox+(e.clientX-drag.x)*1200/b.width;t.y=drag.oy+(e.clientY-drag.y)*700/b.height;redraw()});
svg.addEventListener('pointerup',()=>drag=null);
svg.addEventListener('wheel',e=>{e.preventDefault();const b=svg.getBoundingClientRect();zoom(e.deltaY<0?1.3:1/1.3,(e.clientX-b.left)*1200/b.width,(e.clientY-b.top)*700/b.height)},{passive:false});
document.getElementById('subtitle').textContent=DATA.length+' WWF RGB footprints · original 30 m TMF pixels · offline, shareable HTML';
</script></body></html>"""
file=ROOT/('WWF_TMF_30m_PILOT_OFFLINE.html' if PREVIEW_ONLY else 'WWF_TMF_30m_ALL216_OFFLINE.html')
page=html_template.replace('__DATA__',json.dumps(records,separators=(',',':')).replace('</','<\\/'))
file.write_text(page,encoding='utf-8')
print('Durable offline pixel atlas:',file)
print('Size:',round(file.stat().st_size/1024/1024,2),'MB')
print('Share by downloading this HTML; opening it requires NO Earth Engine account.')


## 5 — Hansen × TMF forest-history integration

**Purpose:** answer whether the WWF RGB survey adequately represents forest structural histories and potential Musanga recovery environments, at both **strip level** (equal-weight comparison) and **survey-wide pixel level** (pooled mapped-pixel composition). This is a descriptive sampling-frame audit, **not a validation of Musanga occurrence** or a claim of unbiased forest-area estimates.

Uses **Hansen Global Forest Change v1.13** (2000 canopy cover, loss years 2001–2014, gain 2000–2012) alongside JRC TMF 2014 forest class. For the 2014–2015 UAV survey, **2014 is the conservative reference year**. The 2015-sensitive table is deliberately NOT produced as exact flight dates are unknown; future loss (>2014) is ignored, avoiding temporal leakage.

**Categorical design:** Hansen 2000 canopy cover bins 0–20, 20–40, 40–60, 60–80, 80–100%; first/recorded stand-replacing forest loss **2001–05, 2006–10, 2011–14, or no loss detected through 2014**; Hansen 2000–12 gain; 2014 TMF categories 1–6; and **TMF class × Hansen loss cohort** (24 joint categories) sampled **on the same 30 m analysis grid**. No Hansen forest-loss year is treated as the age of a living regrowing stand: actual state recovery is assessed using the separate TMF regrowth-state age distributions saved by G.3.

**Outputs:** cached per-batch GEE histograms; per-strip comparison CSV and GeoPackage, pooled pixel cross-tab of TMF class × Hansen loss cohort, pooled forest-history summary, forest-canopy distributions, statistics across strips and comparison plots. All samples are from **LiDAR polygon ∩ TIFF bounding rectangle**, not yet a verified RGB valid-data mask. A Hansen loss pixel with no recent loss can have a loss *after* 2014; our category means no **detected loss by 2014**. Native Hansen and TMF grids differ, so the pixel-level cross-tab uses a common 30 m EE sampling scale; minor discrepancies with original G.3 native-class frequencies are expected.

Official GEE Hansen dataset: `UMD/hansen/global_forest_change_2025_v1_13`. Hansen forest loss is *stand-replacement loss*, not selective degradation or Musanga occurrence. Annual detection algorithms changed around the Landsat 8 introduction: compare historical cohorts with care.

In [ ]:
# 5.A — Calculate Hansen × TMF joint statistics
# Runs independently of G.3/G.3C as long as their saved footprint GeoPackage exists.
from pathlib import Path
import json, tempfile, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from shapely.geometry import mapping
from google.colab import drive
import ee

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G4_DIR=WORK_DIR/'TMF_strip_descriptors'/'hansen_tmf_2014'
G4_DIR.mkdir(parents=True, exist_ok=True)
src=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert src.exists(), 'Missing G.3 RGB footprint GeoPackage. Run G.3 first.'
strips=gpd.read_file(src).to_crs(4326).sort_values('plot_id').reset_index(drop=True)
assert len(strips)==216 and strips.plot_id.is_unique

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G4_BATCH=6
G4_FORCE_RECOMPUTE=False
G4_MAX_STRIPS=None    # None => all 216; set 12 for a short test run
REF_YEAR=2014         # conservative: WWF RGB flights in 2014/15, dates unknown
SCALE=30
assert REF_YEAR==2014, 'G.4 definitions explicitly refer to 2014.'
hansen=ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic().select('Dec2014')
land=hansen.select('datamask').eq(1)
status=tmf.toInt16()
treecover=hansen.select('treecover2000')
lossyear=hansen.select('lossyear').toInt16()
gain=hansen.select('gain').toInt16()
# Hansen 'lossyear' is 1 for 2001, ... 14 for 2014, 25 for 2025.
# 0 = no loss through 2014 (including pixels first lost AFTER 2014).
loss_cohort=(ee.Image(0).where(lossyear.gte(1).And(lossyear.lte(5)),1)
    .where(lossyear.gte(6).And(lossyear.lte(10)),2)
    .where(lossyear.gte(11).And(lossyear.lte(14)),3)).toInt16()
# Store annual 2001...2014 loss category; zero denotes no detected loss through 2014.
loss_year_asof2014=lossyear.where(lossyear.gt(14),0).toInt16()
# 2000 tree canopy bins: 1=0–19,2=20–39,3=40–59,4=60–79,5=80–100.
cover_bin=(treecover.divide(20).floor().add(1).min(5)).toInt16()
# Joint category: TMF class x10 + Hansen loss cohort; e.g. 43 = regrowth, loss 2011–14.
joint=status.multiply(10).add(loss_cohort).toInt16()
# Exclude water/no data and missing TMF classes from comparable denominators.
valid=land.And(status.gte(1)).And(status.lte(6))
cat=ee.Image.cat([
    status.rename('tmf_status'),
    cover_bin.rename('canopy2000_bin'),
    loss_cohort.rename('hansen_loss_cohort'),
    loss_year_asof2014.rename('hansen_loss_year'),
    gain.rename('hansen_gain_2000_2012'),
    joint.rename('tmf_x_hansen')
]).updateMask(valid)
cat_bands=['tmf_status','canopy2000_bin','hansen_loss_cohort',
           'hansen_loss_year','hansen_gain_2000_2012','tmf_x_hansen']
coverage=treecover.updateMask(valid).rename('canopy2000_mean')

# Cache batches so a Colab interruption does not lose completed work.
subset=strips if G4_MAX_STRIPS is None else strips.head(G4_MAX_STRIPS)
records=[]
for start in range(0,len(subset),G4_BATCH):
    chunk=subset.iloc[start:start+G4_BATCH]
    cache=G4_DIR/f'batch_{start:03d}_{start+len(chunk):03d}.json'
    if cache.exists() and not G4_FORCE_RECOMPUTE:
        part=json.loads(cache.read_text())
        records.extend(part)
        print('Reused cached batch',cache.name)
        continue
    feats=ee.FeatureCollection([
        ee.Feature(ee.Geometry(mapping(row.geometry),geodesic=False),
                   {'plot_id':str(row.plot_id)})
        for row in chunk.itertuples(index=False)
    ])
    histfc=cat.reduceRegions(collection=feats,reducer=ee.Reducer.frequencyHistogram(),
                             scale=SCALE,tileScale=4)
    meanfc=coverage.reduceRegions(collection=feats,reducer=ee.Reducer.mean(),
                                  scale=SCALE,tileScale=4)
    hrows=histfc.getInfo()['features']
    mrows=meanfc.getInfo()['features']
    means={f['properties']['plot_id']:f['properties'].get('mean') for f in mrows}
    part=[]
    for f in hrows:
        pr=f['properties'];pid=pr['plot_id']
        d={'plot_id':pid,'canopy2000_mean':means.get(pid)}
        for band in cat_bands:
            h=pr.get(band) or {}
            d[band]={str(k):int(round(v)) for k,v in h.items()}
        part.append(d)
    assert len(part)==len(chunk),'Earth Engine returned incomplete batch'
    cache.write_text(json.dumps(part))
    records.extend(part)
    print('Processed TMF × Hansen strips',start+1,'to',start+len(chunk),'/',len(subset))

assert len(records)==len(subset) and len({d['plot_id'] for d in records})==len(records)
(G4_DIR/'WWF_TMF_Hansen_all_histograms_2014.json').write_text(json.dumps(records,indent=1))
loss_labels={0:'No loss detected through 2014',1:'Loss 2001–2005',
             2:'Loss 2006–2010',3:'Loss 2011–2014'}
tmf_labels={1:'Undisturbed',2:'Degraded',3:'Deforested',
            4:'Regrowth',5:'Water',6:'Other'}
def count(h,k): return int(h.get(str(k),0))
def pct(n,d): return 100*n/d if d else float('nan')

per=[]
pooled={b:{} for b in cat_bands}
for rec in records:
    p={'plot_id':rec['plot_id'],'hansen_treecover2000_mean':rec['canopy2000_mean']}
    for b in cat_bands:
        hist=rec[b]
        for k,n in hist.items():
            pooled[b][int(k)]=pooled[b].get(int(k),0)+int(n)
    total=sum(rec['tmf_status'].values())
    p['n_common_valid_tmf_hansen_pixels']=total
    for k,label in tmf_labels.items():
        p['pct_tmf2014_'+label.lower()]=pct(count(rec['tmf_status'],k),total)
    for k in range(1,6):
        p[f'pct_hansen_canopy2000_bin{k}']=pct(count(rec['canopy2000_bin'],k),total)
    for k,label in loss_labels.items():
        p[f'pct_hansen_loss_cohort{k}']=pct(count(rec['hansen_loss_cohort'],k),total)
    for k in range(0,15):
        p[f'pct_hansen_lossyear_code{k:02d}']=pct(count(rec['hansen_loss_year'],k),total)
    p['pct_hansen_gain_2000_2012']=pct(count(rec['hansen_gain_2000_2012'],1),total)
    for cls in [1,2,3,4]:
        class_total=count(rec['tmf_status'],cls)
        for coh in range(4):
            nn=count(rec['tmf_x_hansen'],cls*10+coh)
            p[f'pct_joint_tmf{cls}_loss{coh}_of_footprint']=pct(nn,total)
            p[f'pct_hansen_loss{coh}_within_tmf{cls}']=pct(nn,class_total)
    # Regrowth–loss overlap: a diagnostic for recovery histories, not a stand-age estimator.
    p['pct_regrowth_with_loss_by2014_of_footprint']=sum(
        count(rec['tmf_x_hansen'],40+k) for k in [1,2,3])*100/max(total,1)
    per.append(p)

per=pd.DataFrame(per)
strips_merged=strips.merge(per,on='plot_id',validate='one_to_one')
out_csv=G4_DIR/'WWF_all_strips_TMF_Hansen_2014_descriptors.csv'
out_gpkg=G4_DIR/'WWF_all_strips_TMF_Hansen_2014_descriptors.gpkg'
strips_merged.drop(columns='geometry').to_csv(out_csv,index=False)
strips_merged.to_file(out_gpkg,driver='GPKG')
print('Saved',out_csv)
print('Saved',out_gpkg)

# Pool HISTOGRAM COUNTS, not percentages: every common valid TMF/Hansen observation
# has equal weight in these tables. Avoid summing strip %s.
pooled_long=[]
for band,values in pooled.items():
    denominator=sum(values.values())
    for key,n in sorted(values.items()):
        pooled_long.append(dict(band=band,value=key,count=n,
                                pct=100*n/denominator if denominator else np.nan,
                                total=denominator))
pooled_df=pd.DataFrame(pooled_long)
pooled_df.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_pooled_histograms.csv',index=False)

cross=pd.DataFrame(0,dtype=int,index=[tmf_labels[i] for i in range(1,7)],
                   columns=[loss_labels[j] for j in range(4)])
for k,n in pooled['tmf_x_hansen'].items():
    cls,coh=divmod(k,10)
    if cls in tmf_labels and coh in loss_labels:
        cross.loc[tmf_labels[cls],loss_labels[coh]]+=n
cross.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_joint_counts.csv')
cross_pct=100*cross/cross.values.sum()
cross_pct.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_joint_pct_all_pixels.csv')
rowpct=cross.div(cross.sum(axis=1).replace(0,np.nan),axis=0)*100
rowpct.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_joint_pct_within_tmf_class.csv')

# Across-strip descriptive statistics; equal strip weighting, *not pooled pixels*.
metrics=[c for c in per.columns if c.startswith('pct_') or c=='hansen_treecover2000_mean']
summ=[]
for c in metrics:
    x=pd.to_numeric(per[c],errors='coerce').dropna()
    if not len(x): continue
    q=x.quantile([.10,.25,.50,.75,.90,.95])
    summ.append(dict(metric=c,n=len(x),mean=x.mean(),std=x.std(),
                     min=x.min(),p10=q.loc[.1],p25=q.loc[.25],median=q.loc[.5],
                     p75=q.loc[.75],p90=q.loc[.9],p95=q.loc[.95],max=x.max(),
                     zero_pct=100*(x==0).mean()))
summ=pd.DataFrame(summ)
summ.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_across_strip_stats.csv',index=False)

# Dedicated readable global summary (including conditional regrowth overlap).
global_n=int(cross.values.sum())
global_stats=[]
for name,nums in [
    ('Common valid 30 m pixel observations',global_n),
    ('TMF regrowth pixels',int(cross.loc['Regrowth'].sum())),
    ('TMF regrowth with Hansen loss 2001–2014',int(cross.loc['Regrowth'].iloc[1:].sum())),
    ('TMF regrowth with Hansen loss 2011–2014',int(cross.loc['Regrowth'].iloc[3])),
    ('Hansen loss 2011–2014 in any TMF status',int(cross.iloc[:,3].sum())),
    ('Hansen loss 2001–2014 in any TMF status',int(cross.iloc[:,1:].values.sum())),
]:
    global_stats.append(dict(description=name,pixels=nums,
                             pct_all=100*nums/global_n if global_n else np.nan))
global_frame=pd.DataFrame(global_stats)
global_frame.to_csv(G4_DIR/'WWF_TMF_Hansen_2014_global_summary.csv',index=False)

print('\n=== GLOBAL pooled Hansen × TMF pixels, 2014 reference ===')
display(global_frame.round(2))
display(cross)
print('\n=== PERCENT OF EACH TMF CLASS BY HANSEN LOSS COHORT ===')
display(rowpct.round(2))
print('\n=== BETWEEN-STRIP variability (selected indicators) ===')
keys=['pct_hansen_loss_cohort3','pct_hansen_loss_cohort2',
      'pct_hansen_gain_2000_2012','pct_tmf2014_regrowth',
      'pct_regrowth_with_loss_by2014_of_footprint',
      'hansen_treecover2000_mean']
display(summ[summ.metric.isin(keys)].round(2))
print('\nTop WWF strips with overlap of regrowth and pre-2015 Hansen forest loss')
display(per.sort_values('pct_regrowth_with_loss_by2014_of_footprint',
                        ascending=False)[['plot_id','pct_tmf2014_regrowth',
                        'pct_regrowth_with_loss_by2014_of_footprint',
                        'pct_hansen_loss_cohort3','pct_hansen_loss_cohort2',
                        'pct_hansen_gain_2000_2012']].head(20).round(2))

# Plots: write to temporary LOCAL filesystem before copying to mounted Drive.
local=Path(tempfile.mkdtemp(prefix='wwf_hansen_tmf_'))
def publish(fig,filename):
    destination=G4_DIR/filename
    source=local/filename
    fig.savefig(source,dpi=110,bbox_inches='tight')
    plt.close(fig)
    shutil.copyfile(source,destination)
    print('Saved figure:',destination)
    display(IPImage(filename=str(source),width=1000))
fig,axs=plt.subplots(1,2,figsize=(13,5))
cross_pct.plot(kind='bar',stacked=True,ax=axs[0],width=.75)
axs[0].set(title='TMF 2014 classes by Hansen loss cohort (pooled)',
           xlabel='TMF 2014 class',ylabel='Percent of all joint pixels')
axs[0].tick_params(axis='x',rotation=40)
rowpct.plot(kind='bar',stacked=True,ax=axs[1],width=.75)
axs[1].set(title='Within each TMF class: Hansen loss cohorts',
           xlabel='TMF 2014 class',ylabel='% of TMF class')
axs[1].tick_params(axis='x',rotation=40)
fig.tight_layout();publish(fig,'TMF_Hansen_cross_class_composition.png')

fig,axs=plt.subplots(2,2,figsize=(13,9))
plot_keys=['pct_hansen_loss_cohort3','pct_tmf2014_regrowth',
           'pct_regrowth_with_loss_by2014_of_footprint',
           'hansen_treecover2000_mean']
for ax,key in zip(axs.flat,plot_keys):
    values=pd.to_numeric(per[key],errors='coerce').dropna()
    ax.hist(values,bins=30,edgecolor='white')
    ax.axvline(values.median(),ls='--',color='black')
    ax.set(title=key.replace('_',' '),xlabel='Percent (or 2000 canopy-cover %)',
           ylabel='Number of WWF strips')
fig.tight_layout();publish(fig,'TMF_Hansen_recovery_gradient_strip_distributions.png')

fig,ax=plt.subplots(figsize=(11,5))
loss_values=pooled['hansen_loss_year']
years=np.arange(2001,2015)
counts=np.array([loss_values.get(int(y-2000),0) for y in years])
ax.bar(years,counts)
ax.set(title='Hansen annual stand-replacing forest loss before WWF RGB acquisition',
       xlabel='Loss year (2001–2014)',ylabel='30 m mapped pixel observations')
ax.set_xticks(years)
ax.tick_params(axis='x',labelrotation=45)
fig.tight_layout();publish(fig,'Hansen_annual_loss_2001_2014.png')

fig,ax=plt.subplots(figsize=(10,5))
cover_counts=[pooled['canopy2000_bin'].get(i,0) for i in range(1,6)]
ax.bar(['0–19%','20–39%','40–59%','60–79%','80–100%'],cover_counts)
ax.set(title='Hansen year-2000 canopy cover across WWF RGB footprints',
       ylabel='Common valid TMF/Hansen 30 m pixel observations',xlabel='Canopy cover 2000')
fig.tight_layout();publish(fig,'Hansen_canopy2000_global_distribution.png')

print('\nG.4 DONE. Output directory:',G4_DIR)
print('IMPORTANT: this is a description of WWF sampling opportunities, NOT a')
print('population-unbiased forest area estimate and NOT Musanga occurrence evidence.')
print('TMF/Hansen pixels need native WWF RGB inspection before choosing circles.')


## 6 — Integrated ecological assessment

**Scientific goal:** evaluate which disturbance/recovery environments are represented in the WWF RGB survey, their global abundance, distribution across **independent strips**, and potential usefulness for Musanga annotations. G.4 offers underlying paired TMF/Hansen data; G.5 assigns an **explicit, mutually exclusive ecological sampling stratum to each 30 m pixel**, then compares strips. A second, nonexclusive **regrowth age × Hansen loss-cohort** cross-tabulation preserves important historical information.

**Sampling-stratum rules (ordered precedence):**
1. **Young TMF regrowth** (1–3 consecutive years in TMF regrowth state as of 2014), whether Hansen detects forest loss or not.
2. **Intermediate regrowth** (4–7 years).
3. **Longer TMF regrowth** (8+ years).
4. **TMF degraded without Hansen loss through 2014** (possible canopy disturbance not detected as stand-replacing loss).
5. **TMF degraded with Hansen loss 2001–2014** (stronger/mixed disturbance evidence).
6. **Recent Hansen loss 2011–2014**, excluding already-assigned regrowth/degraded areas (not assumed to have recovered).
7. **Historical Hansen loss 2001–2010**, excluding already-assigned classes (not assumed to be secondary forest).
8. **Stable high-canopy reference** (TMF undisturbed, no Hansen loss through 2014, Hansen 2000 canopy cover ≥80%).
9. **Stable lower-canopy reference** (TMF undisturbed, no Hansen loss through 2014, Hansen 2000 canopy cover <80%).
10. **Other / ambiguous TMF–Hansen combinations**, including open/other/water and anomalies.

**Interpretation safeguards:** These are **operational sampling environments**, **not measured tree ages**. TMF regrowth-state duration is not stand age. Hansen 2000 canopy cover is a historical baseline and *not* 2014 canopy closure. Hansen 'no loss' means no *detected stand-replacing loss before 2015* (not proof of no disturbance). The G.5 classification is applied to valid Hansen land pixels and valid TMF status; others are excluded.

**Statistics and outputs:** pooled counts and fractions by stratum, equal-weight strip statistics (mean, SD, quantiles, share of strips with zero), number of strips exceeding **0.5%, 1%, 2%, 5%, 10%** of a stratum, top strips for every environment, cross-tabulation of regrowth-state duration × Hansen loss cohort, CSV/GPKG and comparative figures. A separate **G.5B** checks 125 m circle-neighbourhood composition (>=70% target stratum), respecting interior survey boundaries: this measures *candidate centre pixels*, not independent annotation circles. Processing uses saved WWF footprint polygons, GEE batch caching, and no WWF RGB downloads.

In [ ]:
# 6.A — Classify integrated ecological strata
import json, tempfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from shapely.geometry import mapping
from google.colab import drive
import ee

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G5_DIR=WORK_DIR/'TMF_strip_descriptors'/'integrated_ecological_gradient_2014'
G5_DIR.mkdir(parents=True,exist_ok=True)
source=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert source.exists(), 'G.3 potential RGB footprint GeoPackage missing'
foot=gpd.read_file(source).to_crs(4326).sort_values('plot_id').reset_index(drop=True)
assert len(foot)==216 and foot.plot_id.is_unique
EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

G5_BATCH=6
G5_FORCE=False
G5_TEST_N=None   # e.g. 12 for short testing; None means all 216
G5_SCALE=30
# Distinguish useful ecological environments rather than claiming stand ages.
classes={
    1:'Young TMF regrowth (1–3 yr state)',
    2:'Intermediate TMF regrowth (4–7 yr state)',
    3:'Longer TMF regrowth (8+ yr state)',
    4:'TMF degraded / no Hansen loss through 2014',
    5:'TMF degraded / Hansen loss by 2014',
    6:'Recent Hansen loss (2011–14), other TMF states',
    7:'Historical Hansen loss (2001–10), other TMF states',
    8:'Undisturbed / high canopy in 2000 / no Hansen loss',
    9:'Undisturbed / lower canopy in 2000 / no Hansen loss',
    10:'Other or ambiguous combinations'
}
# Easy-to-read immutable codebook for downstream app/sampling.
(G5_DIR/'G5_ecological_class_codebook.json').write_text(json.dumps(classes,indent=2))
h=ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
tmfstack=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
st=tmfstack.select('Dec2014').toInt16()
ly=h.select('lossyear').toInt16()
c00=h.select('treecover2000').toInt16()
valid=h.select('datamask').eq(1).And(st.gte(1)).And(st.lte(6))
loss=(ee.Image(0).where(ly.gte(1).And(ly.lte(5)),1)
    .where(ly.gte(6).And(ly.lte(10)),2)
    .where(ly.gte(11).And(ly.lte(14)),3)).toInt16()
age=ee.Image.constant(0).toInt16()
for y in range(1990,2015):
    cur=tmfstack.select('Dec'+str(y))
    age=age.add(1).multiply(cur.eq(4)).toInt16()
reg=st.eq(4)
# Precedence: specific regrowth age -> degraded -> loss events -> stable references.
eco=ee.Image(10).toInt16()
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.gte(80)),8)
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.lt(80)),9)
eco=eco.where(loss.gte(1).And(loss.lte(2)),7)
eco=eco.where(loss.eq(3),6)
eco=eco.where(st.eq(2).And(loss.eq(0)),4)
eco=eco.where(st.eq(2).And(loss.gt(0)),5)
eco=eco.where(reg.And(age.gte(1)).And(age.lte(3)),1)
eco=eco.where(reg.And(age.gte(4)).And(age.lte(7)),2)
eco=eco.where(reg.And(age.gte(8)),3)
eco=eco.updateMask(valid).rename('ecological_stratum')
# Nonexclusive diagnostics: all combinations of age group and Hansen loss cohort.
agegrp=(ee.Image(0).where(reg.And(age.gte(1)).And(age.lte(3)),1)
       .where(reg.And(age.gte(4)).And(age.lte(7)),2)
       .where(reg.And(age.gte(8)),3)).toInt16()
reg_cross=(agegrp.multiply(10).add(loss)).updateMask(reg.And(valid)).rename('regrowth_age_x_loss')
all_bands=ee.Image.cat(eco,reg_cross)
selection=foot if G5_TEST_N is None else foot.head(G5_TEST_N)
data=[]
for begin in range(0,len(selection),G5_BATCH):
    part=selection.iloc[begin:begin+G5_BATCH]
    cache=G5_DIR/f'g5_hist_{begin:03d}_{begin+len(part):03d}.json'
    if cache.exists() and not G5_FORCE:
        rows=json.loads(cache.read_text())
        data.extend(rows)
        print('Cached',cache.name)
        continue
    fc=ee.FeatureCollection([
        ee.Feature(ee.Geometry(mapping(row.geometry),geodesic=False),
                   {'plot_id':str(row.plot_id)}) for row in part.itertuples(index=False)])
    response=all_bands.reduceRegions(collection=fc,
        reducer=ee.Reducer.frequencyHistogram(),scale=G5_SCALE,
        tileScale=4).getInfo()['features']
    rows=[]
    for f in response:
        p=f['properties']
        rows.append(dict(plot_id=p['plot_id'],
            strata={str(k):int(round(v)) for k,v in
                (p.get('ecological_stratum') or {}).items()},
            regrowth_age_x_loss={str(k):int(round(v)) for k,v in
                (p.get('regrowth_age_x_loss') or {}).items()}))
    assert len(rows)==len(part)
    cache.write_text(json.dumps(rows))
    data.extend(rows)
    print('Processed',begin+1,'–',begin+len(part),'/',len(selection))
assert len(data)==len(selection) and len({x['plot_id'] for x in data})==len(data)
(G5_DIR/'WWF_G5_all_strips_pixel_histograms.json').write_text(json.dumps(data,indent=1))
global_count={k:0 for k in classes}
global_age_loss=np.zeros((3,4),dtype=np.int64)
strip_rows=[]
for d in data:
    n=sum(d['strata'].values())
    row=dict(plot_id=d['plot_id'],n_common_valid_pixels=n)
    for k in classes:
        count=int(d['strata'].get(str(k),0))
        global_count[k]+=count
        row['count_stratum_'+str(k)]=count
        row['pct_stratum_'+str(k)]=100*count/n if n else np.nan
    cross=d['regrowth_age_x_loss']
    for a in range(1,4):
        for l in range(4):
            v=int(cross.get(str(a*10+l),0))
            global_age_loss[a-1,l]+=v
            row[f'count_reg_age{a}_hansen_loss{l}']=v
    strip_rows.append(row)
stripdata=pd.DataFrame(strip_rows)
full=foot.merge(stripdata,on='plot_id',validate='one_to_one')
full.drop(columns='geometry').to_csv(G5_DIR/'WWF_G5_integrated_strata_per_strip.csv',index=False)
full.to_file(G5_DIR/'WWF_G5_integrated_strata_per_strip.gpkg',driver='GPKG')
pooled_n=sum(global_count.values())
global_table=pd.DataFrame([dict(code=k,stratum=name,pixels=global_count[k],
    pct_of_valid_pixels=100*global_count[k]/pooled_n if pooled_n else np.nan,
    pixel_area_ha_nominal=global_count[k]*0.09) for k,name in classes.items()])
global_table.to_csv(G5_DIR/'WWF_G5_global_integrated_strata.csv',index=False)
age_matrix=pd.DataFrame(global_age_loss,
    index=['Regrowth state 1–3y','Regrowth state 4–7y','Regrowth state 8+y'],
    columns=['No Hansen loss through 2014','Loss 2001–05','Loss 2006–10','Loss 2011–14'])
age_matrix.to_csv(G5_DIR/'WWF_G5_regrowth_age_x_hansen_loss_counts.csv')
(age_matrix.div(age_matrix.sum(axis=1),axis=0)*100).to_csv(
    G5_DIR/'WWF_G5_regrowth_age_x_hansen_loss_within_age_pct.csv')
stat=[]
for k,name in classes.items():
    z=stripdata['pct_stratum_'+str(k)].dropna()
    q=z.quantile([.1,.25,.5,.75,.9,.95])
    stat.append(dict(code=k,stratum=name,n=len(z),mean=z.mean(),sd=z.std(),
        p10=q.loc[.1],p25=q.loc[.25],median=q.loc[.5],p75=q.loc[.75],
        p90=q.loc[.9],p95=q.loc[.95],maximum=z.max(),pct_strips_zero=100*(z==0).mean(),
        strips_ge_0_5pct=int((z>=.5).sum()),strips_ge_1pct=int((z>=1).sum()),
        strips_ge_2pct=int((z>=2).sum()),strips_ge_5pct=int((z>=5).sum()),
        strips_ge_10pct=int((z>=10).sum())))
stat=pd.DataFrame(stat)
stat.to_csv(G5_DIR/'WWF_G5_stratum_replication_across_strips.csv',index=False)
ranked=[]
for k,name in classes.items():
    for rank,(idx,rr) in enumerate(stripdata.sort_values('pct_stratum_'+str(k),
                                                ascending=False).head(15).iterrows(),1):
        ranked.append(dict(code=k,stratum=name,rank=rank,plot_id=rr.plot_id,
                           share_pct=rr['pct_stratum_'+str(k)],
                           n_pixels=rr['count_stratum_'+str(k)]))
pd.DataFrame(ranked).to_csv(G5_DIR/'WWF_G5_top15_strips_each_stratum.csv',index=False)
print('G5 — POOLED GLOBAL DISTRIBUTION')
display(global_table.round(2))
print('G5 — BETWEEN-STRIP REPLICATION')
display(stat.round(2))
print('G5 — REGROWTH STATE AGE × HANSEN LOSS COHORT (PIXEL COUNTS)')
display(age_matrix)
print('Row-normalized % within each TMF regrowth-state age')
display((age_matrix.div(age_matrix.sum(axis=1),axis=0)*100).round(2))

# Compact figures. Render locally before copying to mounted Drive.
tmp=Path(tempfile.mkdtemp(prefix='g5_wwf_'))
def figure_out(fig,name):
    src=tmp/name
    fig.savefig(src,dpi=110,bbox_inches='tight')
    plt.close(fig)
    shutil.copyfile(src,G5_DIR/name)
    display(IPImage(filename=str(src),width=1000))
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].bar(global_table['code'].astype(str),global_table['pct_of_valid_pixels'])
axes[0].set(title='All WWF RGB footprints: share of each ecological stratum',
            xlabel='Ecological stratum code (see table)',ylabel='% of valid TMF×Hansen pixels')
axes[1].bar(stat['code'].astype(str),stat['strips_ge_1pct'])
axes[1].set(title='Representation: number of WWF strips ≥1% in each stratum',
            xlabel='Ecological stratum code',ylabel='Number of strips (of 216)')
fig.tight_layout()
figure_out(fig,'G5_global_composition_and_replication.png')
fig,ax=plt.subplots(figsize=(11,4.5))
im=ax.imshow(age_matrix.to_numpy(),aspect='auto',cmap='viridis')
ax.set_xticks(range(4),age_matrix.columns,rotation=25,ha='right')
ax.set_yticks(range(3),age_matrix.index)
for y in range(3):
    for x in range(4):
        ax.text(x,y,f'{age_matrix.iloc[y,x]:,}',ha='center',va='center',fontsize=9,
                color='white' if age_matrix.iloc[y,x]>(age_matrix.to_numpy().max()*.5) else 'black')
ax.set(title='TMF regrowth-state duration × Hansen forest-loss history')
fig.colorbar(im,ax=ax,label='30 m pixel observations')
fig.tight_layout()
figure_out(fig,'G5_regrowth_age_by_hansen_loss_heatmap.png')
# Distribution of ecological gradients by geography (rank strip rows, no geodata download).
fig,ax=plt.subplots(figsize=(11,7))
heat=stripdata.set_index('plot_id')[[f'pct_stratum_{k}' for k in classes]]
order=heat.sort_values(['pct_stratum_1','pct_stratum_2','pct_stratum_3'],
                       ascending=False).index
arr=heat.loc[order].to_numpy()
im=ax.imshow(arr,aspect='auto',interpolation='nearest',cmap='viridis',
             vmin=0,vmax=np.nanpercentile(arr,98))
ax.set_xticks(range(len(classes)),[str(k) for k in classes])
ax.set_yticks(np.arange(0,len(heat),15),order[::15],fontsize=7)
ax.set(xlabel='Ecological stratum code',title='216 WWF strips: disturbance/recovery gradient (sorted by regrowth)')
fig.colorbar(im,ax=ax,label='% of strip footprint')
fig.tight_layout()
figure_out(fig,'G5_all_strips_ecological_composition_heatmap.png')
print('G.5 finished, data and figures:',G5_DIR)
print('Caution: none of these classes establish true stand age or Musanga occurrence.')
print('G.5B can optionally test whether each stratum occupies coherent 125 m neighbourhoods.')


### 6.B — 125 m neighbourhood feasibility

Run **after G.5** if you want to know whether the ecological strata occupy patches large enough for the existing **125 m diameter** annotation circles. On a 30 m working grid, test **candidate centre pixels** where at least 70% of a 62.5 m radius neighbourhood belongs to the target ecological stratum; the centre must be at least 62.5 m inside the *potential* RGB footprint. The resulting candidate counts are **correlated neighbouring centres, not independent 125 m circles**; they provide a feasibility screen only. Final locations still require valid-RGB/no-data verification and spatial separation.

The class image uses exactly the same 2014 TMF/Hansen rules as G.5. This is a separate (potentially more computationally intensive) Earth Engine query, cached in small batches.

In [ ]:
# 6.B.1 — Test 125 m circular neighbourhood feasibility
# Run directly after G.5 in same runtime: reuses ee image 'eco', footprints 'foot' and 'G5_DIR'.
assert 'eco' in globals() and 'foot' in globals() and 'G5_DIR' in globals(), 'Run G.5 first.'
import json, numpy as np, pandas as pd
from shapely.geometry import mapping
G5B_BATCH=3
G5B_FORCE=False
CIRCLE_RADIUS_M=62.5
TARGET_FRACTION=.70
TARGET_CLASSES=[1,2,3,4,5,6,7,8,9]
# A 125 m disk radius is 62.5 m; running mean across its 30 m TMF samples.
# Because eco is masked beyond valid forest pixels, unmask(0) before neighborhood.
# The threshold is applied against the WHOLE circular neighbourhood, not
# conditional on already classified pixels.
kern=ee.Kernel.circle(radius=CIRCLE_RADIUS_M,units='meters',normalize=False)
coherent=ee.Image.cat([
    eco.eq(k).unmask(0).reduceNeighborhood(
        reducer=ee.Reducer.mean(),kernel=kern,skipMasked=False
    ).gte(TARGET_FRACTION).rename('candidate_'+str(k)).toUint8()
    for k in TARGET_CLASSES
])
records=[]
for start in range(0,len(foot),G5B_BATCH):
    chunk=foot.iloc[start:start+G5B_BATCH]
    fn=G5_DIR/f'circle_centres_{start:03d}_{start+len(chunk):03d}.json'
    if fn.exists() and not G5B_FORCE:
        records.extend(json.loads(fn.read_text()))
        print('Cached',fn.name)
        continue
    feats=[]
    for row in chunk.itertuples(index=False):
        geo=ee.Geometry(mapping(row.geometry),geodesic=False)
        # Avoid counting candidate centres whose 125 m circle crosses survey boundary.
        inner=geo.buffer(-CIRCLE_RADIUS_M,maxError=5)
        feats.append(ee.Feature(inner,{'plot_id':str(row.plot_id)}))
    result=coherent.reduceRegions(
        collection=ee.FeatureCollection(feats),reducer=ee.Reducer.sum(),
        scale=30,tileScale=4).getInfo()['features']
    rows=[]
    for f in result:
        p=f['properties']
        d={'plot_id':p['plot_id']}
        for k in TARGET_CLASSES:
            value=p.get('candidate_'+str(k))
            d['candidate_centres_k'+str(k)]=int(round(value or 0))
        rows.append(d)
    assert len(rows)==len(chunk)
    fn.write_text(json.dumps(rows))
    records.extend(rows)
    print('Processed 125m neighbourhoods',start+1,'–',start+len(chunk),'/',len(foot))
assert len(records)==216
screen=pd.DataFrame(records)
screen.to_csv(G5_DIR/'WWF_G5_125m_circle_candidate_centre_counts.csv',index=False)
availability=[]
for k in TARGET_CLASSES:
    values=screen['candidate_centres_k'+str(k)]
    availability.append(dict(stratum_code=k,environment=classes[k],
       strips_with_any_eligible_centres=int((values>0).sum()),
       total_candidate_centre_pixels=int(values.sum()),
       maximum_centres_in_one_strip=int(values.max())))
display(pd.DataFrame(availability))
print('Saved candidate-centre counts; NOT unique or independent circles.')
print('A selected centre still requires checking true 10 cm RGB validity.')


## Stage 3 — Joint ecological × image-quality sampling
Original G6 circles are ecological *candidates*, not verified cloud-clean samples. Review visual cloud screening per circle and preserve the original design.

## 7 — Ecologically stratified annotation-circle selection

From G.5/G.5B to a **reproducible, spatially balanced candidate sampling plan** for the WWF RGB data.

- Start with **rare regrowth classes (1–3)**, protecting geographic replication by limiting them initially to **one centre per stratum per WWF strip**. Then sample degradation and Hansen-loss classes (4–7), followed by preferentially **paired stable-forest controls (8)** in the same strips, and a small number of lower-canopy references (9).
- A **70%-pure 125 m-diameter disk** and a centre at least 62.5 m inside the potential RGB footprint are necessary for consideration (same principle as G.5B). Centres are chosen from stratified Earth Engine samples of eligible 30 m pixels, **not randomly from the strip**.
- Enforce **minimum 250 m distance between all selected centres** (geodesic), even if they belong to different strips/classes. The output is a realistic candidate design, not an inflated count of correlated pixels.
- Use a **deterministic seed** and selection targets; do not force rare targets where actual eligible places are absent. Export individual circle polygons, centres, CSV tables, per-stratum diagnostics, GeoPackage and map. Also save all screened points for revisiting the selection without rerunning Earth Engine.
- **Critical:** TMF/Hansen states are *hypotheses for sampling*, not Musanga labels, and TMF consecutive regrowth-state years are **not tree ages**. The footprint remains *survey polygon ∩ RGB file bounds*; selected circles are **provisional until verified against actual nonblack RGB coverage**. No circles are published to shared assets automatically.

Run **G.6A** (sampling and selection), then optionally **G.6B** (native RGB validity checks). Both have cached outputs; no full RGB mosaic download is required.

In [ ]:
# 7.A — Select spatially separated 125 m circles
# Recreates the G.5 ecological map (standalone in a fresh Colab runtime).
import json, math, hashlib, tempfile, shutil
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point, mapping
from pyproj import Geod
from google.colab import drive
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
import ee

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
G3_DIR=WORK_DIR/'TMF_strip_descriptors'/'rgb_footprint_pixels'
G5_DIR=WORK_DIR/'TMF_strip_descriptors'/'integrated_ecological_gradient_2014'
G6_DIR=WORK_DIR/'TMF_strip_descriptors'/'G6_125m_sampling_design'
G6_DIR.mkdir(parents=True,exist_ok=True)
src=G3_DIR/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
assert src.exists(), 'Run G.3 to create WWF potential RGB footprint polygons first.'
foot=gpd.read_file(src).to_crs(4326).sort_values('plot_id').reset_index(drop=True)
assert len(foot)==216 and foot.plot_id.is_unique
assert (G5_DIR/'WWF_G5_125m_circle_candidate_centre_counts.csv').exists(), \
    'Run G.5B first to verify feasible 125m neighbourhoods.'
g5counts=pd.read_csv(G5_DIR/'WWF_G5_125m_circle_candidate_centre_counts.csv').set_index('plot_id')

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

# Reproducible sampling controls
G6_SEED=20261028
CIRCLE_DIAMETER_M=125
RADIUS_M=CIRCLE_DIAMETER_M/2
NEIGHBOURHOOD_MIN_PURITY=.70
MIN_CENTRE_SEPARATION_M=250  # circle edges then have 125 m between them
MAX_PER_STRIP_TOTAL=5
MAX_PER_STRIP_EACH_STRATUM=2
MAX_PER_STRIP_RARE=1        # for each of 1,2,3
EE_CANDIDATES_PER_STRATUM_PER_STRIP=45
G6_REBUILD_CANDIDATE_CACHE=False
G6_TEST_N=None              # optional first 12 strips, or None for all 216
# Aspirational numbers; actual feasibility and spatial separation take precedence.
G6_TARGETS={1:7,2:10,3:15,4:16,5:12,6:12,7:12,8:22,9:8}
labels={
    1:'Young regrowth (TMF 1–3 yr state)',
    2:'Intermediate regrowth (TMF 4–7 yr state)',
    3:'Longer regrowth (TMF 8+ yr state)',
    4:'Degraded / no Hansen loss',
    5:'Degraded / Hansen loss',
    6:'Recent Hansen loss (2011–14)',
    7:'Historical Hansen loss (2001–10)',
    8:'Stable high-canopy forest reference',
    9:'Stable lower-canopy forest reference'
}
# Recreate EXACT priority rules of G.5
h=ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
st=tmf.select('Dec2014').toInt16()
ly=h.select('lossyear').toInt16()
c00=h.select('treecover2000')
valid=h.select('datamask').eq(1).And(st.gte(1)).And(st.lte(6))
loss=(ee.Image(0).where(ly.gte(1).And(ly.lte(5)),1)
    .where(ly.gte(6).And(ly.lte(10)),2)
    .where(ly.gte(11).And(ly.lte(14)),3)).toInt16()
age=ee.Image(0).toInt16()
for year in range(1990,2015):
    age=age.add(1).multiply(tmf.select('Dec'+str(year)).eq(4)).toInt16()
eco=ee.Image(10).toInt16()
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.gte(80)),8)
eco=eco.where(st.eq(1).And(loss.eq(0)).And(c00.lt(80)),9)
eco=eco.where(loss.gte(1).And(loss.lte(2)),7)
eco=eco.where(loss.eq(3),6)
eco=eco.where(st.eq(2).And(loss.eq(0)),4)
eco=eco.where(st.eq(2).And(loss.gt(0)),5)
eco=eco.where(st.eq(4).And(age.gte(1)).And(age.lte(3)),1)
eco=eco.where(st.eq(4).And(age.gte(4)).And(age.lte(7)),2)
eco=eco.where(st.eq(4).And(age.gte(8)),3)
eco=eco.updateMask(valid)

# Build all 9 local-purity surfaces; precedence is rare-first.
kernel=ee.Kernel.circle(radius=RADIUS_M,units='meters',normalize=False)
eligible_code=ee.Image(0).toInt16()
candidate_purity=ee.Image(0).toInt16()
for k in range(1,10):
    purity=(eco.eq(k).unmask(0).reduceNeighborhood(
        reducer=ee.Reducer.mean(),kernel=kernel,skipMasked=False))
    eligible=purity.gte(NEIGHBOURHOOD_MIN_PURITY).And(eligible_code.eq(0))
    eligible_code=eligible_code.where(eligible,k)
    candidate_purity=candidate_purity.where(eligible,purity.multiply(100).round())
candidate_img=ee.Image.cat(eligible_code.rename('eligible_code'),
                           candidate_purity.rename('purity_pct')).updateMask(eligible_code.gt(0))
sample_foot=foot if G6_TEST_N is None else foot.head(G6_TEST_N)
cache_dir=G6_DIR/'EE_candidate_samples'
cache_dir.mkdir(parents=True,exist_ok=True)
all_candidates=[]
errors=[]
for number,row in enumerate(sample_foot.itertuples(index=False),1):
    pid=str(row.plot_id)
    cache=cache_dir/(pid+'.json')
    if cache.exists() and not G6_REBUILD_CANDIDATE_CACHE:
        samples=json.loads(cache.read_text())
        all_candidates.extend(samples)
        if number%24==0: print('Cached centre samples:',number,'/',len(sample_foot))
        continue
    if not any(g5counts.loc[pid,'candidate_centres_k'+str(k)]>0 for k in range(1,10)):
        cache.write_text('[]')
        continue
    try:
        inner=ee.Geometry(mapping(row.geometry),geodesic=False).buffer(-RADIUS_M,maxError=5)
        fc=candidate_img.stratifiedSample(
            numPoints=0,classBand='eligible_code',classValues=list(range(1,10)),
            classPoints=[EE_CANDIDATES_PER_STRATUM_PER_STRIP]*9,
            region=inner,scale=30,seed=G6_SEED,dropNulls=True,
            geometries=True,tileScale=4)
        feats=fc.getInfo()['features']
        samples=[]
        for f in feats:
            props=f['properties']
            lon,lat=f['geometry']['coordinates']
            k=int(props['eligible_code'])
            samples.append(dict(plot_id=pid,utm_zone=str(row.utm_zone),
                                stratum=k,longitude=float(lon),latitude=float(lat),
                                purity_pct=float(props.get('purity_pct') or 0)))
        cache.write_text(json.dumps(samples))
        all_candidates.extend(samples)
        if number%12==0:print('Sampled eligible points:',number,'/',len(sample_foot))
    except Exception as ex:
        errors.append((pid,str(ex)[:220]))
        print('WARNING: candidate sampling failed:',pid,str(ex)[:170])
if errors:
    (G6_DIR/'EE_sample_errors.json').write_text(json.dumps(errors,indent=2))
    print('WARNING:',len(errors),'strip requests failed; do not treat this as exhaustive.')
assert all_candidates, 'No candidate points obtained. Inspect Earth Engine sampling errors.'
candidates=pd.DataFrame(all_candidates)
candidates.to_csv(G6_DIR/'WWF_G6_screened_candidate_centres.csv',index=False)
print('Total screened candidate centres:',len(candidates),
      'across',candidates.plot_id.nunique(),'WWF strips.')

# Geodesic separation among ALL selected centres, even across strip borders.
geod=Geod(ellps='WGS84')
rng=np.random.default_rng(G6_SEED)
candidates['tiebreak']=rng.random(len(candidates))
chosen=[]
strip_n=Counter()
per_pair_n=Counter()
zones_by_class=defaultdict(Counter)
selected_ids=set()
def far_enough(row):
    for q in chosen:
        # Cheap bounding-box guard: never skip actual nearby points.
        if abs(row.latitude-q['latitude'])>.01 or abs(row.longitude-q['longitude'])>.02:
            continue
        _,_,dist=geod.inv(float(row.longitude),float(row.latitude),
                          float(q['longitude']),float(q['latitude']))
        if dist<MIN_CENTRE_SEPARATION_M:return False
    return True
def add_stratum(k,matched_controls=False):
    opts=candidates[candidates.stratum.eq(k)].copy()
    if matched_controls:
        previous={q['plot_id'] for q in chosen if q['stratum']!=8}
        opts['paired_first']=opts.plot_id.isin(previous).astype(int)
        opts=opts.sort_values(['paired_first','purity_pct','tiebreak'],
                              ascending=[False,False,True])
    else:
        opts=opts.sort_values(['purity_pct','tiebreak'],ascending=[False,True])
    # Spatially distribute across strips/zones in rounds, avoiding a single high-purity strip.
    while sum(q['stratum']==k for q in chosen)<G6_TARGETS[k]:
        possibilities=[]
        for i,row in opts.iterrows():
            if i in selected_ids:continue
            pid=row.plot_id
            if strip_n[pid]>=MAX_PER_STRIP_TOTAL:continue
            limit=MAX_PER_STRIP_RARE if k in [1,2,3] else MAX_PER_STRIP_EACH_STRATUM
            if per_pair_n[(pid,k)]>=limit:continue
            if not far_enough(row):continue
            pfirst=int(pid in {q['plot_id'] for q in chosen if q['stratum'] in range(1,8)}) \
                if matched_controls else 0
            # Coverage first; prioritize unused strips for this stratum, and underused UTM zones.
            key=(-pfirst, per_pair_n[(pid,k)],strip_n[pid],
                 zones_by_class[k][row.utm_zone],-row.purity_pct,row.tiebreak)
            possibilities.append((key,i,row))
        if not possibilities:break
        _,i,row=min(possibilities,key=lambda t:t[0])
        d={col:row[col] for col in ['plot_id','utm_zone','stratum',
                                    'longitude','latitude','purity_pct']}
        d['selection_priority']='paired_reference' if matched_controls else 'ecological_gradient'
        chosen.append(d)
        selected_ids.add(i)
        strip_n[row.plot_id]+=1
        per_pair_n[(row.plot_id,k)]+=1
        zones_by_class[k][row.utm_zone]+=1

# Rare recovery first, then disturbance; matched high-canopy reference AFTER all targets.
for k in [1,2,3,4,5,6,7]:
    add_stratum(k)
add_stratum(8,matched_controls=True)
add_stratum(9)
selection=pd.DataFrame(chosen)
assert len(selection) and selection.stratum.isin(range(1,10)).all()
selection['environment']=selection.stratum.map(labels)
selection.insert(0,'circle_id',['WWF_G6_C'+str(i+1).zfill(3) for i in range(len(selection))])
selection['radius_m']=RADIUS_M
selection['rgb_validity']='NOT_CHECKED'
selection['sampling_status']='PROVISIONAL'
selection.to_csv(G6_DIR/'WWF_G6_provisional_125m_circles.csv',index=False)
# Actual metric-radius polygon circles, in the UTM CRS local to each centre.
centres=gpd.GeoDataFrame(selection.copy(),
    geometry=gpd.points_from_xy(selection.longitude,selection.latitude),crs=4326)
polys=[]
for row in centres.itertuples(index=False):
    geom=Point(float(row.longitude),float(row.latitude))
    local=gpd.GeoSeries([geom],crs='EPSG:4326')
    crs=local.estimate_utm_crs()
    poly=local.to_crs(crs).buffer(RADIUS_M,resolution=32).to_crs(4326).iloc[0]
    polys.append(poly)
circle_gdf=gpd.GeoDataFrame(selection.copy(),geometry=polys,crs=4326)
centres.to_file(G6_DIR/'WWF_G6_provisional_centre_points.gpkg',driver='GPKG')
circle_gdf.to_file(G6_DIR/'WWF_G6_provisional_125m_circles.gpkg',driver='GPKG')
# Also GeoJSON for easy import to QGIS, web maps and Earth Engine.
circle_gdf.to_file(G6_DIR/'WWF_G6_provisional_125m_circles.geojson',driver='GeoJSON')

report=[]
for k in labels:
    this=selection[selection.stratum.eq(k)]
    screened=candidates[candidates.stratum.eq(k)]
    report.append(dict(stratum_code=k,environment=labels[k],target=G6_TARGETS[k],
        n_selected=len(this),n_strips_selected=this.plot_id.nunique(),
        n_utm_zones_selected=this.utm_zone.nunique(),screened_candidate_pixels=len(screened),
        strips_with_sampled_candidate_centres=screened.plot_id.nunique(),
        median_purity_pct=this.purity_pct.median() if len(this) else np.nan,
        target_reached=len(this)>=G6_TARGETS[k]))
summary=pd.DataFrame(report)
summary.to_csv(G6_DIR/'WWF_G6_sampling_balance_and_shortfalls.csv',index=False)
print('\n=== G.6 ECOLOGICAL CIRCLE DESIGN ===')
display(summary.round(2))
print('\nSelected',len(selection),'provisional circles in',
      selection.plot_id.nunique(),'WWF strips; minimum centre spacing',MIN_CENTRE_SEPARATION_M,'m')
print('G.6 output folder:',G6_DIR)
print('NOT YET VALIDATED against actual 10 cm RGB. No training labels inferred.')

# Lightweight QA figures: diversity by stratum/UTM zone and global location overview.
temp=Path(tempfile.mkdtemp(prefix='g6_figs_'))
fig,ax=plt.subplots(figsize=(12,4.5))
xx=np.arange(len(summary))
ax.bar(xx-.2,summary.target,width=.4,label='Aspirational target')
ax.bar(xx+.2,summary.n_selected,width=.4,label='Provisional selected')
ax.set_xticks(xx,[str(k) for k in labels])
ax.set(xlabel='Ecological stratum code',ylabel='125 m circles',
       title='G.6 stratified sample: target vs spatially feasible selection')
ax.legend()
fig.tight_layout()
fig.savefig(temp/'G6_sampling_balance.png',dpi=120)
plt.close(fig)
shutil.copyfile(temp/'G6_sampling_balance.png',G6_DIR/'G6_sampling_balance.png')
display(IPImage(filename=str(temp/'G6_sampling_balance.png'),width=920))
fig,ax=plt.subplots(figsize=(12,6))
foot.boundary.plot(ax=ax,color='#b8b8b8',linewidth=.4)
centres.plot(ax=ax,column='stratum',categorical=True,cmap='tab10',
             legend=True,markersize=13)
ax.set(xlabel='Longitude',ylabel='Latitude',
       title='Geographic replication of provisional 125 m Musanga circles')
fig.tight_layout()
fig.savefig(temp/'G6_geographic_distribution.png',dpi=120)
plt.close(fig)
shutil.copyfile(temp/'G6_geographic_distribution.png',G6_DIR/'G6_geographic_distribution.png')
display(IPImage(filename=str(temp/'G6_geographic_distribution.png'),width=920))


### 7.B — Native WWF RGB screening of selected circles

Checks **only the G.6 shortlisted circles**, not all 264 GB of WWF mosaics. Each 125 m window is read in a small, downsampled rasterio array from the original GCS GeoTIFF using existing Colab access. We check coverage by RGB data values (nodata, black and obvious saturated pixels), and produce per-circle preview/contact sheets. Results remain **screening estimates**, not definitive image-valid masks: black shadow pixels may be real and imagery colour varies; a circle must still be visually reviewed before annotation.

Requires the authenticated Colab/GCS access that worked for NB08 RGB reads. Set `G6_RGB_CHECK=False` to skip this optional step; otherwise the GCS reads will only access bounded windows.

In [ ]:
# 7.B.1 — Screen native RGB image quality
# Important: original WWF RGB GeoTIFFs have no internal overviews.
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.windows import from_bounds, Window
from rasterio.enums import Resampling
from shapely.geometry import Point
from pyproj import Transformer
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive',force_remount=False)
WORK_DIR=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
G6_DIR=WORK_DIR/'TMF_strip_descriptors'/'G6_125m_sampling_design'
input_path=G6_DIR/'WWF_G6_provisional_125m_circles.csv'
assert input_path.exists(),'Run G.6A first'
G6_RGB_CHECK=True
G6_RGB_SAMPLE_SIDE=180    # downsampled read for the circle window
G6_RGB_PASS_FRACTION=0.95
if not G6_RGB_CHECK:
    print('RGB validity check skipped.')
else:
    circles=pd.read_csv(input_path)
    output_path=G6_DIR/'WWF_G6_rgb_coverage_screen.csv'
    old=pd.read_csv(output_path).set_index('circle_id').to_dict('index') if output_path.exists() else {}
    checked=[]
    rgb_root='/vsigs/carbon-maps/001_Lidar_Rasters/RGB_mosaic/'
    for n,c in enumerate(circles.itertuples(index=False),1):
        if c.circle_id in old:
            checked.append({'circle_id':c.circle_id,**old[c.circle_id]})
            continue
        uri=rgb_root+str(c.plot_id)+'_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as ds:
                    tr=Transformer.from_crs('EPSG:4326',ds.crs,always_xy=True)
                    x,y=tr.transform(float(c.longitude),float(c.latitude))
                    x0,y0=x-64,y-64
                    x1,y1=x+64,y+64
                    window=from_bounds(x0,y0,x1,y1,transform=ds.transform)
                    a=ds.read(indexes=[1,2,3],window=window,
                        out_shape=(3,G6_RGB_SAMPLE_SIDE,G6_RGB_SAMPLE_SIDE),
                        resampling=Resampling.nearest,boundless=True,fill_value=0)
            yy,xx=np.mgrid[:G6_RGB_SAMPLE_SIDE,:G6_RGB_SAMPLE_SIDE]
            inside=((xx+0.5-G6_RGB_SAMPLE_SIDE/2)**2+
                    (yy+0.5-G6_RGB_SAMPLE_SIDE/2)**2 <=
                    (G6_RGB_SAMPLE_SIDE*(62.5/128))**2)
            black=np.all(a==0,axis=0)
            saturated=np.all(a>=254,axis=0)
            valid=inside & ~black & ~saturated
            fraction=float(valid.sum()/max(inside.sum(),1))
            result=dict(circle_id=c.circle_id,plot_id=c.plot_id,
                        rgb_valid_pixel_fraction_proxy=fraction,
                        rgb_quality_screen=('PASS' if fraction>=G6_RGB_PASS_FRACTION else 'REVIEW'),
                        rgb_error='')
        except Exception as e:
            result=dict(circle_id=c.circle_id,plot_id=c.plot_id,
                        rgb_valid_pixel_fraction_proxy=np.nan,
                        rgb_quality_screen='ERROR',rgb_error=str(e)[:250])
        checked.append(result)
        if n%12==0:
            pd.DataFrame(checked).to_csv(output_path,index=False)
            print('Screened',n,'/',len(circles),'RGB circle windows')
    rgb=pd.DataFrame(checked)
    rgb.to_csv(output_path,index=False)
    print('Saved',output_path)
    display(rgb.groupby('rgb_quality_screen').size().rename('n_circles').to_frame())
    review=rgb[rgb.rgb_quality_screen.ne('PASS')]
    display(review.head(30))
    print('This is a bounded screen only; inspect RGB visually before accepting circles.')


### Stage 3C — Cross-reference ecology, annotated weather, RGB and solar metadata
Export an auditable original-vs-screened table without replacing, deleting, or retraining on any original circles.

In [ ]:
# Stage 3C — Joint ecological + visual cloud-screen shortlist (cached CSVs only)
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/NB08_WWF_Lidar')
G6=ROOT/'TMF_strip_descriptors'/'G6_125m_sampling_design'
QA=ROOT/'RGB_quality_audit'
base=pd.read_csv(G6/'WWF_G6_provisional_125m_circles.csv')
labels=pd.read_csv(QA/'WWF_G6_manual_cloud_illumination_review.csv')
assert base.circle_id.is_unique and labels.circle_id.is_unique
joint=base.merge(labels,on='circle_id',how='left',validate='one_to_one',indicator=True)
assert joint._merge.eq('both').all(),'Mismatch between annotated IDs and ecology G6 CSV'
joint=joint.drop(columns='_merge')
quality=QA/'WWF_216_strip_RGB_quality_audit.csv'
if quality.exists():
    q=pd.read_csv(quality)
    fields=[x for x in ['plot_id','median_luminance_median_sample',
            'brightness_spread_sample','review_required'] if x in q]
    joint=joint.merge(q[fields].drop_duplicates('plot_id'),on='plot_id',how='left',validate='many_to_one')
solar=QA/'acquisition_timing'/'WWF_RGB_solar_geometry_by_plot.csv'
if solar.exists():
    s=pd.read_csv(solar)
    fields=[x for x in ['plot_id','solar_zenith_deg','solar_elevation_deg',
       'solar_azimuth_deg','UTC_flight_time_proxy'] if x in s]
    joint=joint.merge(s[fields].drop_duplicates('plot_id'),on='plot_id',how='left',validate='many_to_one')
cov=G6/'WWF_G6_rgb_coverage_screen.csv'
if cov.exists():
    c=pd.read_csv(cov)
    fields=[x for x in ['circle_id','rgb_valid_pixel_fraction_proxy','rgb_quality_screen'] if x in c]
    joint=joint.merge(c[fields].drop_duplicates('circle_id'),on='circle_id',how='left',validate='one_to_one')
joint['sampling_status']=joint.weather_screen.fillna('NEEDS_REVIEW')
joint.loc[(joint.sampling_status=='USABLE_WEATHER') &
          (joint.annotation=='P'),'sampling_status']='AVOID_OTHER_QUALITY'
joint.to_csv(G6/'WWF_G6_original_114_ecology_x_cloud_review.csv',index=False)
display(joint.sampling_status.value_counts().rename_axis('status').to_frame('n'))
if 'stratum' in joint:
    display(pd.crosstab(joint.stratum,joint.sampling_status,margins=True))
print('Saved:',G6/'WWF_G6_original_114_ecology_x_cloud_review.csv')
print('No circle reselected. No extrapolation to all 216 RGB strips yet.')


## Appendix — Archived exploratory sampling (optional; superseded by Stage 3)

## 8 — Archived exploratory TMF sampling (optional; superseded by section 7)

**Research aim:** expand Musanga labels to secondary and regenerating forest, degraded canopy, and recently cleared landscapes, plus undisturbed reference forest. The WWF RGB imagery is the annotation source; TMF is a **coarse (30 m), contextual sampling stratifier**, not a Musanga label or a claim about individual 125 m² circles. <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>

**Temporal control:** The WWF survey imagery was acquired during **2014–2015** (individual plot years not yet distinguished). Sample only TMF states that remain consistent across *both* 2014 and 2015; recently cleared candidates must already be cleared by 2014. Later TMF classes are never used as contemporary labels. "Recently cleared" means transition from forest to deforested land within the preceding three years, as measured in the relevant TMF annual maps. "Regrowth" is the official TMF regrowth state at the acquisition year.

**Confirmed annotation unit:** 125 m **diameter** (radius 62.5 m; area approximately 1.227 ha). Candidate circles and exports use this diameter by default.

**Workflow:** verify dates → sample TMF strata within actual WWF LiDAR footprints → inspect 250 m context + native RGB validity → export selected circles and small georeferenced RGB clips to Drive → hand annotation in GIS → independent label QC. No automatic Musanga annotations or giant mosaic downloads.

In [ ]:
# 8.A — Historical pilot sampling metadata
import math, pandas as pd
from pathlib import Path
TMF_OUT=WORK_DIR/'TMF_guided_annotation'
TMF_OUT.mkdir(parents=True,exist_ok=True)
PILOT_PLOTS=['UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
             'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104']
CIRCLE_MODE='diameter_m'  # confirmed: 125 m diameter, not 125 m²
CIRCLE_VALUE=125.0
CIRCLE_RADIUS_M=(math.sqrt(CIRCLE_VALUE/math.pi)
                 if CIRCLE_MODE=='area_m2' else CIRCLE_VALUE/2)
print('Circle mode:',CIRCLE_MODE,'radius (m):',round(CIRCLE_RADIUS_M,2),
      'diameter (m):',round(2*CIRCLE_RADIUS_M,2),
      'area (m²):',round(math.pi*CIRCLE_RADIUS_M**2,1))
# User-confirmed WWF campaign dates: 2014–2015, individual plot years unknown.
# H2 will sample only candidate landscape states stable across BOTH years.
WWF_RGB_ACQUISITION_WINDOW=(2014,2015)
WWF_RGB_ACQUISITION_YEAR={p:None for p in PILOT_PLOTS}
print('Campaign period:',WWF_RGB_ACQUISITION_WINDOW,
      '| individual plot years remain unverified; use stable-class sampling.')
# List potentially useful bucket reports and metadata filenames (metadata requests only)
helpful=df[df.path.str.contains(r'00-raw-data/Reports/|\.xml$|\.pdf$',case=False,regex=True)]
display(helpful[['path','size_GB']].head(55))
print('Note: filename/2016 national products do NOT establish individual RGB flight year.')

### 8.B — Historical acquisition-date metadata
Inspect RGB GeoTIFF tags and list associated metadata/reports. **Do not mistake a file update, TIFF processing timestamp or 2016 map product date for an aerial acquisition date.** This is evidence gathering: candidate years must be verified before H.2.

In [ ]:
# 8.B.1 — Inspect WWF flight-date metadata
import re, json, pandas as pd, rasterio
from pathlib import PurePosixPath
from IPython.display import display
# Recognize acquisition-specific metadata fields; generic TIFF DateTime is often processing time.
date_key=re.compile(r'acqui|capture|flight|survey|sensing|exposure|gps.*date|date.*gps',re.I)
year_re=re.compile(r'(?<!\d)(19\d{2}|20\d{2})(?!\d)')
date_rows=[]
for plot_id in PILOT_PLOTS:
    rgb=f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif'
    hits=[]
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rgb) as ds:
                namespaces=[''] + [k for k in ds.tag_namespaces() if k]
                for ns in dict.fromkeys(namespaces):
                    try:
                        tags=ds.tags(ns=ns) if ns else ds.tags()
                    except Exception:
                        continue
                    for key,value in tags.items():
                        if date_key.search(key):
                            hits.append(dict(source='geotiff_tag',key=f'{ns}:{key}',value=str(value)[:300]))
                        elif re.search(r'datetime|software',key,re.I):
                            hits.append(dict(source='processing_tag_UNVERIFIED',key=f'{ns}:{key}',value=str(value)[:300]))
    except Exception as exc:
        hits.append(dict(source='ERROR',key='read',value=repr(exc)[:300]))
    # Candidate supporting filenames from source raw plot and report directories
    raw_files=df[df.path.str.contains(re.escape(plot_id.split('_')[0]),regex=True,na=False) &
                 df.path.str.contains(r'\.xml$|\.pdf$|\.txt$|\.csv$',case=False,regex=True,na=False)]
    for filename in raw_files.path.head(10):
        hits.append(dict(source='supporting_filename_ONLY',key='object',value=filename))
    if not hits:
        hits=[dict(source='NO_ACQUISITION_DATE_FOUND',key='',value='')]
    for hit in hits:
        date_rows.append(dict(plot_id=plot_id,**hit))
date_evidence=pd.DataFrame(date_rows)
display(date_evidence)
date_evidence.to_csv(TMF_OUT/'wwf_acquisition_date_evidence.csv',index=False)
# Reports inventory is small; inspect original survey reports if tags don't contain actual flight dates.
reports=df[df.path.str.startswith('00-raw-data/Reports/') & df.path.str.endswith('.pdf')]
print('\nSurvey reports (potential flight dates):')
display(reports[['path','size_GB']].head(70))
print('Evidence exported:',TMF_OUT/'wwf_acquisition_date_evidence.csv')
print('Fill WWF_RGB_ACQUISITION_YEAR in H1 only when a genuine flight acquisition year is established.')


In [ ]:
# 8.C — Superseded exploratory TMF circle sampling
# Only sites with matching TMF annual state in Dec2014 and Dec2015 qualify.
# 'Recently cleared' must already be deforested in 2014 and 2015,
# having been forest in 2011 or 2012. No post-2015 forest conditions used.
import ee, geopandas as gpd, numpy as np, pandas as pd, rasterio
from shapely.geometry import Point, mapping
from shapely.ops import transform as shape_transform
from pyproj import Transformer
assert tuple(WWF_RGB_ACQUISITION_WINDOW)==(2014,2015)
# Self-healing setup: GCS authentication alone does NOT initialize Earth Engine.
EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
assert EE_PROJECT, 'EE_PROJECT must not be empty'
print('GEE project:',repr(EE_PROJECT))
try:
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine connected:',EE_PROJECT,'| test:',ee.Number(1).getInfo())
except Exception as first_error:
    print('Earth Engine setup needed:',str(first_error)[:300])
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine authenticated:',EE_PROJECT,'| test:',ee.Number(1).getInfo())
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
annual14=tmf.select('Dec2014')
annual15=tmf.select('Dec2015')
forest11=tmf.select('Dec2011').eq(1).Or(tmf.select('Dec2011').eq(2))
forest12=tmf.select('Dec2012').eq(1).Or(tmf.select('Dec2012').eq(2))
# Stable class 1,2,4; class 3 only if recently deforested before image capture.
recent_clear=annual14.eq(3).And(annual15.eq(3)).And(forest11.Or(forest12))
stable=annual14.eq(annual15)
stratum=ee.Image.constant(0).where(stable.And(annual14.eq(1)),1)
stratum=stratum.where(stable.And(annual14.eq(2)),2)
stratum=stratum.where(recent_clear,3)
stratum=stratum.where(stable.And(annual14.eq(4)),4).rename('stratum').toByte()
# One 30 m TMF pixel is 900 m²: candidate forest category is broad spatial context,
# NOT a verified label for the much smaller annotation circle.
SAMPLES_PER_STRATUM_PER_PLOT=5
KEEP_ONLY_IF_WITHIN_PLOT=True
rows=[]
crs_by_plot={}
for plot_id in PILOT_PLOTS:
    plotrow=plots.loc[plots.plot_id==plot_id]
    assert len(plotrow)==1, plot_id
    geom=plotrow.geometry.iloc[0]
    region=ee.Geometry(mapping(geom),proj='EPSG:4326',geodesic=False)
    fc=stratum.updateMask(stratum.gt(0)).stratifiedSample(
        numPoints=SAMPLES_PER_STRATUM_PER_PLOT,classBand='stratum',
        classValues=[1,2,3,4],
        classPoints=[SAMPLES_PER_STRATUM_PER_PLOT]*4,
        region=region,scale=30,geometries=True,seed=8108,
        dropNulls=True,tileScale=4)
    features=fc.getInfo()['features']
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
        with rasterio.open('/vsigs/'+BUCKET+
             f'/001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif') as src:
            assert src.crs and src.crs.is_projected and src.crs.linear_units=='metre'
            crs_by_plot[plot_id]=src.crs
    for f in features:
        lon,lat=f['geometry']['coordinates']
        cls=int(f['properties']['stratum'])
        rows.append(dict(plot_id=plot_id,TMF_years='2014;2015',
                         time_policy='stable_in_both_2014_and_2015',
                         tmf_stratum=cls,
                         stratum_label={1:'undisturbed',2:'degraded',
                           3:'cleared_by_2014_after_2011_2012_forest',
                           4:'regrowth'}[cls],
                         longitude=float(lon),latitude=float(lat)))
    print(plot_id,':',len(features),'TMF candidate centres')
assert rows,'No eligible TMF candidate centres found'
table=pd.DataFrame(rows)
proposals=[]
for i,r in enumerate(table.itertuples(index=False),1):
    local_crs=crs_by_plot[r.plot_id]
    proj=Transformer.from_crs('EPSG:4326',local_crs,always_xy=True)
    x,y=proj.transform(r.longitude,r.latitude)
    circle=Point(x,y).buffer(CIRCLE_RADIUS_M,resolution=32)
    footprint=gpd.GeoSeries(
        [plots.loc[plots.plot_id==r.plot_id,'geometry'].iloc[0]],
        crs='EPSG:4326').to_crs(local_crs).iloc[0]
    if not footprint.covers(circle):continue
    rev=Transformer.from_crs(local_crs,'EPSG:4326',always_xy=True)
    proposals.append(dict(candidate_id=f'WWF_{i:04d}',plot_id=r.plot_id,
        TMF_years=r.TMF_years,time_policy=r.time_policy,
        tmf_stratum=r.tmf_stratum,stratum_label=r.stratum_label,
        geometry=shape_transform(rev.transform,circle)))
assert proposals,'Candidates exist but circles did not fit in mapped WWF footprints'
gdf=gpd.GeoDataFrame(proposals,geometry='geometry',crs='EPSG:4326')
gdf['rgb_validated']=False
gdf['visually_approved']=False
gdf['musanga_annotation_status']='NOT_ANNOTATED'
gdf.to_file(TMF_OUT/'wwf_tmf_candidate_circles.gpkg',driver='GPKG')
gdf.drop(columns='geometry').to_csv(TMF_OUT/'wwf_tmf_candidate_circles.csv',index=False)
print('Proposed (not visually approved) circles:',len(gdf))
display(gdf.groupby(['plot_id','stratum_label']).size().rename('n').reset_index())
print('TMF classes must be confirmed visually in 2014–2015 RGB.')


In [ ]:
# 8.D — Archived RGB context export
# Select at most one candidate per plot × TMF class initially, no bulk download.
# Images contain full 250 m context; annotation circle stays much smaller.
import rasterio, numpy as np, geopandas as gpd, pandas as pd
from rasterio.windows import from_bounds, Window
from rasterio.windows import transform as window_transform
from rasterio.enums import Resampling
from pathlib import Path
CHIP_CONTEXT_M=250
MAX_CHIPS=12
RUN_CHIP_EXPORT=False  # change to True after inspecting candidate table
CHIP_DIR=TMF_OUT/'rgb_native_context_chips'
CHIP_DIR.mkdir(parents=True,exist_ok=True)
if RUN_CHIP_EXPORT:
    selection=gdf.sort_values(['plot_id','tmf_stratum','candidate_id']).groupby(
        ['plot_id','tmf_stratum']).head(1).head(MAX_CHIPS)
    chip_logs=[]
    for row in selection.itertuples(index=False):
        uri='/vsigs/'+BUCKET+f'/001_Lidar_Rasters/RGB_mosaic/{row.plot_id}_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as src:
                    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(src.crs).iloc[0]
                    x,y=circle.centroid.x,circle.centroid.y
                    half=CHIP_CONTEXT_M/2
                    win=from_bounds(x-half,y-half,x+half,y+half,src.transform)
                    win=win.round_offsets().round_lengths()
                    full=Window(0,0,src.width,src.height)
                    win=win.intersection(full)
                    a=src.read(indexes=[1,2,3],window=win,masked=True)
                    data=np.ma.filled(a,0)
                    valid=(~np.ma.getmaskarray(a).any(axis=0)) & np.any(data!=0,axis=0)
                    fraction=float(valid.mean())
                    if fraction<.70:
                        print('SKIP low-validity',row.candidate_id,round(fraction,3))
                        continue
                    profile=src.profile.copy()
                    profile.update(driver='GTiff',count=3,width=int(win.width),
                                   height=int(win.height),transform=window_transform(win,src.transform),
                                   tiled=True,compress='deflate',BIGTIFF='IF_SAFER')
                    for key in ['photometric','interleave']:
                        profile.pop(key,None)
                    dst=CHIP_DIR/f'{row.candidate_id}_{row.plot_id}_{row.stratum_label}.tif'
                    with rasterio.open(dst,'w',**profile) as sink:
                        sink.write(data)
                        sink.colorinterp=src.colorinterp[:3]
                    chip_logs.append(dict(candidate_id=row.candidate_id,
                                          plot_id=row.plot_id,context_m=CHIP_CONTEXT_M,
                                          native_gsd_cm=100*abs(src.res[0]),
                                          valid_fraction=fraction,rgb_tif=str(dst)))
                    print('Saved:',dst.name,'| valid',round(fraction,3))
        except Exception as ex:
            print('FAILED:',row.candidate_id,repr(ex))
    pd.DataFrame(chip_logs).to_csv(TMF_OUT/'wwf_context_chip_exports.csv',index=False)
    print('Use GeoPackage circles as separate QGIS overlay. Do NOT confuse TMF stratum with a field label.')
else:
    print('Export OFF. Review H2 candidates, then set RUN_CHIP_EXPORT=True.')
print('Working directory:',TMF_OUT)

### 8.E — Exhaustive annotation quality controls

Load `wwf_tmf_candidate_circles.gpkg` and corresponding **georeferenced** RGB chips into QGIS. First check the imagery acquisition year, validity, and apparent disturbance/regrowth stage; reject mismatched areas. Annotate all visible Musanga crowns inside each approved circle and explicitly record true absence as zero. Keep uncertainty labels for ambiguous young crowns; do **not** treat unknown pixels as negative. Record annotator, date, annotation confidence, tree/crown counts, and image quality. For a robust cross-forest validation exercise, group spatially nearby circles by parent WWF plot to avoid leakage.

Sampling reliability: TMF pixels are ~30 m, coarser than the crown-scale detail within a **125 m diameter** circle. TMF determines the surrounding landscape class, not the circle's actual vegetation. Candidate status is never ground truth. Crop valid coverage and human photo interpretation must confirm suitability before annotation.

**Campaign date 2014–2015:** H2 uses only TMF candidate states stable across these two years. Verify per-plot dates later for finer transition sampling.

### 8.F — Optional legacy Earth Engine circle exports

Publish a deterministic 12-circle pilot (three per TMF category, diverse WWF plots) or all 90 circles into `musanga-511110`. Two **separate** table assets are created: the **125 m diameter exhaustive annotation circles** and the **225 m diameter context boundaries**, representing an additional 50 m buffer. Circles are buffered in their own metric UTM CRS before conversion to WGS84. Assets preserve circle identifiers, site, TMF sampling category, and unreviewed annotation status. This operation **does not upload WWF RGB imagery or create editable annotations**; imagery must be separately ingested as EE image assets (subject to redistribution permissions), and annotations should be saved in a new dedicated vector layer.

The cell starts Earth Engine export tasks. Wait for both tasks to complete before using their IDs in the <https://code.earthengine.google.com/> Code Editor. Share the table assets with collaborators using Earth Engine Assets permissions.

In [ ]:
# 8.F.1 — Publish historical pilot candidates (optional)
# This creates TWO table assets: 125 m diameter cores and 225 m context circles.
# Default: same 12 reproducible examples as H.2B (3 per TMF class).
# To publish all candidate circles, change EXPORT_SCOPE = 'all_90'.
import re, ee, pandas as pd, geopandas as gpd
from shapely.geometry import mapping

EE_PROJECT = (globals().get('EE_PROJECT') or 'musanga-511110').strip()
ee.Initialize(project=EE_PROJECT)

EXPORT_SCOPE = 'pilot_12'     # 'pilot_12' or 'all_90'
N_PER_CLASS = 3
SELECTION_SEED = 42
OUTER_BUFFER_M = 50.0        # additional 50 m outside the 62.5 m core
SUBMIT_EXPORT_TASKS = False  # archived pilot: preview by default; opt in for exports

if EXPORT_SCOPE == 'pilot_12':
    labels = ['undisturbed','degraded',
              'cleared_by_2014_after_2011_2012_forest','regrowth']
    selections=[]
    for label in labels:
        sub=gdf[gdf['stratum_label'].eq(label)].sample(frac=1,random_state=SELECTION_SEED)
        distinct=sub.drop_duplicates('plot_id').head(N_PER_CLASS)
        remainder=sub.loc[~sub.index.isin(distinct.index)].head(N_PER_CLASS-len(distinct))
        selections.append(pd.concat([distinct,remainder]))
    export_gdf=gpd.GeoDataFrame(pd.concat(selections,ignore_index=True),
                                geometry='geometry',crs=gdf.crs)
elif EXPORT_SCOPE == 'all_90':
    export_gdf=gdf.copy()
else:
    raise ValueError('EXPORT_SCOPE must be pilot_12 or all_90')

assert len(export_gdf)>0
assert export_gdf.candidate_id.is_unique, 'Duplicate candidate IDs'
assert export_gdf.crs.to_epsg()==4326, 'Expected WGS84 original circles'
# Fail if H1 was accidentally run in 125 m² mode.
assert abs(CIRCLE_RADIUS_M-62.5)<0.01, 'Expected 125 m DIAMETER (radius 62.5 m)'

def plot_utm_epsg(plot_id):
    found=re.match(r'^UTM(\d{1,2})([NS])_',str(plot_id))
    if not found:raise ValueError('Unexpected plot UTM zone: '+str(plot_id))
    zone=int(found.group(1))
    return (32600 if found.group(2)=='N' else 32700)+zone

core_features=[]
context_features=[]
for _,row in export_gdf.iterrows():
    # Geometries are made in a local metric CRS (one of several WWF UTM zones).
    utm=plot_utm_epsg(row['plot_id'])
    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(epsg=utm).iloc[0]
    expanded=circle.buffer(OUTER_BUFFER_M,resolution=48)
    center=circle.centroid
    assert abs(circle.area - 3.141592653589793*62.5**2)<35, (
        'Core does not have the expected 125 m diameter: '+str(row['candidate_id']))
    geom_core=gpd.GeoSeries([circle],crs=f'EPSG:{utm}').to_crs(4326).iloc[0]
    geom_context=gpd.GeoSeries([expanded],crs=f'EPSG:{utm}').to_crs(4326).iloc[0]
    attrs={
        'circle_id':str(row['candidate_id']),
        'plot_id':str(row['plot_id']),
        'tmf_class':str(row['stratum_label']),
        'tmf_code':int(row['tmf_stratum']),
        'image_years':'2014-2015',
        'sampling':'TMF_2014_2015_stable',
        'review':'NOT_REVIEWED',
        'annotation':'NOT_ANNOTATED',
        'core_diam_m':125,
        'context_diam_m':225,
    }
    core_features.append(ee.Feature(
        ee.Geometry(mapping(geom_core),geodesic=False),
        {**attrs,'boundary':'core_125m'}))
    context_features.append(ee.Feature(
        ee.Geometry(mapping(geom_context),geodesic=False),
        {**attrs,'boundary':'context_225m'}))

core_fc=ee.FeatureCollection(core_features)
context_fc=ee.FeatureCollection(context_features)
suffix='pilot12' if EXPORT_SCOPE=='pilot_12' else 'all90'
core_asset=f'projects/{EE_PROJECT}/assets/WWF_TMF_2014_2015_{suffix}_core_125m'
context_asset=f'projects/{EE_PROJECT}/assets/WWF_TMF_2014_2015_{suffix}_context_225m'

print('Circles selected:',len(export_gdf))
display(export_gdf[['candidate_id','plot_id','stratum_label']])
print('Core asset:',core_asset)
print('Context asset:',context_asset)

# Preserve the same candidates locally for annotation logistics.
export_gdf.to_file(
    TMF_OUT/f'WWF_TMF_2014_2015_{suffix}_core125m.gpkg',driver='GPKG')

if SUBMIT_EXPORT_TASKS:
    for title,fc,asset in [
        ('core125m',core_fc,core_asset),
        ('context225m',context_fc,context_asset),
    ]:
        try:
            existing=ee.data.getAsset(asset)
        except ee.EEException as err:
            if any(term in str(err).lower() for term in ('not found', '404', 'does not exist')):
                existing=None
            else:
                raise  # permission/authentication errors must NOT be mistaken for missing assets
        if existing is not None:
            print('ALREADY EXISTS (not overwritten):',asset)
            continue
        task=ee.batch.Export.table.toAsset(
            collection=fc,
            description=f'WWF_TMF_{suffix}_{title}',
            assetId=asset)
        task.start()
        print('Submitted export:',title,'task ID:',task.id)
    print('Check the Earth Engine Tasks tab. An export is not immediately available.')
else:
    print('Dry run only; set SUBMIT_EXPORT_TASKS=True to create EE assets.')


## Legacy final report (original ecological design, not cloud-screened)
The existing report still documents the original 114 ecological candidates. Do not mistake it for the Stage 3 quality-screened shortlist.

## 9 — Final two-part WWF report (single endpoint)

**Run 9.A only.** Sections 1–8 provide the underlying analysis, but their output caches are reused. No reprocessing of TMF, Hansen, G.5 or G.6 is required.

The report is deliberately limited to two parts:

**Part I — WWF forest history:** data and analytical methods; TMF–Hansen classification and measured distributions across **216** WWF RGB survey strips; pooled and between-strip statistics; a **satellite-background overview map showing all 216 strips**. The background is retrieved from Esri World Imagery at runtime (with attribution); if satellite imagery cannot be downloaded, the code explicitly stops instead of substituting a misleading basemap.

**Part II — Selection of candidate Musanga circles:** transparent 125 m circle criteria, balanced allocation of the **114 selected circles across 83 strips**, actual purity, spatial separation and RGB screening; the 114 locations on a second map; **two large, native-resolution WWF RGB examples per landscape A4 page**, grouped by ecological category. The original RGB is accessed directly from the private WWF bucket only for bounded image crops; existing GCS read permissions are sufficient.

**One output directory only:** `Final/Outputs/V3_1/NB08_WWF_Lidar/Report/`. All NB08 analyses, caches and the final Report now use this single output tree. Run the one-time **0.A migration** first if the former root folder still exists; the generator never moves caches itself. Produces a PDF and an editable Word version, along with an auditable native-RGB preview cache for the 114 selected circles. The report fails rather than silently producing empty RGB plates.

In [ ]:
# 9.A — ONE CLEAN, TWO-PART WWF REPORT (A4 LANDSCAPE, 2 LARGE RGB EXAMPLES PER PAGE)
# Reuse G.3–G.6 results; do not move, duplicate or delete other notebook caches.
import sys, subprocess, importlib.util, math, json, io, os, textwrap
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from pyproj import Transformer
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import drive, auth
import requests, google.auth
from google.auth.transport.requests import Request
for package,install in [('reportlab','reportlab'),('docx','python-docx')]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',install])
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A3, A4, landscape
from reportlab.lib.colors import HexColor
from reportlab.lib.utils import ImageReader
from docx import Document
from docx.shared import Cm, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK=BASE/'Final'/'Outputs'/'V3_1'/'NB08_WWF_Lidar'
ANALYSIS=WORK/'TMF_strip_descriptors'
assert ANALYSIS.exists(),f'TMF/Hansen cache missing at {ANALYSIS}'
G3=ANALYSIS/'rgb_footprint_pixels'
G5=ANALYSIS/'integrated_ecological_gradient_2014'
G6=ANALYSIS/'G6_125m_sampling_design'
OUT=BASE/'Final'/'Outputs'/'V3_1'/'NB08_WWF_Lidar'/'Report'
CHIPS=OUT/'RGB_previews_125m'
OUT.mkdir(parents=True,exist_ok=True);CHIPS.mkdir(parents=True,exist_ok=True)
print('FINAL REPORT DIRECTORY:',OUT)
CIRC=G6/'WWF_G6_provisional_125m_circles.csv'
BAL=G6/'WWF_G6_sampling_balance_and_shortfalls.csv'
ECO=G5/'WWF_G5_global_integrated_strata.csv'
REPL=G5/'WWF_G5_stratum_replication_across_strips.csv'
FEAS=G5/'WWF_G5_125m_circle_candidate_centre_counts.csv'
FOOT=G3/'WWF_all_RGB_footprints_TMF_pixel_descriptors.gpkg'
RGB=G6/'WWF_G6_rgb_coverage_screen.csv'
AGE=G5/'WWF_G5_regrowth_age_x_hansen_loss_counts.csv'
for x in [CIRC,BAL,ECO,REPL,FEAS,FOOT,RGB,AGE]:
    assert x.exists(),f'Missing saved analysis result: {x}'
circ=pd.read_csv(CIRC);balance=pd.read_csv(BAL)
eco=pd.read_csv(ECO);rep=pd.read_csv(REPL)
feas=pd.read_csv(FEAS);rgb=pd.read_csv(RGB)
age=pd.read_csv(AGE,index_col=0)
foot=gpd.read_file(FOOT).to_crs(4326)
assert len(foot)==216 and len(circ)==114
assert foot.plot_id.is_unique and circ.circle_id.is_unique
assert circ.plot_id.nunique()==83
assert rgb.rgb_quality_screen.eq('PASS').sum()==114
cent=gpd.GeoDataFrame(circ,geometry=gpd.points_from_xy(circ.longitude,circ.latitude),crs=4326)
LABELS={1:'Young TMF regrowth (1–3 years)',2:'Intermediate regrowth (4–7 years)',
 3:'Longer regrowth (8+ years)',4:'Degraded forest, no Hansen loss',
 5:'Degraded forest, Hansen loss',6:'Recent Hansen loss (2011–2014)',
 7:'Historical Hansen loss (2001–2010)',8:'Undisturbed high-canopy reference',
 9:'Undisturbed lower-canopy reference',10:'Other / ambiguous — not sampled'}
COLORS=['#2c8c53','#86be44','#1f613c','#f2aa3e','#bf753a',
        '#c94837','#8d3e4b','#397b9b','#9880b5']
# Read actual G.5B per-strip centre counts to avoid any unexplained hard-coded values.
feas_summary=[]
for k in range(1,10):
    col=f'candidate_centres_k{k}'
    v=feas[col]
    feas_summary.append((k,int((v>0).sum()),int(v.sum())))
# ---- Satellite map: full WWF geography, consistent georeferencing in EPSG:4326 ----
from PIL import Image as PILImage
lonmin,latmin,lonmax,latmax=foot.total_bounds
padx=max((lonmax-lonmin)*.07,.35);pady=max((latmax-latmin)*.07,.35)
bbox=[lonmin-padx,latmin-pady,lonmax+padx,latmax+pady]
SAT=OUT/'WWF_216_strips_satellite_location_map.png'
SAT_SELECTED=OUT/'WWF_216_strips_and_114_circles_satellite_map.png'
basemap_cache=OUT/'Esri_World_Imagery_background.jpg'
if not basemap_cache.exists():
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/export'
    params=dict(bbox=','.join(f'{v:.7f}' for v in bbox),bboxSR=4326,imageSR=4326,
                size='2200,1300',format='jpg',f='image')
    res=requests.get(url,params=params,timeout=100)
    res.raise_for_status()
    try:
        bg=PILImage.open(io.BytesIO(res.content));bg.verify()
    except Exception as err:
        raise RuntimeError('Satellite-background service returned no valid image. '
            'No substitute basemap will be labelled as satellite.') from err
    basemap_cache.write_bytes(res.content)
assert basemap_cache.exists()
bg=PILImage.open(basemap_cache).convert('RGB')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
def make_map(path,show_selection):
    fig,ax=plt.subplots(figsize=(14,8))
    ax.imshow(bg,extent=(bbox[0],bbox[2],bbox[1],bbox[3]),
              interpolation='bilinear',zorder=0,aspect='auto')
    foot.boundary.plot(ax=ax,color='#f6f9f7',linewidth=.9,alpha=.88,zorder=2)
    if show_selection:
        for k in range(1,10):
            sel=cent[cent.stratum==k]
            sel.plot(ax=ax,markersize=24,marker='o',color=COLORS[k-1],
                     edgecolor='#111111',linewidth=.35,zorder=4)
        handles=[Line2D([0],[0],marker='o',linestyle='None',markersize=7,
                    markerfacecolor=COLORS[k-1],markeredgecolor='black',
                    label=f'{k}: {LABELS[k]}') for k in range(1,10)]
        ax.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,-.32),
                  ncol=3,fontsize=8,frameon=True)
    ax.set_xlim(bbox[0],bbox[2]);ax.set_ylim(bbox[1],bbox[3])
    ax.set_xlabel('Longitude (°E)');ax.set_ylabel('Latitude (°N)')
    ax.set_title('216 WWF RGB survey strips'+
                 (' and 114 proposed 125 m circles' if show_selection else ''),
                 fontsize=14)
    ax.text(.01,.012,'Imagery: Esri World Imagery / contributors • WWF polygons: survey footprints',
            transform=ax.transAxes,fontsize=7,color='white',
            bbox=dict(facecolor='black',alpha=.65,pad=3),zorder=10)
    fig.tight_layout()
    fig.savefig(path,dpi=180,facecolor='white',bbox_inches='tight')
    plt.close(fig)
make_map(SAT,False)
make_map(SAT_SELECTED,True)
print('Satellite location maps saved.')

# ---- Native WWF RGB: unique, large (1400×1400 px) bounded window per circle ----
# These are detailed previews for forest-structure inspection, not substitutes
# for full native 10 cm orthomosaics during exhaustive crown annotation.
RGB_PIXELS=1400
RGB_WINDOW_METRES=170.0
REMAKE_CHIPS=False
def fetch_token():
    try:
        creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
        if not creds.valid or not creds.token:creds.refresh(Request())
        assert creds.token
        return str(creds.token)
    except Exception:
        auth.authenticate_user()
        creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/devstorage.read_only'])
        if not creds.valid or not creds.token:creds.refresh(Request())
        assert creds.token
        return str(creds.token)
def crop_one(q,token):
    path=CHIPS/(str(q.circle_id)+'_RGB1400.jpg')
    if path.exists() and not REMAKE_CHIPS:
        try:
            with PILImage.open(path) as im:
                im.verify()
            return path
        except Exception:
            path.unlink()
    srcuri='/vsigs/carbon-maps/001_Lidar_Rasters/RGB_mosaic/'+str(q.plot_id)+'_RGB.tif'
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
                      GS_OAUTH2_ACCESS_TOKEN=token):
        with rasterio.open(srcuri) as ds:
            x,y=Transformer.from_crs('EPSG:4326',ds.crs,always_xy=True).transform(
                float(q.longitude),float(q.latitude))
            h=RGB_WINDOW_METRES/2
            win=from_bounds(x-h,y-h,x+h,y+h,transform=ds.transform)
            raw=ds.read(indexes=(1,2,3),window=win,boundless=True,fill_value=0,
                        out_shape=(3,RGB_PIXELS,RGB_PIXELS),
                        resampling=Resampling.bilinear)
    if raw.dtype==np.uint8:
        a=raw.transpose(1,2,0).copy()
    else:
        z=raw.astype('float32')
        mask=np.any(z>0,axis=0)
        if not mask.any():raise RuntimeError(f'Empty native RGB crop: {q.circle_id}')
        lo=np.percentile(z[:,mask],1);hi=np.percentile(z[:,mask],99)
        a=np.clip((z-lo)/(max(hi-lo,1e-6))*255,0,255).astype('uint8').transpose(1,2,0)
    if np.all(a==0):raise RuntimeError(f'All-black native RGB crop: {q.circle_id}')
    im=PILImage.fromarray(a,'RGB')
    dr=ImageDraw.Draw(im)
    m=RGB_PIXELS/2;rad=RGB_PIXELS*62.5/RGB_WINDOW_METRES
    dr.ellipse((m-rad,m-rad,m+rad,m+rad),outline='#ff412e',width=7)
    im.save(path,quality=91,optimize=True,subsampling=0)
    return path

to_make=[q for q in circ.itertuples(index=False)
         if REMAKE_CHIPS or not (CHIPS/(q.circle_id+'_RGB1400.jpg')).exists()]
if to_make:
    token=fetch_token()
    for i,q in enumerate(to_make,1):
        crop_one(q,token)
        if i%10==0:print(f'Native RGB previews: {i}/{len(to_make)}')
else:
    print('All 114 detailed native RGB previews already cached.')
assert all((CHIPS/(q.circle_id+'_RGB1400.jpg')).is_file()
           for q in circ.itertuples(index=False)), 'Incomplete image atlas'
# Print actual selection summary — do not infer target count as independent sampling.
print('Selected:',len(circ),'circles across',circ.plot_id.nunique(),'WWF strips')
display(balance[['stratum_code','target','n_selected','n_strips_selected','median_purity_pct']])
print('All imagery cached. Building two-part report.')

# ---- Single authoritative PDF document, landscape A3 ----
from reportlab.pdfgen import canvas as PDFCanvas
from reportlab.lib.pagesizes import A3 as A3_SIZE,landscape as LR
from reportlab.lib.colors import HexColor as HC
from reportlab.lib.utils import ImageReader as Reader
W,H=LR(A3_SIZE)  # Virtual layout coordinates: scaled proportionally to physical A4 landscape
PW,PH=LR(A4)
PDF=OUT/'WWF_Musanga_TMF_Hansen_and_circle_selection_A4_landscape.pdf'
c=PDFCanvas.Canvas(str(PDF),pagesize=(PW,PH),pageCompression=1)
PAGE_SCALE=PW/W
c.scale(PAGE_SCALE,PAGE_SCALE)
INK=HC('#173c34');GREY=HC('#44524e');LINE=HC('#d7e3de')
page_no=0
def title(text,subtitle=''):
    c.setFillColor(INK);c.rect(0,H-70,W,70,stroke=0,fill=1)
    c.setFillColor(HC('#ffffff'));c.setFont('Helvetica-Bold',21)
    c.drawString(40,H-38,text[:94])
    c.setFont('Helvetica',10);c.drawString(41,H-57,subtitle[:145])
def footer():
    global page_no
    page_no+=1
    c.setStrokeColor(LINE);c.line(35,32,W-35,32)
    c.setFont('Helvetica',8);c.setFillColor(GREY)
    c.drawString(37,19,'WWF Musanga | 2014 reference | candidate annotations, not observed stand ages')
    c.drawRightString(W-37,19,str(page_no))
def end():
    footer();c.showPage();c.scale(PAGE_SCALE,PAGE_SCALE)
def linewrap(t,x,y,maxchars=153,size=10,lead=15):
    c.setFillColor(GREY);c.setFont('Helvetica',size)
    for l in textwrap.wrap(str(t),width=maxchars):
        c.drawString(x,y,l);y-=lead
    return y
def label(t,y):
    c.setFillColor(INK);c.setFont('Helvetica-Bold',13);c.drawString(40,y,t);return y-25
def draw_table(headers,rows,y,colx,rowheight=20,fontsize=9):
    c.setFillColor(INK);c.setFont('Helvetica-Bold',fontsize)
    for x,h in zip(colx,headers):c.drawString(x,y,str(h)[:58])
    y-=rowheight
    for i,row in enumerate(rows):
        if i%2==0:
            c.setFillColor(HC('#f0f5f2'))
            c.rect(35,y-6,W-70,rowheight,stroke=0,fill=1)
        c.setFillColor(GREY);c.setFont('Helvetica',fontsize)
        for x,v in zip(colx,row):c.drawString(x,y,str(v)[:70])
        y-=rowheight
    return y

title('PART I — Forest disturbance and recovery across WWF RGB strips',
      '216 WWF LiDAR/RGB survey strips, Democratic Republic of Congo | primary forest history through 2014')
y=H-105
y=label('Objective and data',y)
y=linewrap('Characterize the disturbance–recovery environments represented in the WWF aerial RGB survey and construct a sampling frame for exhaustive Musanga cecropioides crown annotation. The inventory comprises 216 WWF 10 cm RGB orthomosaics (2014–2015 acquisition), survey footprints, JRC Tropical Moist Forest (TMF; ~30 m), and Hansen Global Forest Change (~30 m).',40,y)
y-=15;y=label('TMF × Hansen methods',y)
for t in [
 'Footprints are WWF survey polygons intersected with RGB image bounding rectangles (not exact pixel-valid footprint masks).',
 'TMF 2014 status, historical degradation/deforestation, and consecutive regrowth-state duration were summarized for all strips.',
 'Hansen 2000 canopy cover, forest gain 2000–2012, and forest loss cohorts 2001–05, 2006–10, 2011–14 were paired with TMF.',
 'Classes were applied by mutually exclusive priority rules. Percentages are across valid paired pixels; equal-strip distributions were also calculated.'
]:
    y=linewrap('• '+t,44,y,maxchars=145,size=9.5,lead=15);y-=5
y-=9;y=label('Core findings',y)
y=linewrap(f'{int(eco.pixels.sum()):,} valid paired 30 m pixel observations. TMF 2014: undisturbed forest 77.66%, degraded forest 5.30%, deforested 5.53%, regrowth 0.51%, other 10.81% and water 0.15%. Across WWF strips, median undisturbed share was 93.9% and median regrowth 0.05% (maximum 9.83%). Hansen detected loss through 2014 in 3.81% of paired pixels; only 6.21% of TMF regrowth overlapped Hansen forest loss in 2001–2014.',40,y,maxchars=150)
y-=14;y=label('Ecological classification — share of paired pixels',y)
rows=[[int(r.code),LABELS[int(r.code)],f'{r.pct_of_valid_pixels:.2f}%'] for r in eco.itertuples(index=False)]
y=draw_table(['Code','Satellite-derived ecological environment','Share of survey'],rows,y,[45,100,715],rowheight=21,fontsize=9)
y-=15
y=linewrap('IMPORTANT: TMF consecutive regrowth-state duration is not measured forest stand age. Hansen loss is a stand-replacing loss detection, not a complete history of canopy gaps.',40,y,maxchars=150,size=9)
end()

title('PART I — Geographic distribution of WWF RGB survey strips',
      'Satellite backdrop: Esri World Imagery / contributors | outlined polygons: WWF potential RGB survey footprints')
c.drawImage(Reader(str(SAT)),35,59,width=W-70,height=H-155,
            preserveAspectRatio=True,anchor='c')
end()

title('PART I — Replication and forest recovery histories',
      'Pixel distributions do not imply independent sites; each WWF strip is a separate sampling unit')
y=H-110
y=label('Spatial occurrence of ecological environments',y)
repro={int(r.code):r for r in rep.itertuples(index=False)}
rows=[]
for k in range(1,10):
    rr=repro[k]
    rows.append([str(k),LABELS[k],f'{rr.median:.2f}%',
                 f'{rr.pct_strips_zero:.1f}%',int(getattr(rr,'strips_ge_1pct'))])
y=draw_table(['Code','Ecological environment','Median / strip','Strips with 0','Strips >=1%'],
             rows,y,[45,100,675,825,990],rowheight=24,fontsize=9)
y-=17;y=label('TMF regrowth-state duration × Hansen forest-loss cohort',y)
cols=['No detected loss','2001–05','2006–10','2011–14']
rows=[[idx]+[f'{int(v):,}' for v in vals] for idx,vals in age.iterrows()]
y=draw_table(['TMF regrowth state']+cols,rows,y,[45,350,550,730,890],
             rowheight=27,fontsize=9)
y-=20
y=linewrap('Most TMF regrowth pixels have no Hansen loss through 2014: 90.59% for 1–3 years, 91.70% for 4–7 years and 96.89% for 8+ years. These products detect different dimensions of forest change; absence of Hansen loss does not establish absence of disturbance.',40,y,maxchars=155)
end()

title('PART II — Ecologically stratified selection of annotation circles',
      '114 selected 125 m diameter circles in 83 WWF strips | automated WWF RGB screening: 114/114 PASS')
y=H-106;y=label('Sampling criteria',y)
for t in [
 'Candidates: >=70% target ecological class within a 62.5 m radius disk; centre >=62.5 m inside potential RGB footprint.',
 'Minimum 250 m spacing between selected centre locations, checked across all WWF strips.',
 'Rare TMF regrowth classes (1–3) prioritized; disturbance classes (4–7) sampled over independent strips.',
 'Stable forest controls (8–9) added, preferably paired with disturbed strips; maximum 5 circles per strip.',
 'Target total 114 was an a priori design allocation, NOT an optimized sample-size calculation or unbiased random census.',
 'All 114 passed simple RGB coverage screening; final human review for crown visibility remains necessary.'
]:
    y=linewrap('• '+t,42,y,maxchars=148,size=10);y-=8
y-=10;y=label('Actual allocation (all class targets achieved)',y)
rows=[]
for rr in balance.itertuples(index=False):
    k=int(rr.stratum_code)
    rows.append([k,LABELS[k],int(rr.n_selected),
        int(rr.n_strips_selected),f'{rr.median_purity_pct:.0f}%'])
y=draw_table(['Code','Ecological environment','Circles','WWF strips','Median purity'],
             rows,y,[45,105,710,850,970],rowheight=23,fontsize=9)
y-=10;y=linewrap('Class 10 (other / ambiguous) is included in the landscape assessment but excluded from targeted circle selection. The 114 centres are not independent landscape replicates; strip-level clustering must be considered during subsequent modelling.',40,y,maxchars=155,size=9)
end()

title('PART II — WWF strips and selected annotation circles',
      'Satellite backdrop: Esri World Imagery / contributors | circle centres coloured by ecological class')
c.drawImage(Reader(str(SAT_SELECTED)),35,55,width=W-70,height=H-150,
            preserveAspectRatio=True,anchor='c')
end()

# Each class on one or more A4 LANDSCAPE pages: TWO large RGB crops side by side.
for k in range(1,10):
    group=circ[circ.stratum.eq(k)].sort_values('circle_id')
    records=list(group.itertuples(index=False))
    for start in range(0,len(records),2):
        items=records[start:start+2]
        title(f'PART II — Class {k}: {LABELS[k]}',
              f'Two large WWF RGB crops per A4 landscape page • image extent 170 × 170 m • red ring: 125 m circle')
        margin=45; gapx=24
        panelW=(W-2*margin-gapx)/2
        side=min(panelW-12,H-190)
        for j,q in enumerate(items):
            col=j
            x=margin+col*(panelW+gapx)
            top=H-106
            ipath=CHIPS/(q.circle_id+'_RGB1400.jpg')
            px=x+(panelW-side)/2;py=top-side
            c.drawImage(Reader(str(ipath)),px,py,width=side,height=side)
            c.setFillColor(INK);c.setFont('Helvetica-Bold',11)
            c.drawString(x+8,py-17,f'{q.circle_id} • {q.plot_id}')
            c.setFillColor(GREY);c.setFont('Helvetica',9)
            c.drawString(x+8,py-30,f'Purity {q.purity_pct:.0f}% • stratum {k} • RGB 10 cm source')
        end()
print('Writing PDF:',PDF)
c.save()
assert PDF.is_file() and PDF.stat().st_size>2_000_000

# ---- Editable companion Word, same two parts, same maps, 2 images per A4 landscape page ----
from docx.enum.section import WD_ORIENT
D=Document();sec=D.sections[0]
sec.orientation=WD_ORIENT.LANDSCAPE
sec.page_width=Cm(29.7);sec.page_height=Cm(21)
sec.left_margin=Cm(1.8);sec.right_margin=Cm(1.8)
sec.top_margin=Cm(1.5);sec.bottom_margin=Cm(1.5)
normal=D.styles['Normal'];normal.font.name='Aptos';normal.font.size=Pt(10)
normal.paragraph_format.space_after=Pt(4)
def addp(text):D.add_paragraph(str(text))
def addt(headers,rows):
    t=D.add_table(rows=1,cols=len(headers));t.style='Light Shading Accent 1'
    for i,v in enumerate(headers):t.rows[0].cells[i].text=str(v)
    for vals in rows:
        cells=t.add_row().cells
        for i,v in enumerate(vals):cells[i].text=str(v)
    for rr in t.rows:
        for cell in rr.cells:
            for p in cell.paragraphs:
                for run in p.runs:run.font.size=Pt(9)
D.add_heading('Part I — WWF forest disturbance and recovery',0)
addp('Democratic Republic of Congo | 216 WWF RGB/LiDAR survey strips | 2014 forest-history reference')
D.add_heading('Objective and datasets',1)
addp('Characterize forest disturbance and recovery environments represented in the 2014–2015 WWF RGB aerial survey and develop an ecological sampling frame for exhaustive Musanga cecropioides crown annotation. Sources: WWF 10 cm RGB orthomosaics and survey footprints, JRC TMF ~30 m annual forest status, and Hansen Global Forest Change ~30 m.')
D.add_heading('Methods and statistics',1)
addp('TMF status, historical disturbance and consecutive recovery-state duration through 2014 were cross-tabulated with Hansen 2000 canopy cover, 2000–2012 gain and 2001–2014 loss cohorts. Across 216 strips, valid paired observations total {:,}.'.format(int(eco.pixels.sum())))
addp('TMF 2014: 77.66% undisturbed, 5.30% degraded, 5.53% deforested, 0.51% regrowth, 10.81% other, 0.15% water. Median per-strip undisturbed cover 93.9%; median regrowth 0.05%. Hansen loss 2001–14: 3.81%. Only 6.21% of TMF regrowth overlapped Hansen loss 2001–14.')
addt(['Code','Integrated ecological environment','Survey share (%)'],
    [[int(q.code),LABELS[int(q.code)],f'{q.pct_of_valid_pixels:.2f}']
     for q in eco.itertuples(index=False)])
addp('Interpretation: TMF consecutive regrowth-state duration is not tree or forest stand age; Hansen detected loss is not a complete record of canopy disturbance.')
D.add_page_break();D.add_heading('Location of all 216 WWF survey strips',1)
D.add_picture(str(SAT),width=Cm(24.5))
addp('Satellite background: Esri World Imagery / contributors. White outlines: potential WWF RGB footprints.')
D.add_page_break();D.add_heading('Between-strip replication and regrowth histories',1)
addt(['Code','Environment','Median % / strip','Strips with 0%','Strips >=1%'],
 [[k,LABELS[k],f'{repro[k].median:.2f}',
  f'{repro[k].pct_strips_zero:.1f}%',int(repro[k].strips_ge_1pct)]
 for k in range(1,10)])
addt(['TMF regrowth state','No loss','Loss 2001–05','Loss 2006–10','Loss 2011–14'],
 [[idx]+[int(v) for v in row] for idx,row in age.iterrows()])
addp('Most TMF regrowth pixels did not overlap Hansen forest loss before 2015: 90.59% (1–3y state), 91.70% (4–7y state), 96.89% (8+y state).')
D.add_page_break();D.add_heading('Part II — Annotation-circle sampling',0)
addp('114 selected 125 m circles from 83 WWF strips. Candidate centre requirements: >=70% target stratum within a 62.5 m disk; 62.5 m interior buffer; >=250 m between selected centre locations. Rare TMF regrowth classes were prioritized. Up to five circles per strip. 114 is a deliberately set sampling allocation, not an optimized sample size.')
addt(['Class','Environment','Selected circles','WWF strips','Median purity'],
 [[int(q.stratum_code),LABELS[int(q.stratum_code)],int(q.n_selected),
   int(q.n_strips_selected),f'{q.median_purity_pct:.0f}%']
 for q in balance.itertuples(index=False)])
addp('All selected circles passed a bounded automated RGB coverage screen (114/114). Final acceptance requires a manual native-image crown visibility check. Class 10 was not sampled.')
D.add_page_break();D.add_heading('Selected circles on the WWF satellite map',1)
D.add_picture(str(SAT_SELECTED),width=Cm(24.5))
D.add_page_break()
for k in range(1,10):
    records=list(circ[circ.stratum.eq(k)].sort_values('circle_id').itertuples(index=False))
    for first in range(0,len(records),2):
        if not (k==1 and first==0):D.add_page_break()
        D.add_heading(f'Class {k} — {LABELS[k]}',level=1)
        grid=D.add_table(rows=1,cols=2)
        grid.autofit=False
        for idx,q in enumerate(records[first:first+2]):
            cell=grid.cell(0,idx)
            cell.text=''
            pp=cell.paragraphs[0];pp.alignment=WD_ALIGN_PARAGRAPH.CENTER
            pp.add_run().add_picture(str(CHIPS/(q.circle_id+'_RGB1400.jpg')),
                                     width=Cm(11.4),height=Cm(11.4))
            cap=cell.add_paragraph(f'{q.circle_id} | {q.plot_id}\nPurity {q.purity_pct:.0f}%')
            cap.alignment=WD_ALIGN_PARAGRAPH.CENTER
            for run in cap.runs:run.font.size=Pt(8)
        addp('WWF native RGB crop (170 × 170 m). Red ring: proposed 125 m diameter annotation circle.')
DOCX=OUT/'WWF_Musanga_TMF_Hansen_and_circle_selection_A4_landscape.docx'
D.save(DOCX)
assert DOCX.is_file()
print('COMPLETED — two-part report:')
print(' PDF:',PDF,'MB:',round(PDF.stat().st_size/1e6,1))
print(' WORD:',DOCX,'MB:',round(DOCX.stat().st_size/1e6,1))
print(' Satellite maps:',SAT.name,'and',SAT_SELECTED.name)
print(' Native RGB circle previews:',len(list(CHIPS.glob('*_RGB1400.jpg'))))
print(' No caches were moved. No outdated report folder was modified.')
